# PDDL PLANNER & Val

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%cd /content
!git clone https://github.com/KCL-Planning/VAL.git
%cd /content/VAL
!cmake -S . -B build -DCMAKE_BUILD_TYPE=Release
!cmake --build build -j2

In [ ]:
%cd /content
!git clone https://github.com/aibasel/downward.git
%cd /content/downward
!./build.py

# Qwen

In [ ]:
# ============================================================
# 0. Install dependencies if needed
# ============================================================
!pip install -U bitsandbytes>=0.46.1
!pip install -U transformers accelerate

In [ ]:
# ============================================================
# 1. Imports
# ============================================================

import torch
import json
import re
from pathlib import Path
from typing import Dict, List, Any, Optional

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    pipeline,
    BitsAndBytesConfig
)

In [ ]:
# ============================================================
# 2. Model Initialization
# ============================================================

MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

print("Loading model in 4-bit...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map="auto",
    quantization_config=bnb_config,
    torch_dtype=torch.float16,
)

generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=2000,
    do_sample=False,
    return_full_text=False,
)

print("Model loaded successfully.")

1. Shared helpers
2. Stakeholder + plan-fragment generation
3. Path intent generation: EXISTENCE / NON_EXISTENCE
4. Goal predicate generation
5. Final controller:
      EXISTENCE     -> validity pipeline
      NON_EXISTENCE -> invalidity pipeline

In [ ]:
import time
query = "q4-OT-Valid-2"

In [ ]:
# ============================================================
# Cell 1. Imports and paths
# ============================================================

from pathlib import Path
from collections import deque
from typing import Dict, List, Any, Optional, Set, Tuple
import os
import re
import json
import subprocess
import shutil
import hashlib


BASE_DIR = "/content/drive/MyDrive/Colab Notebooks/EARG/"

# Main input
INPUT_TXT = BASE_DIR + f"Query/{query}.txt"

# Domain/problem used for LLM mapping and goal inference
FULL_DOMAIN_PDDL = BASE_DIR + "models_test/FULL/mod_domain.pddl"
FULL_PROBLEM_PDDL = BASE_DIR + "models_test/FULL/mod_problem.pddl"

# Lattice
LATTICE_JSON = BASE_DIR + "lattice_output/Lattice_with_bridge_nodes.json"

# Files created by earlier stages
STAKEHOLDER_JSON = BASE_DIR + f"Traces/stakeholder_and_action_mapping_output_{query}.json"
PLAN_FILE = BASE_DIR + f"Traces/mapped_plan_fragments_{query}.txt"

PATH_INTENT_JSON = BASE_DIR + f"Traces/path_existence_intent_{query}.json"

GOAL_FILE = BASE_DIR + f"Traces/inferred_goal_predicate_{query}.txt"
GOAL_OUTPUT_JSON = BASE_DIR + f"Traces/goal_predicate_inference_output_{query}.json"

# VAL path for invalidity branch
VAL_BIN = "/content/VAL/build/bin/Validate"

# Validity outputs
MEA_OUTPUT_JSON = BASE_DIR + f"Traces/mea_lattice_search_result_{query}.json"
MEA_OUTPUT_REPORT = BASE_DIR + f"Traces/mea_lattice_search_report_{query}.txt"

# Invalidity outputs
INVALIDITY_TRACE_DIR = BASE_DIR + f"Traces/invalidity_val_traces_{query}"
INVALIDITY_RESULT_JSON = BASE_DIR + f"Traces/plan_invalidity_result_{query}.json"
INVALIDITY_REPORT_TXT = BASE_DIR + f"Traces/plan_invalidity_report_{query}.txt"

INVALID_DIAGNOSIS_JSON = BASE_DIR + f"Traces/invalid_plan_diagnosis_{query}.json"
INVALID_DIAGNOSIS_TXT = BASE_DIR + f"Traces/invalid_plan_diagnosis_{query}.txt"

#Explanations
VALID_EXPLANATION_JSON = BASE_DIR + f"Traces/llm_valid_path_explanation_{query}.json"
VALID_EXPLANATION_TXT = BASE_DIR + f"Traces/llm_valid_path_explanation_{query}.txt"

INVALID_EXPLANATION_JSON = BASE_DIR + f"Traces/llm_invalid_path_explanation_{query}.json"
INVALID_EXPLANATION_TXT = BASE_DIR + f"Traces/llm_invalid_path_explanation_{query}.txt"

# Goal update / restore outputs
GOAL_UPDATE_REPORT_TXT = BASE_DIR + f"Traces/goal_update_report_{query}.txt"
GOAL_RESTORE_REPORT_TXT = BASE_DIR + f"Traces/goal_restore_report_{query}.txt"
GOAL_BACKUP_DIR = BASE_DIR + f"Traces/problem_goal_backups_{query}"

# Metrics output paths
METRICS_DIR = BASE_DIR + "Experiment_Metrics"
METRICS_CSV = METRICS_DIR + "/earg_test_metrics.csv"
METRICS_XLSX = METRICS_DIR + "/earg_test_metrics.xlsx"

#Plan Repair Fail explanation
PLAN_REPAIR_FAILED_EXPLANATION_JSON = BASE_DIR+f"Traces/plan_repair_failed_explanation_{query}.json"
PLAN_REPAIR_FAILED_EXPLANATION_TXT = BASE_DIR+f"Traces/plan_repair_failed_explanation_{query}.txt"

#Plan Repair Fail explanation
NOT_INVALID_EXPLANATION_JSON =BASE_DIR+f"Traces/not_invalid_explanation_{query}.json"
NOT_INVALID_EXPLANATION_TXT = BASE_DIR+f"Traces/not_invalid_explanation_{query}.txt"


In [ ]:
# ============================================================
# Metrics logger: no raw JSON in CSV/Excel
# ============================================================

import csv
import os
from pathlib import Path
from datetime import datetime

def read_file_content_for_metrics(path, max_chars=30000):
    """
    Reads text file content for metrics.
    Returns empty string if the file does not exist yet.
    Does not read JSON files.
    """
    if path is None:
        return ""

    try:
        path = str(path)

        if path.lower().endswith(".json"):
            return ""

        if not os.path.exists(path):
            return ""

        with open(path, "r", encoding="utf-8", errors="replace") as f:
            content = f.read()

        if len(content) > max_chars:
            return content[:max_chars] + "\n\n[TRUNCATED FOR EXCEL CELL LIMIT]"

        return content

    except Exception as e:
        return f"[READ ERROR: {e}]"


def join_items(items, sep=" | "):
    """
    Converts a list of strings into readable text.
    Does not use json.dumps.
    """
    if items is None:
        return ""

    if isinstance(items, list):
        return sep.join(str(x) for x in items)

    return str(items)


def count_items(items):
    if isinstance(items, list):
        return len(items)
    return 0


def append_metrics_row(row, csv_path):
    """
    Appends one row to the metrics CSV.
    If the file does not exist, it creates it.
    If new columns appear, it expands the CSV safely.
    """
    Path(csv_path).parent.mkdir(parents=True, exist_ok=True)

    file_exists = os.path.exists(csv_path)

    if file_exists:
        with open(csv_path, "r", newline="", encoding="utf-8") as f:
            reader = csv.DictReader(f)
            old_fieldnames = reader.fieldnames or []
            old_rows = list(reader)

        fieldnames = list(old_fieldnames)

        for key in row.keys():
            if key not in fieldnames:
                fieldnames.append(key)

        old_rows.append(row)

        with open(csv_path, "w", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=fieldnames)
            writer.writeheader()
            writer.writerows(old_rows)

    else:
        fieldnames = list(row.keys())

        with open(csv_path, "w", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=fieldnames)
            writer.writeheader()
            writer.writerow(row)

    print("Appended metrics row to:", csv_path)


def export_metrics_to_excel(csv_path, xlsx_path):
    """
    Regenerates Excel from the CSV.
    CSV is the append-safe source.
    """
    try:
        import pandas as pd

        if not os.path.exists(csv_path):
            print("Metrics CSV does not exist yet:", csv_path)
            return

        df = pd.read_csv(csv_path)

        Path(xlsx_path).parent.mkdir(parents=True, exist_ok=True)

        with pd.ExcelWriter(xlsx_path, engine="openpyxl") as writer:
            df.to_excel(writer, index=False, sheet_name="EARG Metrics")

            worksheet = writer.sheets["EARG Metrics"]
            worksheet.freeze_panes = "A2"

            for column_cells in worksheet.columns:
                max_length = 0
                column_letter = column_cells[0].column_letter

                for cell in column_cells:
                    value = str(cell.value) if cell.value is not None else ""
                    max_length = max(max_length, len(value))

                worksheet.column_dimensions[column_letter].width = min(max_length + 2, 45)

        print("Exported metrics Excel to:", xlsx_path)

    except Exception as e:
        print("Could not export Excel.")
        print("CSV was still saved.")
        print("Reason:", e)

In [ ]:
def ground_goal_with_problem_objects(goal_text: str, problem_path: str) -> str:
    """
    Ensures the inferred goal is grounded using constants from the problem file.
    If the LLM returns ?plc, ?server, etc., this tries to replace it with
    the matching object from (:objects ...) or init atoms.
    """

    goal = normalize_goal_predicate(goal_text)
    objects = extract_constants_from_problem(problem_path)

    if not objects:
        raise ValueError(f"No problem objects found in: {problem_path}")

    objects = [obj.lower() for obj in objects]

    inside = goal.strip()[1:-1].strip()
    parts = inside.split()

    if not parts:
        raise ValueError(f"Empty goal predicate: {goal}")

    predicate = parts[0].lower()
    args = parts[1:]

    grounded_args = []

    for arg in args:
        arg = arg.lower()

        # Already grounded
        if not arg.startswith("?"):
            if arg not in objects:
                raise ValueError(
                    f"Goal uses object '{arg}', but it is not declared in the problem file. "
                    f"Available objects: {objects}"
                )
            grounded_args.append(arg)
            continue

        # Variable case, e.g., ?plc
        hint = arg[1:].lower()

        candidates = [obj for obj in objects if hint in obj]

        # Extra useful heuristics for your CPS model
        if len(candidates) != 1:
            for key in ["plc", "network", "server", "windows"]:
                if key in hint or key in predicate:
                    key_candidates = [obj for obj in objects if key in obj]
                    if len(key_candidates) == 1:
                        candidates = key_candidates
                        break

        if len(candidates) == 1:
            grounded_args.append(candidates[0])
        else:
            raise ValueError(
                f"Could not safely ground variable '{arg}' in goal '{goal}'. "
                f"Available problem objects: {objects}. "
                f"Fix the LLM goal prompt or provide a deterministic grounding rule."
            )

    if grounded_args:
        return f"({predicate} {' '.join(grounded_args)})"

    return f"({predicate})"

In [ ]:
# ============================================================
# Collect metrics from validity pipeline
# ============================================================

def collect_validity_metrics_no_json(result):
    result = result or {}

    repair = result.get("plan_repair_result", {}) or {}
    repair_rounds = repair.get("repair_rounds", [])

    missing_predicates = []
    inserted_actions = []

    for r in repair_rounds:
        missing_predicates.extend(r.get("missing_positive_preconditions_string", []))
        inserted_actions.extend(r.get("inserted_actions_string", []))

    return {
        "pipeline_type": "validity",
        "pipeline_status": result.get("status", "validity_completed"),
        "intent": result.get("intent"),
        "declared_stakeholder": result.get("declared_stakeholder") or result.get("predicted_stakeholder"),
        "stakeholder_source": result.get("stakeholder_source", "declared_in_query_file"),
        "predicted_stakeholder": result.get("predicted_stakeholder"),
        "start_node": result.get("start_node"),
        "branch_prefix": result.get("branch_prefix"),
        "goal": result.get("goal"),

        "repair_status": repair.get("status"),
        "plan_needed_repair": "yes" if len(repair_rounds) > 0 else "no",
        "repair_round_count": len(repair_rounds),
        "missing_predicate_count_before_repair": len(missing_predicates),
        "missing_predicates_before_repair": join_items(missing_predicates),
        "inserted_repair_action_count": len(inserted_actions),
        "inserted_repair_actions": join_items(inserted_actions),

        "mea_was_run": result.get("mea_was_run", True),
        "selected_valid_node_policy": result.get("selected_valid_node_policy"),
        "abstract_valid_node": result.get("abstract_valid_node"),
        "first_valid_node": result.get("first_valid_node"),
        "first_bridge_valid_node": result.get("first_bridge_valid_node"),
        "first_structured_bridge_valid_node": result.get("first_structured_bridge_valid_node"),
        "first_structured_valid_node": result.get("first_structured_valid_node"),
        "bridge_selection_constraint": result.get("bridge_selection_constraint"),
        "last_valid_node": result.get("last_valid_node"),

        "search_chain_length": count_items(result.get("search_chain", [])),
        "search_chain": join_items(result.get("search_chain", [])),
        "visited_node_count": count_items(result.get("visited_nodes", [])),
        "visited_nodes": join_items(result.get("visited_nodes", [])),
        "valid_node_count": count_items(result.get("valid_nodes", [])),
        "valid_nodes": join_items(result.get("valid_nodes", [])),
        "invalid_node_count": count_items(result.get("invalid_nodes", [])),
        "invalid_nodes": join_items(result.get("invalid_nodes", [])),

        "full_valid_by_mea": result.get("full_valid_by_mea"),
        "full_unachieved_subgoal_count": count_items(result.get("full_unachieved_subgoals", [])),
        "full_unachieved_subgoals": join_items(result.get("full_unachieved_subgoals", [])),

        "abstract_precondition_count": count_items(result.get("abstract_preconditions", [])),
        "abstract_preconditions": join_items(result.get("abstract_preconditions", [])),
        "relaxed_constraint_count": count_items(result.get("relaxed_constraints", [])),
        "relaxed_constraints": join_items(result.get("relaxed_constraints", [])),

        "original_plan_content": read_file_content_for_metrics(result.get("original_plan_file")),
        "repaired_plan_content": read_file_content_for_metrics(result.get("repaired_plan_file")),
        "mea_report_content": read_file_content_for_metrics(MEA_OUTPUT_REPORT),
        "valid_explanation_content": read_file_content_for_metrics(VALID_EXPLANATION_TXT)
    }


In [ ]:
# ============================================================
# Collect metrics from invalidity pipeline
# ============================================================

def collect_invalidity_metrics_no_json(result):
    result = result or {}

    invalidity_result = result.get("invalidity_result", {}) or {}
    diagnosis = result.get("diagnosis", {}) or {}

    first_invalid = invalidity_result.get("first_invalid_result", {}) or {}
    first_failure = diagnosis.get("first_failure", {}) or {}

    full_concrete_check = invalidity_result.get("full_concrete_check", {}) or {}

    missing_preconditions = first_failure.get("missing_preconditions", [])

    return {
        "pipeline_type": "invalidity",
        "pipeline_status": invalidity_result.get("status", "invalidity_completed"),
        "intent": invalidity_result.get("intent"),
        "declared_stakeholder": invalidity_result.get("declared_stakeholder") or invalidity_result.get("predicted_stakeholder"),
        "stakeholder_source": invalidity_result.get("stakeholder_source", "declared_in_query_file"),
        "predicted_stakeholder": invalidity_result.get("predicted_stakeholder"),
        "start_node": invalidity_result.get("start_node"),

        "full_concrete_status": full_concrete_check.get("status"),

        "first_invalid_node": invalidity_result.get("first_invalid_node"),
        "first_invalid_val_status": first_invalid.get("status"),
        "first_invalid_val_trace": first_invalid.get("trace_file"),

        "visited_node_count": count_items(invalidity_result.get("visited_nodes", [])),
        "visited_nodes": join_items(invalidity_result.get("visited_nodes", [])),
        "valid_node_count": count_items(invalidity_result.get("valid_nodes", [])),
        "valid_nodes": join_items(invalidity_result.get("valid_nodes", [])),
        "invalid_node_count": count_items(invalidity_result.get("invalid_nodes", [])),
        "invalid_nodes": join_items(invalidity_result.get("invalid_nodes", [])),
        "error_node_count": count_items(invalidity_result.get("error_nodes", [])),
        "error_nodes": join_items(invalidity_result.get("error_nodes", [])),

        "diagnosis_was_run": "yes" if diagnosis else "no",
        "plan_executed_completely": diagnosis.get("plan_executed"),
        "first_failure_step": first_failure.get("step_number"),
        "first_failure_action": first_failure.get("plan_action"),
        "first_failure_type": first_failure.get("failure_type"),
        "missing_precondition_count": count_items(missing_preconditions),
        "missing_preconditions": join_items(missing_preconditions),
        "execution_trace_step_count": count_items(diagnosis.get("execution_trace", [])),

        "invalidity_report_content": read_file_content_for_metrics(INVALIDITY_REPORT_TXT),
        "invalid_diagnosis_content": read_file_content_for_metrics(INVALID_DIAGNOSIS_TXT),
        "invalid_explanation_content": read_file_content_for_metrics(INVALID_EXPLANATION_TXT)
    }

In [ ]:
# ============================================================
# Build one metrics row for the current test case
# ============================================================

def build_test_case_metrics_row_no_json(
    query_name,
    preprocess_result,
    pipeline_result,
    total_runtime_seconds=None,
    notes=""
):
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    run_id = datetime.now().strftime("%Y%m%d_%H%M%S")

    row = {
        "run_id": run_id,
        "timestamp": timestamp,
        "test_case_id": query_name,
        "query_content": read_file_content_for_metrics(INPUT_TXT),
        "notes": notes,
        "total_runtime_seconds": total_runtime_seconds
    }

    # --------------------------------------------------------
    # Preprocessing metrics
    # --------------------------------------------------------
    mapping = preprocess_result.get("mapping", {}) if preprocess_result else {}
    intent_result = preprocess_result.get("intent", {}) if preprocess_result else {}
    goal_result = preprocess_result.get("goal", {}) if preprocess_result else {}

    mapped_fragments = mapping.get("mapped_plan_fragments", [])
    step_mapping = mapping.get("step_action_mapping", [])

    row.update({
        "declared_stakeholder": mapping.get("declared_stakeholder") or mapping.get("stakeholder"),
        "stakeholder_source": mapping.get("stakeholder_source"),
        "stakeholder_inference_used": mapping.get("stakeholder_inference_used", False),
        "llm_predicted_stakeholder": mapping.get("llm_predicted_stakeholder"),
        "final_stakeholder_view": mapping.get("declared_stakeholder") or mapping.get("stakeholder") or mapping.get("predicted_stakeholder"),
        "mapped_plan_fragment_count": count_items(mapped_fragments),
        "mapped_plan_fragments": join_items(mapped_fragments),
        "step_action_mapping_count": count_items(step_mapping),
        "path_intent": intent_result.get("intent"),
        "path_intent_confidence": intent_result.get("confidence"),
        "inferred_goal": goal_result.get("goal_predicate"),
        "goal_justification": goal_result.get("justification")
    })

    # --------------------------------------------------------
    # Pipeline metrics
    # --------------------------------------------------------
    if pipeline_result:
        row["pipeline_intent"] = pipeline_result.get("intent")
        row["pipeline_ran"] = pipeline_result.get("ran")

        result = pipeline_result.get("result", {})

        if pipeline_result.get("intent") == "EXISTENCE":
            row.update(collect_validity_metrics_no_json(result))

        elif pipeline_result.get("intent") == "NON_EXISTENCE":
            row.update(collect_invalidity_metrics_no_json(result))

    return row

In [ ]:
# ============================================================
# Cell 2. Shared file helpers
# ============================================================

def read_text(path: str) -> str:
    return Path(path).read_text(encoding="utf-8", errors="ignore")


def write_text(path: str, text: str) -> None:
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    Path(path).write_text(text, encoding="utf-8")


def read_json(path: str) -> Dict[str, Any]:
    return json.loads(Path(path).read_text(encoding="utf-8"))


def write_json(path: str, data: Dict[str, Any]) -> None:
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    Path(path).write_text(json.dumps(data, indent=2), encoding="utf-8")


def ensure_dir(path: str) -> None:
    os.makedirs(path, exist_ok=True)


def check_file_exists(path: str, label: str) -> None:
    if not os.path.exists(path):
        raise FileNotFoundError(f"{label} not found: {path}")
    print(f"Found {label}: {path}")


def normalize_atom(atom: str) -> str:
    atom = atom.strip().lower()
    atom = re.sub(r"\s+", " ", atom)
    return atom


def resolve_path(path_str: str, lattice_json_path: str) -> str:
    """
    Resolve domain/problem paths from lattice.json.
    Absolute Colab paths are used directly if they exist.
    Relative paths are checked against lattice folder, BASE_DIR, and cwd.
    """
    p = Path(path_str)

    if p.is_absolute():
        if p.exists():
            return str(p)
        raise FileNotFoundError(f"Absolute path from lattice does not exist: {p}")

    candidates = [
        Path(lattice_json_path).parent / p,
        Path(BASE_DIR) / p,
        Path.cwd() / p
    ]

    for candidate in candidates:
        if candidate.exists():
            return str(candidate)

    raise FileNotFoundError(
        "Could not resolve path:\n"
        f"  raw path: {path_str}\n"
        "Tried:\n" +
        "\n".join(f"  - {c}" for c in candidates)
    )


def extract_json_from_text(text: str) -> Dict[str, Any]:
    """
    Robustly extract JSON from LLM output.
    """
    text = text.strip()

    try:
        return json.loads(text)
    except Exception:
        pass

    match = re.search(r"\{.*\}", text, flags=re.DOTALL)

    if match:
        try:
            return json.loads(match.group(0))
        except Exception:
            pass

    return {
        "error": "Could not parse JSON from LLM output.",
        "raw_output": text
    }

In [ ]:
# ============================================================
# Cell 3. User query parser
# ============================================================

def normalize_declared_stakeholder_label(value: str) -> str:
    """
    Normalizes an explicitly declared stakeholder view.

    Accepted values are IT and OT only.
    This function does not infer stakeholder view from actions or predicates.
    """
    value = str(value or "").strip().upper()

    if value in ["IT", "OT"]:
        return value

    return ""


def extract_declared_stakeholder_from_text(input_text: str) -> str:
    """
    Extracts the stakeholder view explicitly declared by the user.

    Supported query-file formats include:

    Stakeholder: IT
    Stakeholder: OT
    Stakeholder View: IT
    View: OT
    Perspective: IT
    Role: OT

    The function may also read explicit phrases such as
    "IT stakeholder" or "OT stakeholder" if exactly one appears.

    It intentionally does not classify or infer the stakeholder from
    the path actions, predicates, or majority vote.
    """

    text = input_text or ""

    # Prefer explicit key-value fields.
    field_patterns = [
        r"(?im)^\s*(?:stakeholder(?:\s*view)?|view|perspective|role)\s*:\s*(IT|OT)\b",
        r"(?im)^\s*(?:stakeholder(?:\s*view)?|view|perspective|role)\s*=\s*(IT|OT)\b",
        r"(?im)^\s*(IT|OT)\s+stakeholder\s*:",
        r"(?im)^\s*\[(IT|OT)\]\s*",
    ]

    matches = []

    for pattern in field_patterns:
        for match in re.finditer(pattern, text):
            label = normalize_declared_stakeholder_label(match.group(1))
            if label:
                matches.append(label)

    # Accept explicit natural-language declaration only when unambiguous.
    phrase_matches = []
    if re.search(r"(?i)\bIT\s+stakeholder\b", text):
        phrase_matches.append("IT")
    if re.search(r"(?i)\bOT\s+stakeholder\b", text):
        phrase_matches.append("OT")
    if re.search(r"(?i)\bstakeholder\s+(?:is|=)\s*IT\b", text):
        phrase_matches.append("IT")
    if re.search(r"(?i)\bstakeholder\s+(?:is|=)\s*OT\b", text):
        phrase_matches.append("OT")

    matches.extend(phrase_matches)
    unique_matches = sorted(set(matches))

    if len(unique_matches) == 1:
        return unique_matches[0]

    if len(unique_matches) > 1:
        raise ValueError(
            "Ambiguous stakeholder declaration. The query mentions both IT and OT "
            "as stakeholder views. Use exactly one line such as 'Stakeholder: IT' "
            "or 'Stakeholder: OT'."
        )

    raise ValueError(
        "Missing explicit stakeholder view in the query file. Add one of these lines "
        "before the query or steps:\n"
        "  Stakeholder: IT\n"
        "  Stakeholder: OT\n"
        "The pipeline no longer infers IT/OT from the actions or majority vote."
    )


def parse_user_txt(input_text: str) -> Dict[str, Any]:
    """
    Expected input style:

    Stakeholder: IT

    Query:
    Why does this attack-fault path lead to flare flameout?

    Steps:
    1. Exploit CVE...
    2. Access PLC port...

    The stakeholder view is required and must be explicitly declared as IT or OT.
    The pipeline does not infer the stakeholder view from the path.
    """

    declared_stakeholder = extract_declared_stakeholder_from_text(input_text)

    query = input_text.strip()
    steps = []

    query_match = re.search(
        r"query\s*:\s*(.*?)(?:steps\s*:|actions\s*:|plan\s*:|$)",
        input_text,
        flags=re.IGNORECASE | re.DOTALL
    )

    steps_match = re.search(
        r"(?:steps|actions|plan)\s*:\s*(.*)",
        input_text,
        flags=re.IGNORECASE | re.DOTALL
    )

    if query_match:
        query = query_match.group(1).strip()

    if steps_match:
        raw_steps = steps_match.group(1).strip()
        lines = [line.strip() for line in raw_steps.splitlines() if line.strip()]

        for line in lines:
            line = re.sub(r"^\d+[\.]\s*", "", line)
            line = re.sub(r"^\d+[\)]\s*", "", line)
            line = re.sub(r"^-+\s*", "", line)
            steps.append(line)

    return {
        "query": query,
        "steps": steps,
        "declared_stakeholder": declared_stakeholder,
        "stakeholder": declared_stakeholder,
        "stakeholder_source": "declared_in_query_file",
        "stakeholder_inference_used": False,
        "raw_text": input_text.strip()
    }


In [ ]:
# ============================================================
# Cell 4. PDDL parsing helpers
# ============================================================

def find_matching_paren(text: str, open_index: int) -> int:
    depth = 0

    for i in range(open_index, len(text)):
        if text[i] == "(":
            depth += 1
        elif text[i] == ")":
            depth -= 1

            if depth == 0:
                return i

    return -1


def extract_pddl_actions(domain_text: str) -> Dict[str, str]:
    actions = {}
    pattern = re.compile(r"\(:action\s+([^\s\)]+)", re.IGNORECASE)

    for match in pattern.finditer(domain_text):
        action_name = match.group(1).lower()
        start = match.start()
        end = find_matching_paren(domain_text, start)

        if end == -1:
            print(f"Warning: could not close action {action_name}")
            continue

        actions[action_name] = domain_text[start:end + 1].strip()

    return actions


def extract_pddl_section(action_block: str, section_name: str) -> str:
    match = re.search(rf":{section_name}\s*", action_block, flags=re.IGNORECASE)

    if not match:
        return ""

    start = match.end()

    while start < len(action_block) and action_block[start].isspace():
        start += 1

    if start >= len(action_block):
        return ""

    if action_block[start] == "(":
        end = find_matching_paren(action_block, start)

        if end == -1:
            return action_block[start:].strip()

        return action_block[start:end + 1].strip()

    next_section = re.search(r"\s:\w+", action_block[start:], flags=re.IGNORECASE)

    if next_section:
        end = start + next_section.start()
        return action_block[start:end].strip()

    return action_block[start:].strip()


def extract_action_parameters(action_block: str) -> List[str]:
    match = re.search(
        r":parameters\s*\((.*?)\)",
        action_block,
        flags=re.IGNORECASE | re.DOTALL
    )

    if not match:
        return []

    return re.findall(r"\?[a-zA-Z0-9_-]+", match.group(1))


def extract_predicate_atoms(section_text: str) -> List[str]:
    """
    Lightweight positive-atom extractor.
    """
    if not section_text:
        return []

    atoms = re.findall(r"\([^\(\)]+\)", section_text)
    cleaned = []

    for atom in atoms:
        atom = normalize_atom(atom)

        if atom in ["(and)", "(not)", "()", "(:init)", "(init)"]:
            continue

        if atom.startswith("(:"):
            continue

        if atom.startswith("(="):
            continue

        cleaned.append(atom)

    return cleaned


def ground_atom(atom: str, binding: Dict[str, str]) -> str:
    grounded = atom

    for var in sorted(binding.keys(), key=len, reverse=True):
        grounded = grounded.replace(var, binding[var])

    return normalize_atom(grounded)


def parse_plan_file(plan_file: str) -> List[Dict[str, Any]]:
    parsed = []

    for line in read_text(plan_file).splitlines():
        line = line.strip()

        if not line or line.startswith(";"):
            continue

        line = re.sub(r"^\d+(\.\d+)?\s*:\s*", "", line)
        line = line.strip().strip('"').strip("'")

        if not line.startswith("(") or not line.endswith(")"):
            print("Skipping malformed plan line:", line)
            continue

        inside = line[1:-1].strip()
        parts = inside.split()

        if not parts:
            continue

        parsed.append({
            "raw": normalize_atom(line),
            "action_name": parts[0].lower(),
            "args": [p.lower() for p in parts[1:]]
        })

    return parsed


def read_goal_predicate(goal_file: str) -> str:
    goal = read_text(goal_file).strip()
    goal = goal.strip().strip('"').strip("'")
    return normalize_atom(goal)


def extract_problem_init_section(problem_text: str) -> str:
    match = re.search(r"\(:init", problem_text, flags=re.IGNORECASE)

    if not match:
        return ""

    start = match.start()
    end = find_matching_paren(problem_text, start)

    if end == -1:
        return problem_text[start:].strip()

    return problem_text[start:end + 1].strip()


def extract_initial_state_atoms(problem_text: str) -> Set[str]:
    init = extract_problem_init_section(problem_text)

    if not init:
        return set()

    atoms = re.findall(r"\([^\(\)]+\)", init)
    cleaned = set()

    for atom in atoms:
        atom = normalize_atom(atom)

        if atom in ["(:init)", "(init)", "(and)", "()"]:
            continue

        if atom.startswith("(:"):
            continue

        cleaned.add(atom)

    return cleaned

In [ ]:
def check_plan_action_arity_against_domain(domain_path: str, plan_path: str) -> None:
    domain_text = read_text(domain_path)
    domain_actions = extract_pddl_actions(domain_text)
    plan_actions = parse_plan_file(plan_path)

    errors = []

    for step_num, plan_action in enumerate(plan_actions, start=1):
        action_name = plan_action["action_name"]
        args = plan_action["args"]

        if action_name not in domain_actions:
            errors.append(
                f"Step {step_num}: action '{action_name}' not found in domain."
            )
            continue

        action_block = domain_actions[action_name]
        params = extract_action_parameters(action_block)

        if len(args) != len(params):
            errors.append(
                f"Step {step_num}: {action_name} expects {len(params)} argument(s) "
                f"{params}, but plan gives {len(args)} argument(s): {args}"
            )

    if errors:
        print("Plan/domain mismatch found:")
        for e in errors:
            print(" -", e)
        raise ValueError("Plan action arity does not match domain action signatures.")

    print("All mapped plan actions match the domain action signatures.")

In [ ]:
# ============================================================
# Grounded plan validation helpers
# ============================================================

def extract_problem_objects(problem_path: str) -> List[str]:
    """
    Extract object names from the (:objects ...) section of a PDDL problem file.
    """

    problem_text = read_text(problem_path)

    match = re.search(
        r"\(:objects\s+(.*?)\)",
        problem_text,
        flags=re.IGNORECASE | re.DOTALL
    )

    if not match:
        return []

    objects_text = match.group(1)

    # Remove type markers and type names after '-'
    tokens = re.findall(r"[a-zA-Z0-9_\-]+|\-", objects_text)

    objects = []
    skip_type = False

    i = 0
    while i < len(tokens):
        token = tokens[i].lower()

        if token == "-":
            # Skip the type name after '-'
            i += 2
            continue

        objects.append(token)
        i += 1

    # Remove common PDDL keywords accidentally captured
    bad = {"objects", "object", "node"}
    objects = [x for x in objects if x not in bad]

    return sorted(set(objects))


def extract_constants_from_problem(problem_path: str) -> List[str]:
    """
    Extract likely object constants from both objects and init atoms.
    This helps when objects are not explicitly listed in (:objects ...).
    """

    problem_text = read_text(problem_path)

    objects = set(extract_problem_objects(problem_path))

    init_atoms = extract_initial_state_atoms(problem_text)

    for atom in init_atoms:
        inside = atom.strip()[1:-1].strip()
        parts = inside.split()

        # Skip predicate name, keep arguments
        for arg in parts[1:]:
            if not arg.startswith("?"):
                objects.add(arg.lower())

    return sorted(objects)


def contains_ungrounded_variables_in_line(line: str) -> bool:
    """
    Detects variables like ?plc, ?at, ?to.
    """

    return bool(re.search(r"\?[a-zA-Z0-9_\-]+", line))

def verify_mapped_plan_file_is_grounded(plan_file: str) -> None:
    lines = read_text(plan_file).splitlines()

    bad = []

    for line in lines:
        line = line.strip()

        if not line:
            continue

        # Skip comments if any
        if line.startswith(";"):
            continue

        if contains_ungrounded_variables_in_line(line):
            bad.append(line)

    if bad:
        print("Ungrounded actions found in mapped plan:")
        for b in bad:
            print("  -", b)

        raise ValueError(
            "mapped_plan_fragments.txt contains variables. "
            "The mapping stage must output fully grounded actions."
        )

    print("mapped_plan_fragments.txt is fully grounded.")


def assert_plan_fragments_are_grounded(fragments: List[str]) -> None:
    """
    Raises an error if any mapped fragment contains variables.
    """

    bad = []

    for frag in fragments:
        if contains_ungrounded_variables_in_line(frag):
            bad.append(frag)

    if bad:
        message = "Mapped plan contains ungrounded variables:\n"
        message += "\n".join(f"  - {x}" for x in bad)
        message += (
            "\n\nEvery mapped action must use concrete problem objects, "
            "not variables such as ?at, ?to, or ?plc."
        )
        raise ValueError(message)


def clean_mapped_fragments(fragments: List[str]) -> List[str]:
    """
    Normalize mapped plan fragments and keep only action lines.
    """

    cleaned = []

    for frag in fragments:
        frag = str(frag).strip().lower()
        frag = frag.strip('"').strip("'")

        if not frag:
            continue

        if not frag.startswith("(") or not frag.endswith(")"):
            continue

        frag = re.sub(r"\s+", " ", frag)
        cleaned.append(frag)

    return cleaned

In [ ]:
# ============================================================
# Declared stakeholder-view helpers
# ============================================================

def get_declared_stakeholder_from_user_data(user_data: Dict[str, Any]) -> str:
    """
    Returns the stakeholder view explicitly declared in the query file.
    No action-based or keyword-based inference is performed.
    """

    stakeholder = normalize_declared_stakeholder_label(
        user_data.get("declared_stakeholder") or user_data.get("stakeholder")
    )

    if stakeholder not in ["IT", "OT"]:
        raise ValueError(
            "The query file must explicitly declare exactly one stakeholder view: IT or OT."
        )

    return stakeholder


def validation_start_node_from_stakeholder(stakeholder: str) -> str:
    stakeholder = normalize_declared_stakeholder_label(stakeholder)

    if stakeholder == "IT":
        return "MIT3"

    if stakeholder == "OT":
        return "MOT2"

    raise ValueError(f"Unsupported stakeholder view: {stakeholder}. Expected IT or OT.")


def start_node_from_stakeholder(stakeholder: str) -> str:
    return validation_start_node_from_stakeholder(stakeholder)


In [ ]:
# ============================================================
# PDDL action utility helpers
# ============================================================

def get_pddl_action_name(fragment: str) -> str:
    fragment = str(fragment).strip().lower()
    fragment = fragment.strip("()")
    if not fragment:
        return ""
    return fragment.split()[0]

# Stakeholder-view inference from action fragments has intentionally been removed.
# The pipeline now uses only the IT/OT stakeholder view explicitly declared in
# the query file.


In [ ]:
# ============================================================
# Cell 5. Stage 1: Declared stakeholder view + action mapping
# ============================================================

def summarize_domain_actions_for_prompt(domain_path: str) -> str:
    domain_text = read_text(domain_path)
    actions = extract_pddl_actions(domain_text)

    lines = []

    for action_name, block in actions.items():
        params = extract_action_parameters(block)
        pre = extract_pddl_section(block, "precondition")
        eff = extract_pddl_section(block, "effect")

        lines.append(f"Action: {action_name}")
        lines.append(f"Parameters: {' '.join(params)}")
        lines.append(f"Precondition: {pre}")
        lines.append(f"Effect: {eff}")
        lines.append("")

    return "\n".join(lines)


def generate_stakeholder_and_plan_mapping(
    input_txt: str,
    domain_pddl: str,
    output_json: str,
    output_plan_txt: str,
    generator
) -> Dict[str, Any]:
    """
    Uses the explicitly declared IT/OT stakeholder view from the query file and
    uses the LLM only to map user steps to fully grounded PDDL plan fragments.

    The LLM does not infer, predict, vote on, or override the stakeholder view.

    Critical:
    mapped_plan_fragments must contain concrete objects, not variables.
    """

    user_data = parse_user_txt(read_text(input_txt))
    declared_stakeholder = get_declared_stakeholder_from_user_data(user_data)

    domain_summary = summarize_domain_actions_for_prompt(domain_pddl)

    # Use objects from the FULL problem file
    available_objects = extract_constants_from_problem(FULL_PROBLEM_PDDL)

    prompt = f"""
You map user-described CPS attack-fault steps to existing PDDL actions.

The stakeholder view is already declared by the user.
Declared stakeholder view: {declared_stakeholder}

Do NOT infer the stakeholder view.
Do NOT change the stakeholder view.
Do NOT classify the user as IT or OT.
Use the declared stakeholder view only as context for preserving the user's perspective;
your task is only to map each user step to a fully grounded PDDL action.

Critical grounding and argument-order rules:
- mapped_plan_fragments must be fully grounded PDDL actions.
- Do NOT output variables such as ?at, ?to, ?plc, ?node, ?x, or ?y.
- Every action argument must be a concrete object from the problem file.
- Use the exact number of arguments required by the PDDL action schema.
- Do not omit required arguments.
- Do not swap source and target objects.
- For exploits-vulnerability-cve-2018-0296, use:
  (exploits-vulnerability-cve-2018-0296 microsoft-windows-12-server internal-network)
- For access-to-plc-port-1132-tcp, use:
  (access-to-plc-port-1132-tcp internal-network allen-bradley-controllogix-plc)
- For exploits-vulnerability-cve-2017-9312, use:
  (exploits-vulnerability-cve-2017-9312 internal-network allen-bradley-controllogix-plc)
- For process-fault actions, use the PLC object:
  allen-bradley-controllogix-plc

Bad examples:
- (exploits-vulnerability-cve-2018-0296 internal-network allen-bradley-controllogix-plc)
- (exploits-vulnerability-cve-2017-9312 allen-bradley-controllogix-plc)
- (access-to-plc-port-1132-tcp ?plc)

Good examples:
- (exploits-vulnerability-cve-2018-0296 microsoft-windows-12-server internal-network)
- (access-to-plc-port-1132-tcp internal-network allen-bradley-controllogix-plc)
- (exploits-vulnerability-cve-2017-9312 internal-network allen-bradley-controllogix-plc)

Return only valid JSON.

User declared stakeholder view:
{declared_stakeholder}

User query:
{user_data["query"]}

User steps:
{json.dumps(user_data["steps"], indent=2)}

Available problem objects:
{json.dumps(available_objects, indent=2)}

Available PDDL actions:
{domain_summary}

Return JSON with this exact schema:
{{
  "declared_stakeholder": "{declared_stakeholder}",
  "stakeholder_source": "declared_in_query_file",
  "mapped_plan_fragments": [
    "(fully-grounded-action concrete-object-1 concrete-object-2)"
  ],
  "step_action_mapping": [
    {{
      "user_step": "original user step",
      "pddl_action": "(fully-grounded-action concrete-object-1 concrete-object-2)",
      "justification": "why this grounded action matches"
    }}
  ]
}}
""".strip()

    output = generator(prompt)[0]["generated_text"]
    parsed = extract_json_from_text(output)

    fragments = parsed.get("mapped_plan_fragments", [])
    fragments = clean_mapped_fragments(fragments)

    # Enforce the user-declared stakeholder view deterministically.
    # Downstream code keeps predicted_stakeholder as a backward-compatible key,
    # but its value is not predicted; it is copied from the declared query field.
    llm_echoed_stakeholder = normalize_declared_stakeholder_label(
        parsed.get("declared_stakeholder") or parsed.get("predicted_stakeholder")
    )

    parsed["declared_stakeholder"] = declared_stakeholder
    parsed["stakeholder"] = declared_stakeholder
    parsed["stakeholder_source"] = "declared_in_query_file"
    parsed["stakeholder_inference_used"] = False
    parsed["stakeholder_override_used"] = False
    parsed["llm_echoed_stakeholder"] = llm_echoed_stakeholder or None
    parsed["llm_predicted_stakeholder"] = None

    # Backward-compatible key used by older parts of the notebook.
    parsed["predicted_stakeholder"] = declared_stakeholder

    parsed["validation_start_node"] = validation_start_node_from_stakeholder(
        declared_stakeholder
    )

    if llm_echoed_stakeholder and llm_echoed_stakeholder != declared_stakeholder:
        parsed["stakeholder_echo_warning"] = (
            f"LLM echoed {llm_echoed_stakeholder}, but the declared query stakeholder "
            f"is {declared_stakeholder}. The pipeline used the declared value."
        )

    # Critical check: fail if the LLM still used ?variables
    assert_plan_fragments_are_grounded(fragments)

    parsed["mapped_plan_fragments"] = fragments

    # Also clean step_action_mapping if present
    cleaned_step_mapping = []
    for item in parsed.get("step_action_mapping", []):
        pddl_action = str(item.get("pddl_action", "")).strip().lower()

        if pddl_action:
            if contains_ungrounded_variables_in_line(pddl_action):
                raise ValueError(
                    f"Ungrounded pddl_action in step_action_mapping: {pddl_action}"
                )

        cleaned_step_mapping.append(item)

    parsed["step_action_mapping"] = cleaned_step_mapping

    write_json(output_json, parsed)

    if fragments:
        write_text(output_plan_txt, "\n".join(fragments))
    else:
        write_text(output_plan_txt, "")

    print("Declared stakeholder view:", declared_stakeholder)
    print("Mapped plan fragments:")
    for frag in fragments:
        print(" ", frag)

    return parsed


In [ ]:
def extract_json_object_robust(text: str) -> Dict[str, Any]:
    """
    Extracts the first valid JSON object from messy LLM output.

    Handles outputs like:
      { ... } explanation text ```json { ... } ```
    """
    text = text.strip()

    # Try direct JSON parsing first.
    try:
        return json.loads(text)
    except Exception:
        pass

    # Try fenced JSON blocks.
    fenced_matches = re.findall(
        r"```(?:json)?\s*(\{.*?\})\s*```",
        text,
        flags=re.DOTALL | re.IGNORECASE
    )

    for block in fenced_matches:
        try:
            return json.loads(block)
        except Exception:
            continue

    # Try extracting the first balanced JSON object.
    starts = [i for i, ch in enumerate(text) if ch == "{"]

    for start in starts:
        depth = 0

        for end in range(start, len(text)):
            if text[end] == "{":
                depth += 1

            elif text[end] == "}":
                depth -= 1

                if depth == 0:
                    candidate = text[start:end + 1]

                    try:
                        return json.loads(candidate)
                    except Exception:
                        break

    raise ValueError("Could not parse JSON from LLM output.")


def normalize_path_intent(intent: str) -> str:
    """
    Normalizes LLM labels into the exact labels used by the pipeline.
    """
    if intent is None:
        return "UNKNOWN"

    value = str(intent).strip().upper().replace("-", "_").replace(" ", "_")

    existence_labels = {
        "EXIST",
        "EXISTS",
        "EXISTENCE",
        "VALID",
        "PATH_EXISTS",
        "REACHABLE",
        "YES"
    }

    non_existence_labels = {
        "NON_EXISTENCE",
        "NONEXISTENCE",
        "NON_EXIST",
        "NOT_EXIST",
        "NOT_EXISTS",
        "DOES_NOT_EXIST",
        "INVALID",
        "UNREACHABLE",
        "NO"
    }

    if value in existence_labels:
        return "EXISTENCE"

    if value in non_existence_labels:
        return "NON_EXISTENCE"

    return "UNKNOWN"

In [ ]:
# ============================================================
# Cell 6. Stage 2: Path intent classification
# ============================================================

def classify_path_existence_intent(
    input_txt: str,
    output_json: str,
    generator
) -> Dict[str, Any]:
    user_data = parse_user_txt(read_text(input_txt))

    prompt = f"""
Classify whether the user is asking about path existence or path non-existence.

Definitions:
- EXISTENCE: The user asks why a path exists, why it is valid, why it reaches the goal, or how the steps lead to the outcome.
- NON_EXISTENCE: The user asks why a path does not exist, why it is invalid, why it fails, or why it cannot reach the goal.

Allowed intent labels:
- EXISTENCE
- NON_EXISTENCE

Important:
- Return only valid JSON.
- Do not include markdown.
- Do not include explanation outside JSON.
- Do not use labels such as EXISTS, EXIST, VALID, INVALID, YES, or NO.

User query:
{user_data["query"]}

User steps:
{json.dumps(user_data["steps"], indent=2)}

Return JSON with exactly this schema:
{{
  "intent": "EXISTENCE or NON_EXISTENCE",
  "confidence": "low, medium, or high",
  "evidence": [
    "short evidence from query"
  ]
}}
""".strip()

    output = generator(prompt)[0]["generated_text"]

    try:
        parsed = extract_json_object_robust(output)

        raw_intent = parsed.get("intent")
        normalized_intent = normalize_path_intent(raw_intent)

        result = {
            "intent": normalized_intent,
            "raw_intent": raw_intent,
            "confidence": parsed.get("confidence", "unknown"),
            "evidence": parsed.get("evidence", []),
            "raw_output": output
        }

    except Exception as e:
        result = {
            "error": str(e),
            "raw_output": output,
            "intent": "UNKNOWN",
            "raw_intent": None,
            "confidence": "unknown",
            "evidence": []
        }

    write_json(output_json, result)

    return result

In [ ]:
# ============================================================
# Cell 7. Stage 3: Goal predicate inference
# ============================================================

def infer_goal_predicate_with_llm(
    input_txt: str,
    domain_pddl: str,
    problem_pddl: str,
    output_goal_txt: str,
    output_json: str,
    generator
) -> Dict[str, Any]:

    user_data = parse_user_txt(read_text(input_txt))
    domain_text = read_text(domain_pddl)

    problem_objects = extract_constants_from_problem(problem_pddl)

    actions = extract_pddl_actions(domain_text)

    action_effects = []
    for action_name, block in actions.items():
        eff = extract_pddl_section(block, "effect")
        action_effects.append(f"Action: {action_name}\nEffect: {eff}")

    prompt = f"""
Infer the final goal predicate from the user's query and steps.

Use only predicates/effects that appear in the domain evidence.
Return only one grounded PDDL atom.

Important:
- Output one goal predicate.
- Do not invent predicates.
- Do not use variables such as ?plc, ?node, ?at, or ?to.
- Every argument must be one of the problem objects listed below.
- If the final action has a zero-argument effect, use the zero-argument predicate exactly.
- Return JSON only.

Problem objects:
{json.dumps(problem_objects, indent=2)}

User query:
{user_data["query"]}

User steps:
{json.dumps(user_data["steps"], indent=2)}

Domain action effects:
{chr(10).join(action_effects)}

Return JSON:
{{
  "goal_predicate": "(predicate concrete-object-if-needed)",
  "justification": "brief explanation"
}}
""".strip()

    output = generator(prompt)[0]["generated_text"]
    parsed = extract_json_from_text(output)

    raw_goal = parsed.get("goal_predicate", "").strip()

    grounded_goal = ground_goal_with_problem_objects(
        goal_text=raw_goal,
        problem_path=problem_pddl
    )

    parsed["goal_predicate_raw"] = raw_goal
    parsed["goal_predicate"] = grounded_goal
    parsed["problem_objects_used_for_grounding"] = problem_objects

    write_text(output_goal_txt, grounded_goal)
    write_json(output_json, parsed)

    return parsed

In [ ]:
# ============================================================
# Cell 7B. Update and restore lattice problem goals
# Replaces the goal only when the inferred goal is different
# from the current goal in each problem file.
# ============================================================

def find_matching_paren(text: str, start_index: int) -> int:
    """
    Return the index of the matching closing parenthesis for text[start_index].
    """
    if start_index < 0 or start_index >= len(text) or text[start_index] != "(":
        raise ValueError("find_matching_paren must start at an opening parenthesis.")

    depth = 0

    for i in range(start_index, len(text)):
        if text[i] == "(":
            depth += 1
        elif text[i] == ")":
            depth -= 1
            if depth == 0:
                return i

    raise ValueError("Could not find matching closing parenthesis.")


def compact_pddl(text: str) -> str:
    """
    Canonical whitespace/lowercase form for comparison only.
    """
    text = text.strip().lower()
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"\(\s+", "(", text)
    text = re.sub(r"\s+\)", ")", text)
    return text.strip()


def strip_code_fences_and_quotes(text: str) -> str:
    text = str(text).strip()
    text = text.replace("```lisp", "")
    text = text.replace("```pddl", "")
    text = text.replace("```", "")
    text = text.strip().strip('"').strip("'").strip()
    return text


def remove_redundant_single_wrapping(expr: str) -> str:
    """
    Converts accidental ((p a)) into (p a).
    Does not remove meaningful wrappers such as (and (p a) (q b)).
    """
    expr = expr.strip()

    changed = True
    while changed:
        changed = False
        expr = expr.strip()

        if expr.startswith("((") and expr.endswith("))"):
            candidate = expr[1:-1].strip()

            if candidate.startswith("("):
                try:
                    end = find_matching_paren(candidate, 0)
                    if end == len(candidate) - 1:
                        expr = candidate
                        changed = True
                except Exception:
                    pass

    return expr


def split_top_level_s_exprs(text: str) -> List[str]:
    """
    Splits a string containing top-level PDDL expressions.
    Example: '(p a) (q b)' -> ['(p a)', '(q b)']
    """
    items = []
    i = 0

    while i < len(text):
        while i < len(text) and text[i].isspace():
            i += 1

        if i >= len(text):
            break

        if text[i] == "(":
            end = find_matching_paren(text, i)
            items.append(text[i:end + 1].strip())
            i = end + 1
        else:
            j = i
            while j < len(text) and not text[j].isspace():
                j += 1
            items.append(text[i:j].strip())
            i = j

    return [x for x in items if x]


def normalize_goal_predicate(goal_text: str) -> str:
    """
    Normalize a goal so it can be inserted inside a problem file.

    Accepts:
      (p a)
      (and (p a))
      (:goal (p a))
      (:goal (and (p a)))
      (:goal (and ((p a))))

    Returns a formula without the outer :goal wrapper:
      (p a)
      or
      (and (p a) (q b))
    """
    goal_text = strip_code_fences_and_quotes(goal_text)

    if not goal_text:
        raise ValueError("Goal text is empty.")

    # If a full (:goal ...) block was passed, remove the outer :goal wrapper.
    if goal_text.lower().startswith("(:goal"):
        end = find_matching_paren(goal_text, 0)
        outer_content = goal_text[1:end].strip()  # ':goal ...'
        goal_text = re.sub(
            r"^:goal\b",
            "",
            outer_content,
            flags=re.IGNORECASE
        ).strip()

    if not goal_text.startswith("("):
        goal_text = f"({goal_text})"

    goal_text = remove_redundant_single_wrapping(goal_text)

    # Fix accidental (and ((p a))) for singleton goals.
    if goal_text.lower().startswith("(and"):
        end = find_matching_paren(goal_text, 0)
        outer_content = goal_text[1:end].strip()  # 'and ...'
        rest = re.sub(r"^and\b", "", outer_content, flags=re.IGNORECASE).strip()
        children = [remove_redundant_single_wrapping(x) for x in split_top_level_s_exprs(rest)]

        if len(children) == 1:
            goal_text = children[0]
        elif len(children) > 1:
            goal_text = "(and " + " ".join(children) + ")"
        else:
            raise ValueError(f"Malformed goal formula: {goal_text}")

    goal_text = re.sub(r"\s+", " ", goal_text).strip()

    if contains_ungrounded_variables_in_line(goal_text):
        raise ValueError(
            f"Inferred goal is not grounded: {goal_text}\n"
            "Problem goals cannot contain variables such as ?plc, ?node, ?at, or ?to."
        )

    return goal_text.lower()


def extract_current_goal_block(problem_text: str) -> str:
    """
    Extract the full (:goal ...) block from a PDDL problem file.
    """
    match = re.search(r"\(\s*:goal\b", problem_text, flags=re.IGNORECASE)

    if not match:
        raise ValueError("No (:goal ...) block found in problem file.")

    goal_start = match.start()
    goal_end = find_matching_paren(problem_text, goal_start)

    return problem_text[goal_start:goal_end + 1]


def canonicalize_goal_for_compare(goal_text: str) -> str:
    """
    Canonical form used only for equality checking.

    This makes the following compare as equal:
      (:goal (and (p a)))
      (:goal (p a))
      (and (p a))
      (p a)
      ((p a))
    """
    normalized = normalize_goal_predicate(goal_text)
    return compact_pddl(normalized)


def replace_problem_goal(problem_text: str, new_goal_predicate: str) -> str:
    """
    Replace the (:goal ...) block with the normalized inferred goal.
    """
    match = re.search(r"\(\s*:goal\b", problem_text, flags=re.IGNORECASE)

    if not match:
        raise ValueError("No (:goal ...) block found in problem file.")

    goal_start = match.start()
    goal_end = find_matching_paren(problem_text, goal_start)

    new_goal_formula = normalize_goal_predicate(new_goal_predicate)

    new_goal_block = f"""(:goal
    {new_goal_formula}
  )"""

    return problem_text[:goal_start] + new_goal_block + problem_text[goal_end + 1:]


def get_all_lattice_problem_files(lattice_data: Dict[str, Any], lattice_json_path: str) -> List[str]:
    problem_files = []

    for node_id, node_info in lattice_data.get("nodes", {}).items():
        problem_path = node_info.get("problem")

        if problem_path:
            resolved_path = resolve_path(problem_path, lattice_json_path)
            problem_files.append(resolved_path)

    seen = set()
    unique_files = []

    for path in problem_files:
        if path not in seen:
            seen.add(path)
            unique_files.append(path)

    return unique_files


def make_backup_path(problem_path: str, backup_dir: str) -> str:
    Path(backup_dir).mkdir(parents=True, exist_ok=True)

    abs_path = str(Path(problem_path).resolve())
    path_hash = hashlib.sha1(abs_path.encode("utf-8")).hexdigest()[:12]
    backup_name = f"{path_hash}_{Path(problem_path).name}.bak"

    return str(Path(backup_dir) / backup_name)


def backup_problem_file(problem_path: str, backup_dir: str) -> str:
    """
    Always copy the current file before changing it.
    This prevents stale backups from older runs.
    """
    backup_path = make_backup_path(problem_path, backup_dir)
    shutil.copy(problem_path, backup_path)
    return backup_path


def update_all_lattice_problem_goals(
    lattice_json_path: str,
    goal_file: str,
    backup_dir: str,
    report_txt_path: str
) -> Dict[str, Any]:
    """
    Update every lattice problem file only if its current goal differs
    from the inferred goal.
    """

    lattice_data = read_json(lattice_json_path)

    raw_goal = read_text(goal_file).strip()
    new_goal_predicate = normalize_goal_predicate(raw_goal)
    new_goal_norm = canonicalize_goal_for_compare(new_goal_predicate)

    problem_files = get_all_lattice_problem_files(
        lattice_data=lattice_data,
        lattice_json_path=lattice_json_path
    )

    updated_files = []
    skipped_files = []
    failed_files = []

    for problem_path in problem_files:
        if not os.path.exists(problem_path):
            failed_files.append({
                "problem": problem_path,
                "reason": "file does not exist"
            })
            continue

        try:
            old_text = read_text(problem_path)

            current_goal_block = extract_current_goal_block(old_text)
            current_goal_norm = canonicalize_goal_for_compare(current_goal_block)

            if current_goal_norm == new_goal_norm:
                skipped_files.append({
                    "problem": problem_path,
                    "reason": "goal already matches inferred goal",
                    "current_goal": current_goal_norm
                })
                continue

            backup_path = backup_problem_file(
                problem_path=problem_path,
                backup_dir=backup_dir
            )

            new_text = replace_problem_goal(
                problem_text=old_text,
                new_goal_predicate=new_goal_predicate
            )

            write_text(problem_path, new_text)

            updated_files.append({
                "problem": problem_path,
                "backup": backup_path,
                "old_goal": current_goal_norm,
                "new_goal": new_goal_norm
            })

        except Exception as e:
            failed_files.append({
                "problem": problem_path,
                "reason": str(e)
            })

    lines = []
    lines.append("=== GOAL UPDATE REPORT ===")
    lines.append("")
    lines.append(f"New goal predicate: {new_goal_predicate}")
    lines.append(f"Updated files: {len(updated_files)}")
    lines.append(f"Skipped files: {len(skipped_files)}")
    lines.append(f"Failed files: {len(failed_files)}")
    lines.append("")

    for item in updated_files:
        lines.append(f"Updated: {item['problem']}")
        lines.append(f"Backup:  {item['backup']}")
        lines.append(f"Old goal: {item['old_goal']}")
        lines.append(f"New goal: {item['new_goal']}")
        lines.append("")

    for item in skipped_files:
        lines.append(f"Skipped: {item['problem']}")
        lines.append(f"Reason:  {item['reason']}")
        lines.append(f"Goal:    {item['current_goal']}")
        lines.append("")

    for item in failed_files:
        lines.append(f"Failed: {item['problem']}")
        lines.append(f"Reason: {item['reason']}")
        lines.append("")

    write_text(report_txt_path, "\n".join(lines))

    print("\nChecked all lattice problem goals.")
    print("New goal:", new_goal_predicate)
    print("Updated files:", len(updated_files))
    print("Skipped files:", len(skipped_files))
    print("Failed files:", len(failed_files))
    print("Report:", report_txt_path)

    return {
        "new_goal_predicate": new_goal_predicate,
        "new_goal_norm": new_goal_norm,
        "updated_files": updated_files,
        "skipped_files": skipped_files,
        "failed_files": failed_files,
        "backup_dir": backup_dir,
        "report_txt": report_txt_path
    }


def restore_lattice_problem_goals_from_backup(
    goal_update_result: Dict[str, Any],
    report_txt_path: str
) -> Dict[str, Any]:
    """
    Restore only the files that were actually changed.
    Skipped files are not restored because they were never modified.
    """

    restored_files = []
    failed_files = []
    skipped_files = goal_update_result.get("skipped_files", [])

    for item in goal_update_result.get("updated_files", []):
        problem_path = item.get("problem")
        backup_path = item.get("backup")

        if not problem_path or not backup_path:
            failed_files.append({
                "problem": problem_path,
                "backup": backup_path,
                "reason": "missing problem or backup path"
            })
            continue

        if not os.path.exists(backup_path):
            failed_files.append({
                "problem": problem_path,
                "backup": backup_path,
                "reason": "backup file does not exist"
            })
            continue

        try:
            shutil.copy(backup_path, problem_path)

            restored_files.append({
                "problem": problem_path,
                "backup": backup_path
            })

        except Exception as e:
            failed_files.append({
                "problem": problem_path,
                "backup": backup_path,
                "reason": str(e)
            })

    lines = []
    lines.append("=== GOAL RESTORE REPORT ===")
    lines.append("")
    lines.append(f"Restored files: {len(restored_files)}")
    lines.append(f"Skipped files never modified: {len(skipped_files)}")
    lines.append(f"Failed restores: {len(failed_files)}")
    lines.append("")

    for item in restored_files:
        lines.append(f"Restored: {item['problem']}")
        lines.append(f"From:     {item['backup']}")
        lines.append("")

    for item in skipped_files:
        lines.append(f"Not restored because unchanged: {item['problem']}")
        lines.append("")

    for item in failed_files:
        lines.append(f"Failed: {item.get('problem')}")
        lines.append(f"Backup: {item.get('backup')}")
        lines.append(f"Reason: {item.get('reason')}")
        lines.append("")

    write_text(report_txt_path, "\n".join(lines))

    print("\nRestored lattice problem goals from backup.")
    print("Restored files:", len(restored_files))
    print("Skipped files never modified:", len(skipped_files))
    print("Failed restores:", len(failed_files))
    print("Report:", report_txt_path)

    return {
        "restored_files": restored_files,
        "skipped_files": skipped_files,
        "failed_files": failed_files,
        "report_txt": report_txt_path
    }

In [ ]:
# ============================================================
# Cell 9. Intent and declared stakeholder readers
# ============================================================

def find_path_intent_from_json(json_path: str) -> Dict[str, Any]:
    data = read_json(json_path)

    if "intent" in data:
        return {
            "intent": normalize_path_intent(data.get("intent", "UNKNOWN")),
            "confidence": data.get("confidence", "unknown"),
            "evidence": data.get("evidence", [])
        }

    if "path_existence_intent" in data:
        x = data["path_existence_intent"]
        return {
            "intent": normalize_path_intent(data.get("intent", "UNKNOWN")),
            "confidence": x.get("confidence", "unknown"),
            "evidence": x.get("evidence", [])
        }

    return {
        "intent": "UNKNOWN",
        "confidence": "low",
        "evidence": []
    }


def find_declared_stakeholder_from_json(json_path: str) -> str:
    """
    Reads the stakeholder view declared in the query file and saved in the
    preprocessing JSON.

    This function intentionally does not infer or default to IT/OT. If the
    declared view is missing, the pipeline stops with a clear error.
    """

    data = read_json(json_path)

    stakeholder = normalize_declared_stakeholder_label(
        data.get("declared_stakeholder") or data.get("stakeholder")
    )

    # Backward-compatible fallback for older JSON files generated before this
    # update. This is still treated as declared only if the JSON explicitly says
    # the value came from the query file.
    if stakeholder not in ["IT", "OT"]:
        source = data.get("stakeholder_source")
        old_value = normalize_declared_stakeholder_label(data.get("predicted_stakeholder"))
        if source == "declared_in_query_file" and old_value in ["IT", "OT"]:
            stakeholder = old_value

    if stakeholder not in ["IT", "OT"]:
        raise ValueError(
            f"Declared stakeholder view not found in {json_path}. Re-run preprocessing "
            "after adding 'Stakeholder: IT' or 'Stakeholder: OT' to the query file."
        )

    return stakeholder


# Backward-compatible alias. The value returned is declared, not predicted.
def find_predicted_stakeholder_from_json(json_path: str) -> str:
    return find_declared_stakeholder_from_json(json_path)


In [ ]:
# ============================================================
# Cell 10. Lattice helpers
# ============================================================

def get_node_level(node_id: str, lattice_data: Dict[str, Any]) -> int:
    node_info = lattice_data.get("nodes", {}).get(node_id, {})
    label = str(node_info.get("label", ""))

    m = re.search(r"L(\d+)", label, re.IGNORECASE)
    if m:
        return int(m.group(1))

    if node_id == "M_ABS":
        return 99

    if node_id == "FULL":
        return 0

    m = re.search(r"(\d+)", node_id)
    if m:
        return int(m.group(1))

    return -1


def infer_branch_prefix(start_node: str) -> Optional[str]:
    if start_node.startswith("MIT"):
        return "MIT"

    if start_node.startswith("MOT"):
        return "MOT"

    return None


def node_matches_branch(node_id: str, branch_prefix: Optional[str]) -> bool:
    if branch_prefix is None:
        return True

    if node_id in ["FULL", "M_ABS"]:
        return True

    return node_id.startswith(branch_prefix)


def build_edge_maps(lattice_data: Dict[str, Any]) -> Dict[str, Dict[str, List[str]]]:
    nodes = lattice_data.get("nodes", {})
    edges = lattice_data.get("edges", [])

    children = {node_id: [] for node_id in nodes.keys()}
    parents = {node_id: [] for node_id in nodes.keys()}

    for edge in edges:
        src = edge.get("from")
        dst = edge.get("to")

        if src and dst:
            children.setdefault(src, []).append(dst)
            parents.setdefault(dst, []).append(src)

    return {
        "children": children,
        "parents": parents
    }


def get_neighbors_from_lattice_json(
    lattice_data: Dict[str, Any],
    node_id: str,
    direction: str
) -> List[str]:
    maps = build_edge_maps(lattice_data)

    if direction == "down":
        return maps["children"].get(node_id, [])

    if direction == "up":
        return maps["parents"].get(node_id, [])

    raise ValueError("direction must be 'down' or 'up'")


def is_bridge_node(node_id: str, lattice_data: Dict[str, Any]) -> bool:
    node_info = lattice_data.get("nodes", {}).get(node_id, {})
    return node_info.get("node_type") == "bridge"


def get_edge_type(lattice_data: Dict[str, Any], src: str, dst: str) -> str:
    for edge in lattice_data.get("edges", []):
        if edge.get("from") == src and edge.get("to") == dst:
            return edge.get("edge_type", "")
    return ""


def bridge_target_matches(
    node_id: str,
    target_branch: Optional[str],
    lattice_data: Dict[str, Any]
) -> bool:
    """
    Keeps bridge nodes whose target node belongs to the desired branch.

    Example:
        MOT2-MIT3 has target_node MIT3.
        If target_branch is MIT, this bridge is allowed.
    """
    if target_branch is None:
        return True

    node_info = lattice_data.get("nodes", {}).get(node_id, {})

    if node_info.get("node_type") != "bridge":
        return True

    target_node = node_info.get("target_node", "")

    return str(target_node).startswith(target_branch)

def bridge_sort_key(node_id: str, lattice_data: Dict[str, Any]):
    """
    Sort bridge nodes so that:
        MOT2-MIT3, MOT2-MIT2, MOT2-MIT1, MOT2-MIT0

    come before refinement nodes such as:
        MOT1

    For non-bridge nodes, use a lower priority after bridge nodes.
    """

    node_info = lattice_data.get("nodes", {}).get(node_id, {})

    if node_info.get("node_type") == "bridge":
        target_node = node_info.get("target_node", "")

        # Extract number from target node, e.g., MIT3 -> 3
        match = re.search(r"(\d+)", target_node)
        target_level = int(match.group(1)) if match else -1

        # First value 0 means bridge nodes come first.
        # Negative target_level gives descending order:
        # MIT3 before MIT2 before MIT1 before MIT0.
        return (0, -target_level, node_id)

    # Non-bridge refinement nodes come after bridge nodes.
    return (1, get_node_level(node_id, lattice_data), node_id)

def derive_downward_chain_from_lattice_json(
    lattice_data: Dict[str, Any],
    start_node: str,
    branch_prefix: Optional[str] = None,
    include_bridge_nodes: bool = True,
    include_full_model: bool = False
) -> List[str]:
    """
    Bridge-aware search order.

    This replaces the old linear chain builder.

    Old behavior:
        MOT2 -> MOT1 -> MOT0 -> FULL

    Current behavior:
        MOT2 -> MOT2-MIT0 -> MOT2-MIT1 -> MOT2-MIT2 -> MOT2-MIT3
             -> MOT1 -> MOT1-MIT0 ... -> MOT0 ...

    By default, FULL is excluded from the explanation search chain. The FULL
    concrete model may still be used by the repair module, but it is not used
    as a validity/invalidity explanation-search node.

    The exact order depends on the lattice edges.
    """

    nodes = lattice_data.get("nodes", {})

    if start_node not in nodes:
        raise ValueError(f"Start node not found in lattice.json: {start_node}")

    if branch_prefix is None:
        branch_prefix = infer_branch_prefix(start_node)

    # If we start from OT, bridge nodes should target IT.
    # If we start from IT, bridge nodes should target OT.
    if branch_prefix == "MOT":
        target_branch = "MIT"
    elif branch_prefix == "MIT":
        target_branch = "MOT"
    else:
        target_branch = None

    children = build_edge_maps(lattice_data)["children"]

    visited = set()
    order = []
    queue = deque([start_node])

    while queue:
        current = queue.popleft()

        if current in visited:
            continue

        visited.add(current)
        order.append(current)

        candidates = children.get(current, [])
        eligible = []

        for child in candidates:
            if child in visited:
                continue

            edge_type = get_edge_type(lattice_data, current, child)

            # Do not include FULL in the explanation-search chain unless
            # explicitly requested. This prevents both valid-path and
            # invalid-path searches from selecting/checking the concrete FULL node.
            if child == "FULL":
                if include_full_model:
                    eligible.append(child)
                continue

            # Same-branch refinement nodes, e.g., MOT2 -> MOT1.
            if edge_type == "refinement":
                if branch_prefix is None or child.startswith(branch_prefix):
                    eligible.append(child)

            # Bridge nodes, e.g., MOT2 -> MOT2-MIT3.
            elif edge_type == "bridge" and include_bridge_nodes:
                # Bridge source should match the current branch.
                if branch_prefix is not None and not child.startswith(branch_prefix):
                    continue

                # Bridge target should be the opposite branch.
                if not bridge_target_matches(child, target_branch, lattice_data):
                    continue

                eligible.append(child)

        # Visit bridge nodes before normal refinement nodes.
        eligible = sorted(
        eligible,
        key=lambda x: bridge_sort_key(x, lattice_data)
        )

        for child in eligible:
            queue.append(child)

    return order

def select_first_valid_bridge_result(
    mea_search_result: Dict[str, Any],
    lattice_data: Dict[str, Any]
) -> Tuple[Optional[str], Optional[Dict[str, Any]]]:
    """
    Selects the first valid bridge node from the MEA results.
    """

    for result in mea_search_result.get("all_results", []):
        node_id = result.get("node")

        if result.get("valid_by_mea") is True and is_bridge_node(node_id, lattice_data):
            return node_id, result

    return None, None

In [ ]:
# ============================================================
# Cell 11. Build grounded model for one node
# ============================================================

def get_domain_problem_paths_for_node(
    lattice_data: Dict[str, Any],
    lattice_json_path: str,
    node_id: str
) -> Dict[str, str]:
    nodes = lattice_data.get("nodes", {})

    if node_id not in nodes:
        raise ValueError(f"Node not found in lattice.json: {node_id}")

    domain_raw = nodes[node_id].get("domain")
    problem_raw = nodes[node_id].get("problem")

    if not domain_raw:
        raise ValueError(f"No domain path for node: {node_id}")

    if not problem_raw:
        raise ValueError(f"No problem path for node: {node_id}")

    domain_path = resolve_path(domain_raw, lattice_json_path)
    problem_path = resolve_path(problem_raw, lattice_json_path)

    if not os.path.exists(domain_path):
        raise FileNotFoundError(f"Domain file not found: {domain_path}")

    if not os.path.exists(problem_path):
        raise FileNotFoundError(f"Problem file not found: {problem_path}")

    return {
        "domain": domain_path,
        "problem": problem_path
    }


def build_grounded_action_model_for_node(
    lattice_data: Dict[str, Any],
    lattice_json_path: str,
    node_id: str,
    plan_actions: List[Dict[str, Any]]
) -> Dict[str, Any]:
    paths = get_domain_problem_paths_for_node(
        lattice_data=lattice_data,
        lattice_json_path=lattice_json_path,
        node_id=node_id
    )

    domain_text = read_text(paths["domain"])
    problem_text = read_text(paths["problem"])

    domain_actions = extract_pddl_actions(domain_text)
    initial_state = extract_initial_state_atoms(problem_text)

    grounded_actions = []

    for plan_action in plan_actions:
        action_name = plan_action["action_name"]

        if action_name not in domain_actions:
            grounded_actions.append({
                "plan_action": plan_action["raw"],
                "action_name": action_name,
                "schema_found": False,
                "preconditions": set(),
                "effects": set(),
                "missing_reason": "action schema not found"
            })
            continue

        action_block = domain_actions[action_name]

        params = extract_action_parameters(action_block)
        args = plan_action["args"]

        binding = {}

        for i, param in enumerate(params):
            if i < len(args):
                binding[param] = args[i]

        pre_text = extract_pddl_section(action_block, "precondition")
        eff_text = extract_pddl_section(action_block, "effect")

        pre_atoms = extract_predicate_atoms(pre_text)
        eff_atoms = extract_predicate_atoms(eff_text)

        grounded_pre = {
            ground_atom(atom, binding)
            for atom in pre_atoms
        }

        grounded_eff = {
            ground_atom(atom, binding)
            for atom in eff_atoms
        }

        grounded_actions.append({
            "plan_action": plan_action["raw"],
            "action_name": action_name,
            "schema_found": True,
            "parameters": params,
            "arguments": args,
            "binding": binding,
            "preconditions": grounded_pre,
            "effects": grounded_eff,
            "precondition_text": pre_text,
            "effect_text": eff_text
        })

    return {
        "node": node_id,
        "domain": paths["domain"],
        "problem": paths["problem"],
        "initial_state": initial_state,
        "grounded_actions": grounded_actions
    }

In [ ]:
def is_bridge_node(node_id: str, lattice_data: Dict[str, Any]) -> bool:
    node_info = lattice_data.get("nodes", {}).get(node_id, {})
    return node_info.get("node_type") == "bridge"

In [ ]:
# ============================================================
# Cell 12. EXISTENCE branch: backward MEA validity
# ============================================================

def run_mea_backward(
    node_model: Dict[str, Any],
    goal_predicate: str
) -> Dict[str, Any]:
    C = {normalize_atom(goal_predicate)}

    trace = []
    grounded_actions = node_model["grounded_actions"]
    initial_state = node_model["initial_state"]

    missing_action_schemas = [
        action["action_name"]
        for action in grounded_actions
        if not action.get("schema_found")
    ]

    for reverse_index, action in enumerate(reversed(grounded_actions), start=1):
        original_step = len(grounded_actions) - reverse_index + 1

        preconditions = set(action.get("preconditions", set()))
        effects = set(action.get("effects", set()))

        achieved = C.intersection(effects)
        remaining = C.difference(effects)
        new_C = remaining.union(preconditions)

        trace.append({
            "original_step": original_step,
            "plan_action": action["plan_action"],
            "schema_found": action.get("schema_found", False),
            "subgoals_before": sorted(C),
            "effects": sorted(effects),
            "preconditions_added": sorted(preconditions),
            "achieved_subgoals_removed": sorted(achieved),
            "unachieved_subgoals_carried": sorted(remaining),
            "subgoals_after": sorted(new_C)
        })

        C = new_C

    unachieved = C.difference(initial_state)

    valid = (
        len(missing_action_schemas) == 0
        and len(unachieved) == 0
    )

    return {
        "node": node_model["node"],
        "domain": node_model["domain"],
        "problem": node_model["problem"],
        "goal": normalize_atom(goal_predicate),
        "initial_state_count": len(initial_state),
        "initial_state": sorted(initial_state),
        "missing_action_schemas": missing_action_schemas,
        "final_required_subgoals": sorted(C),
        "unachieved_subgoals": sorted(unachieved),
        "valid_by_mea": valid,
        "backward_trace": list(reversed(trace))
    }




def result_satisfies_min_pre_eff_per_action(
    mea_result: Dict[str, Any],
    min_preconditions: int = 1,
    min_effects: int = 1
) -> Tuple[bool, List[Dict[str, Any]]]:
    """
    Checks whether a candidate explanation node preserves enough structure for
    a backward-MEA explanation.

    For bridge-node selection, do NOT rely only on raw action effects. A useful
    explanation node must contain:
        1. at least one grounded/abstract precondition, and
        2. at least one MEA-relevant postcondition.

    Here, a postcondition means an effect that actually satisfied a current
    backward-MEA subgoal, stored as `achieved_subgoals_removed`. This is stronger
    than checking whether the action schema has any effect at all.

    The function name is kept for compatibility with older notebook cells, but
    the check is now node-level: the selected bridge node must have at least one
    precondition and at least one MEA-relevant postcondition across its backward
    trace.
    """
    trace = mea_result.get("backward_trace", []) or []

    if not trace:
        return False, []

    details = []
    total_preconditions = 0
    total_postconditions = 0

    for item in trace:
        preconditions = item.get("preconditions_added", []) or []
        # MEA-relevant postconditions: effects that actually achieved a needed subgoal.
        postconditions = item.get("achieved_subgoals_removed", []) or []
        raw_effects = item.get("effects", []) or []
        schema_found = item.get("schema_found", False)

        pre_count = len(preconditions)
        post_count = len(postconditions)
        raw_effect_count = len(raw_effects)

        if schema_found:
            total_preconditions += pre_count
            total_postconditions += post_count

        details.append({
            "original_step": item.get("original_step"),
            "plan_action": item.get("plan_action"),
            "schema_found": schema_found,
            "precondition_count": pre_count,
            "postcondition_count": post_count,
            "raw_effect_count": raw_effect_count,
            "preconditions": preconditions,
            "postconditions": postconditions,
            "raw_effects": raw_effects,
            "has_precondition": schema_found is True and pre_count >= min_preconditions,
            "has_postcondition": schema_found is True and post_count >= min_effects
        })

    structure_ok = (
        total_preconditions >= min_preconditions
        and total_postconditions >= min_effects
    )

    return structure_ok, details


def select_first_valid_bridge_with_min_pre_eff(
    mea_search_result: Dict[str, Any],
    lattice_data: Dict[str, Any],
    min_preconditions: int = 1,
    min_effects: int = 1
) -> Tuple[Optional[str], Optional[Dict[str, Any]], List[Dict[str, Any]]]:
    """
    Selects the first valid bridge node that has at least one precondition and
    one MEA-relevant postcondition in the bridge-node chain.

    A node is accepted for explanation only if:
        - it is a bridge node,
        - it is valid by backward MEA,
        - its backward trace contains at least one precondition, and
        - its backward trace contains at least one postcondition that actually
          satisfies a required MEA subgoal.

    This prevents the explanation from selecting a bridge node that is valid but
    too abstract to show both the before-condition and after-condition needed for
    a meaningful causal explanation.
    """
    bridge_candidates = []

    for result in mea_search_result.get("all_results", []) or []:
        node_id = result.get("node")

        if not is_bridge_node(node_id, lattice_data):
            continue

        if result.get("valid_by_mea") is not True:
            continue

        structure_ok, structure_details = result_satisfies_min_pre_eff_per_action(
            mea_result=result,
            min_preconditions=min_preconditions,
            min_effects=min_effects
        )

        precondition_count = sum(
            d.get("precondition_count", 0)
            for d in structure_details
            if d.get("schema_found") is True
        )
        postcondition_count = sum(
            d.get("postcondition_count", 0)
            for d in structure_details
            if d.get("schema_found") is True
        )

        result["bridge_pre_post_structure_ok"] = structure_ok
        result["bridge_pre_post_structure_details"] = structure_details
        result["bridge_precondition_count"] = precondition_count
        result["bridge_postcondition_count"] = postcondition_count

        # Keep legacy keys too, so older reporting cells do not break.
        result["bridge_action_structure_ok"] = structure_ok
        result["bridge_action_structure_details"] = structure_details

        candidate_record = {
            "node": node_id,
            "valid_by_mea": True,
            "bridge_pre_post_structure_ok": structure_ok,
            "precondition_count": precondition_count,
            "postcondition_count": postcondition_count,
            "structure_details": structure_details
        }
        bridge_candidates.append(candidate_record)

        if structure_ok:
            return node_id, result, bridge_candidates

    return None, None, bridge_candidates



def select_first_valid_node_with_min_pre_eff(
    mea_search_result: Dict[str, Any],
    min_preconditions: int = 1,
    min_effects: int = 1
) -> Tuple[Optional[str], Optional[Dict[str, Any]], List[Dict[str, Any]]]:
    """
    Fallback selector for the bridge-aware search chain.

    If no valid bridge node satisfies the pre/post constraint, this selects the
    first valid node in the same bridge-aware search chain that still has at
    least one precondition and one MEA-relevant postcondition. It does not select
    a structurally empty valid node for explanation.
    """
    candidates = []

    for result in mea_search_result.get("all_results", []) or []:
        node_id = result.get("node")

        if result.get("valid_by_mea") is not True:
            continue

        structure_ok, structure_details = result_satisfies_min_pre_eff_per_action(
            mea_result=result,
            min_preconditions=min_preconditions,
            min_effects=min_effects
        )

        precondition_count = sum(
            d.get("precondition_count", 0)
            for d in structure_details
            if d.get("schema_found") is True
        )
        postcondition_count = sum(
            d.get("postcondition_count", 0)
            for d in structure_details
            if d.get("schema_found") is True
        )

        result["selected_node_pre_post_structure_ok"] = structure_ok
        result["selected_node_pre_post_structure_details"] = structure_details
        result["selected_node_precondition_count"] = precondition_count
        result["selected_node_postcondition_count"] = postcondition_count

        candidate_record = {
            "node": node_id,
            "valid_by_mea": True,
            "pre_post_structure_ok": structure_ok,
            "precondition_count": precondition_count,
            "postcondition_count": postcondition_count,
            "structure_details": structure_details
        }
        candidates.append(candidate_record)

        if structure_ok:
            return node_id, result, candidates

    return None, None, candidates
def search_lattice_with_mea(
    lattice_data: Dict[str, Any],
    lattice_json_path: str,
    search_chain: List[str],
    plan_actions: List[Dict[str, Any]],
    goal_predicate: str,
    stop_on_first_valid: bool = False,
    stop_on_first_valid_bridge: bool = False
) -> Dict[str, Any]:
    results = []

    first_valid_node = None
    first_valid_result = None

    first_valid_bridge_node = None
    first_valid_bridge_result = None

    last_valid_node = None
    last_valid_result = None

    for node_id in search_chain:
        print(f"\n>>> Running MEA at node: {node_id}")

        try:
            paths = get_domain_problem_paths_for_node(
                lattice_data=lattice_data,
                lattice_json_path=lattice_json_path,
                node_id=node_id
            )

            print("MEA domain:", paths["domain"])
            print("MEA domain exists:", os.path.exists(paths["domain"]))
            print("MEA problem:", paths["problem"])
            print("MEA problem exists:", os.path.exists(paths["problem"]))

            node_model = build_grounded_action_model_for_node(
                lattice_data=lattice_data,
                lattice_json_path=lattice_json_path,
                node_id=node_id,
                plan_actions=plan_actions
            )

            mea_result = run_mea_backward(
                node_model=node_model,
                goal_predicate=goal_predicate
            )

            mea_result["node"] = node_id
            mea_result["domain"] = paths["domain"]
            mea_result["problem"] = paths["problem"]

        except Exception as e:
            mea_result = {
                "node": node_id,
                "valid_by_mea": False,
                "error": str(e),
                "domain": paths["domain"] if "paths" in locals() else None,
                "problem": paths["problem"] if "paths" in locals() else None,
                "missing_action_schemas": [],
                "final_required_subgoals": [],
                "unachieved_subgoals": [],
                "backward_trace": []
            }

        results.append(mea_result)

        print("Valid by MEA:", mea_result.get("valid_by_mea"))
        print("Unachieved:", mea_result.get("unachieved_subgoals"))

        if mea_result.get("valid_by_mea") is True:
            if first_valid_node is None:
                first_valid_node = node_id
                first_valid_result = mea_result

            if is_bridge_node(node_id, lattice_data) and first_valid_bridge_node is None:
                first_valid_bridge_node = node_id
                first_valid_bridge_result = mea_result

                if stop_on_first_valid_bridge:
                    print("Stopping at first valid bridge node:", node_id)
                    break

            last_valid_node = node_id
            last_valid_result = mea_result

            if stop_on_first_valid:
                print("Stopping at first valid node:", node_id)
                break

    valid_nodes = [
        r["node"] for r in results
        if r.get("valid_by_mea") is True
    ]

    invalid_nodes = [
        r["node"] for r in results
        if r.get("valid_by_mea") is not True
    ]

    return {
        "search_chain": search_chain,
        "visited_nodes": [r["node"] for r in results],

        "first_valid_node": first_valid_node,
        "first_valid_result": first_valid_result,

        "first_valid_bridge_node": first_valid_bridge_node,
        "first_valid_bridge_result": first_valid_bridge_result,

        "last_valid_node": last_valid_node,
        "last_valid_result": last_valid_result,

        "valid_nodes": valid_nodes,
        "invalid_nodes": invalid_nodes,
        "all_results": results
    }


def collect_plan_preconditions(mea_result: Dict[str, Any]) -> Set[str]:
    preconditions = set()

    for item in mea_result.get("backward_trace", []):
        for p in item.get("preconditions_added", []):
            preconditions.add(normalize_atom(p))

    return preconditions


def find_result_for_node(search_result: Dict[str, Any], node_id: str) -> Optional[Dict[str, Any]]:
    for item in search_result.get("all_results", []):
        if item.get("node") == node_id:
            return item

    return None


# FULL-model MEA helper removed from the active validity pipeline.
# Valid-path explanation-node search now stays in the stakeholder/bridge chain.


In [ ]:
def extract_predicate_dependency_chain(selected_result: Dict[str, Any]) -> List[Dict[str, Any]]:
    """
    Extracts predicate-only dependency links from the selected first-valid MEA result.

    This avoids explaining validity using action names.
    Each link records which predicate(s) were required to support which achieved predicate(s).
    """
    predicate_links = []

    for item in selected_result.get("backward_trace", []):
        required = item.get("preconditions_added", [])
        achieved = item.get("achieved_subgoals_removed", [])

        # Skip steps that do not contribute predicate dependencies
        # at the selected abstraction.
        if not required and not achieved:
            continue

        predicate_links.append({
            "original_step": item.get("original_step"),
            "required_predicates": required,
            "achieved_predicates": achieved,
            "subgoals_before": item.get("subgoals_before", []),
            "subgoals_after": item.get("subgoals_after", [])
        })

    return predicate_links

In [ ]:
# ============================================================
# Cell 13. EXISTENCE report writer
# ============================================================

def generate_mea_report(output: Dict[str, Any]) -> str:
    lines = []

    lines.append("=== MEA LATTICE PLAN VALIDITY REPORT ===")
    lines.append("")
    lines.append(f"Intent: {output.get('intent')}")
    lines.append(f"Declared stakeholder view: {output.get('declared_stakeholder') or output.get('predicted_stakeholder')}")
    lines.append(f"Start node: {output.get('start_node')}")
    lines.append(f"Branch: {output.get('branch_prefix')}")
    lines.append(f"Goal: {output.get('goal')}")
    lines.append("")

    # --------------------------------------------------------
    # Plan files
    # --------------------------------------------------------
    lines.append("Plan files:")
    lines.append(f"  Original mapped plan: {output.get('original_plan_file')}")
    lines.append(f"  Repaired plan used by MEA: {output.get('repaired_plan_file')}")
    lines.append("")

    # --------------------------------------------------------
    # Plan repair summary
    # --------------------------------------------------------
    lines.append("Plan repair:")
    repair = output.get("plan_repair_result", {})
    repair_rounds = repair.get("repair_rounds", [])

    plan_needed_repair = len(repair_rounds) > 0

    inserted_repair_actions = []
    for r in repair_rounds:
        inserted_repair_actions.extend(r.get("inserted_actions_string", []) or [])

    lines.append(f"  Plan needed repair: {'Yes' if plan_needed_repair else 'No'}")
    lines.append(f"  Number of missing actions added: {len(inserted_repair_actions)}")
    lines.append(f"  Status: {repair.get('status')}")
    lines.append(f"  Message: {repair.get('message')}")
    lines.append(f"  JSON report: {output.get('plan_repair_json')}")
    lines.append(f"  TXT report: {output.get('plan_repair_txt')}")

    for r in repair_rounds:
        lines.append("")
        lines.append(f"  Repair round {r.get('round')}: {r.get('repair_type')}")

        if r.get("failed_action_string"):
            lines.append(f"    Failed original action: {r.get('failed_action_string')}")

        missing = r.get("missing_positive_preconditions_string", [])
        if missing:
            lines.append("    Missing predicates before repair:")
            for item in missing:
                lines.append(f"      - {item}")

        repair_goal = r.get("repair_goal_string", [])
        if repair_goal:
            lines.append("    Predicates established by repair:")
            for item in repair_goal:
                lines.append(f"      - {item}")

        inserted = r.get("inserted_actions_string", [])
        if inserted:
            lines.append("    Inserted repair actions:")
            for item in inserted:
                lines.append(f"      - {item}")

    lines.append("")

    # --------------------------------------------------------
    # Lattice search summary
    # --------------------------------------------------------
    lines.append("Search chain:")
    lines.append("  " + " -> ".join(output.get("search_chain", [])))
    lines.append("")

    lines.append("Selected valid-node policy:")
    lines.append(f"  {output.get('selected_valid_node_policy')}")
    lines.append("")

    lines.append(f"First valid node by backward MEA: {output.get('first_valid_node')}")
    lines.append(f"Last valid node visited by backward MEA: {output.get('last_valid_node')}")
    lines.append(f"First bridge node satisfying pre/post constraint: {output.get('first_structured_bridge_valid_node')}")
    lines.append(f"First valid node satisfying pre/post constraint: {output.get('first_structured_valid_node')}")
    lines.append(f"Selected abstract valid node: {output.get('abstract_valid_node')}")
    lines.append(f"Bridge selection constraint: {output.get('bridge_selection_constraint')}")
    lines.append("")

    lines.append("Valid nodes visited before stopping:")
    valid_nodes = output.get("valid_nodes", [])
    if valid_nodes:
        for node in valid_nodes:
            lines.append(f"  - {node}")
    else:
        lines.append("  - None")

    lines.append("")

    lines.append("Invalid nodes visited before stopping:")
    invalid_nodes = output.get("invalid_nodes", [])
    if invalid_nodes:
        for node in invalid_nodes:
            lines.append(f"  - {node}")
    else:
        lines.append("  - None")

    lines.append("")

    # --------------------------------------------------------
    # Important new section:
    # selected first-valid predicate chain
    # --------------------------------------------------------
    lines.append("Selected first-valid predicate chain:")
    lines.append(
        "  This is the predicate-level backward MEA evidence used for explanation."
    )
    lines.append(
        "  The explanation should be based on these predicate dependencies, not action names."
    )

    selected_result = output.get("abstract_valid_result")

    if selected_result is None:
        selected_result = output.get("search_result", {}).get("first_valid_result")

    if selected_result is None:
        selected_result = output.get("search_result", {}).get("last_valid_result")

    if selected_result:
        predicate_chain = extract_predicate_dependency_chain(selected_result)

        if predicate_chain:
            for i, link in enumerate(predicate_chain, start=1):
                lines.append("")
                lines.append(f"Predicate link {i}:")

                required = link.get("required_predicates", [])
                achieved = link.get("achieved_predicates", [])

                lines.append("  Required predicates:")
                if required:
                    for p in required:
                        lines.append(f"    - {p}")
                else:
                    lines.append("    - None at this abstraction")

                lines.append("  Achieved predicates:")
                if achieved:
                    for p in achieved:
                        lines.append(f"    - {p}")
                else:
                    lines.append("    - None")

                subgoals_before = link.get("subgoals_before", [])
                subgoals_after = link.get("subgoals_after", [])

                lines.append("  Subgoals before:")
                if subgoals_before:
                    for sg in subgoals_before:
                        lines.append(f"    - {sg}")
                else:
                    lines.append("    - None")

                lines.append("  Subgoals after regression:")
                if subgoals_after:
                    for sg in subgoals_after:
                        lines.append(f"    - {sg}")
                else:
                    lines.append("    - None")

        else:
            lines.append("")
            lines.append("  No non-empty predicate dependency links found.")
    else:
        lines.append("")
        lines.append("  No selected valid MEA result found.")

    lines.append("")

    # --------------------------------------------------------
    # Selected node final predicate status
    # --------------------------------------------------------
    lines.append("Selected first-valid node predicate status:")

    if selected_result:
        lines.append(f"  Valid by MEA: {selected_result.get('valid_by_mea')}")
        lines.append("  Final required predicates:")

        final_required = selected_result.get("final_required_subgoals", [])
        if final_required:
            for sg in final_required:
                lines.append(f"    - {sg}")
        else:
            lines.append("    - None; all required predicates are supported at this abstraction.")

        lines.append("  Unachieved predicates:")

        unachieved = selected_result.get("unachieved_subgoals", [])
        if unachieved:
            for sg in unachieved:
                lines.append(f"    - {sg}")
        else:
            lines.append("    - None; the predicate chain is valid.")
    else:
        lines.append("  No selected result available.")

    lines.append("")

    # --------------------------------------------------------
    # Visited node results
    # --------------------------------------------------------
    lines.append("Visited node results:")

    for result in output.get("search_result", {}).get("all_results", []):
        lines.append("")
        lines.append(f"Node: {result.get('node')}")
        lines.append(f"  Valid by MEA: {result.get('valid_by_mea')}")
        lines.append(f"  Missing action schemas: {result.get('missing_action_schemas')}")
        lines.append(f"  Final required subgoals: {result.get('final_required_subgoals')}")
        lines.append(f"  Unachieved subgoals: {result.get('unachieved_subgoals')}")

    lines.append("")

    # --------------------------------------------------------
    # Search boundary
    # --------------------------------------------------------
    lines.append("Search boundary:")
    lines.append(f"  FULL model checked as validity/repair guard: {output.get('full_model_checked_as_consistency_guard', True)}")
    lines.append(f"  FULL model used for plan repair: {output.get('full_model_used_for_plan_repair', True)}")
    lines.append("  FULL model checked during explanation search: False")
    lines.append(f"  FULL model used as explanation node: {output.get('full_model_used_as_explanation_node', False)}")
    lines.append("  Note: FULL may be used to check/repair the plan before valid-path analysis, but it is excluded from the valid explanation-node search.")

    return "\n".join(lines)


In [ ]:

# ============================================================
# Cell 13B. Plan repair before EXISTENCE / validity branch
# ============================================================

from dataclasses import dataclass
import itertools
import shutil

# Fast Downward is optional. If it is not available, the repair code uses
# the internal BFS planner below.
FAST_DOWNWARD = None

PLAN_REPAIR_WORKDIR = BASE_DIR + f"plan_repair_work_{query}"
PLAN_REPAIR_JSON = PLAN_REPAIR_WORKDIR + "/plan_repair_report.json"
PLAN_REPAIR_TXT = PLAN_REPAIR_WORKDIR + "/plan_repair_report.txt"
REPAIRED_PLAN_FILE = PLAN_REPAIR_WORKDIR + "/repaired_plan.plan"


# -----------------------------
# PDDL s-expression parser
# -----------------------------

def repair_strip_comments(text: str) -> str:
    return re.sub(r";.*", "", text)


def repair_tokenize_pddl(text: str) -> List[str]:
    text = repair_strip_comments(text).lower()
    text = text.replace("(", " ( ").replace(")", " ) ")
    return text.split()


def repair_parse_sexp(tokens: List[str]) -> Any:
    if not tokens:
        raise ValueError("Unexpected end of PDDL tokens.")

    token = tokens.pop(0)

    if token == "(":
        expr = []
        while tokens and tokens[0] != ")":
            expr.append(repair_parse_sexp(tokens))

        if not tokens:
            raise ValueError("Missing closing parenthesis in PDDL.")

        tokens.pop(0)
        return expr

    if token == ")":
        raise ValueError("Unexpected closing parenthesis in PDDL.")

    return token


def repair_parse_pddl_file(path: str) -> Any:
    tokens = repair_tokenize_pddl(read_text(path))
    return repair_parse_sexp(tokens)


def repair_sexp_to_pddl(expr: Any) -> str:
    if isinstance(expr, str):
        return expr

    return "(" + " ".join(repair_sexp_to_pddl(x) for x in expr) + ")"


def repair_parse_typed_list(items: List[Any], variable_mode: bool = False) -> Tuple[List[str], Dict[str, str]]:
    """
    Parses typed PDDL lists.

    Parameters example:
        ?at - node ?to - node

    Objects example:
        a b - node c - plc
    """
    names = []
    types = {}

    pending = []
    i = 0

    while i < len(items):
        token = items[i]

        if token == "-":
            type_name = items[i + 1]

            for name in pending:
                names.append(name)
                types[name] = type_name

            pending = []
            i += 2
            continue

        if isinstance(token, str):
            if variable_mode:
                if token.startswith("?"):
                    pending.append(token)
            else:
                pending.append(token)

        i += 1

    for name in pending:
        names.append(name)
        types[name] = "object"

    return names, types


def repair_positive_literals(expr: Any) -> List[Tuple[str, ...]]:
    """
    Extracts only positive literals.
    This notebook assumes no negative preconditions.
    """
    positive = []

    if expr is None or isinstance(expr, str):
        return positive

    if len(expr) == 0:
        return positive

    head = expr[0]

    if head == "and":
        for sub in expr[1:]:
            positive.extend(repair_positive_literals(sub))

    elif head == "not":
        # Negative preconditions are not used in this model.
        return positive

    else:
        positive.append(tuple(expr))

    return positive


def repair_effect_literals(expr: Any) -> Tuple[List[Tuple[str, ...]], List[Tuple[str, ...]]]:
    """
    Extracts add and delete effects.
    Delete effects are kept for safety even though the current model mainly uses adds.
    """
    add_effects = []
    delete_effects = []

    if expr is None or isinstance(expr, str):
        return add_effects, delete_effects

    if len(expr) == 0:
        return add_effects, delete_effects

    head = expr[0]

    if head == "and":
        for sub in expr[1:]:
            adds, deletes = repair_effect_literals(sub)
            add_effects.extend(adds)
            delete_effects.extend(deletes)

    elif head == "not":
        delete_effects.append(tuple(expr[1]))

    else:
        add_effects.append(tuple(expr))

    return add_effects, delete_effects


def repair_ground_lit(lit: Tuple[str, ...], binding: Dict[str, str]) -> Tuple[str, ...]:
    return tuple([lit[0]] + [binding.get(arg, arg) for arg in lit[1:]])


def repair_format_lit(lit: Tuple[str, ...]) -> str:
    return "(" + " ".join(lit) + ")"


def repair_action_to_string(action: Dict[str, Any]) -> str:
    return "(" + " ".join([action["name"]] + action["args"]) + ")"


def repair_lits_to_strings(literals: List[Tuple[str, ...]]) -> List[str]:
    return [repair_format_lit(lit) for lit in literals]


@dataclass
class RepairActionSchema:
    name: str
    parameters: List[str]
    parameter_types: Dict[str, str]
    precondition: Any
    effect: Any


@dataclass
class RepairDomain:
    name: str
    actions: Dict[str, RepairActionSchema]


@dataclass
class RepairProblem:
    name: str
    domain_name: str
    objects_expr: List[Any]
    objects: List[str]
    object_types: Dict[str, str]
    init: Set[Tuple[str, ...]]
    goal_expr: Any


def repair_parse_domain(domain_path: str) -> RepairDomain:
    ast = repair_parse_pddl_file(domain_path)

    domain_name = None
    actions = {}

    for item in ast[1:]:
        if not isinstance(item, list) or not item:
            continue

        if item[0] == "domain":
            domain_name = item[1]

        elif item[0] == ":action":
            action_name = item[1]
            parameters = []
            parameter_types = {}
            precondition = ["and"]
            effect = ["and"]

            i = 2
            while i < len(item):
                key = item[i]

                if key == ":parameters":
                    parameters, parameter_types = repair_parse_typed_list(
                        item[i + 1],
                        variable_mode=True
                    )
                    i += 2

                elif key == ":precondition":
                    precondition = item[i + 1]
                    i += 2

                elif key == ":effect":
                    effect = item[i + 1]
                    i += 2

                else:
                    i += 1

            actions[action_name] = RepairActionSchema(
                name=action_name,
                parameters=parameters,
                parameter_types=parameter_types,
                precondition=precondition,
                effect=effect
            )

    return RepairDomain(
        name=domain_name,
        actions=actions
    )


def repair_parse_problem(problem_path: str) -> RepairProblem:
    ast = repair_parse_pddl_file(problem_path)

    problem_name = None
    domain_name = None
    objects_expr = []
    objects = []
    object_types = {}
    init = set()
    goal_expr = ["and"]

    for item in ast[1:]:
        if not isinstance(item, list) or not item:
            continue

        if item[0] == "problem":
            problem_name = item[1]

        elif item[0] == ":domain":
            domain_name = item[1]

        elif item[0] == ":objects":
            objects_expr = item[1:]
            objects, object_types = repair_parse_typed_list(objects_expr, variable_mode=False)

        elif item[0] == ":init":
            for lit in item[1:]:
                if isinstance(lit, list) and lit and lit[0] != "=":
                    init.add(tuple(lit))

        elif item[0] == ":goal":
            goal_expr = item[1]

    return RepairProblem(
        name=problem_name,
        domain_name=domain_name,
        objects_expr=objects_expr,
        objects=objects,
        object_types=object_types,
        init=init,
        goal_expr=goal_expr
    )


# -----------------------------
# Plan I/O and simulation
# -----------------------------

def repair_read_plan(plan_path: str) -> List[Dict[str, Any]]:
    actions = []

    for line in read_text(plan_path).splitlines():
        line = line.strip().lower()

        if not line or line.startswith(";"):
            continue

        line = re.sub(r"^\d+(\.\d+)?\s*:\s*", "", line)
        line = line.strip().strip('"').strip("'")

        if not line.startswith("(") or not line.endswith(")"):
            continue

        parts = line[1:-1].strip().split()

        if not parts or parts[0] == "cost":
            continue

        actions.append({
            "name": parts[0],
            "args": parts[1:]
        })

    return actions


def repair_write_plan(plan: List[Dict[str, Any]], out_path: str) -> None:
    Path(out_path).parent.mkdir(parents=True, exist_ok=True)
    Path(out_path).write_text(
        "\n".join(repair_action_to_string(a) for a in plan) + "\n",
        encoding="utf-8"
    )


def repair_check_preconditions(
    domain: RepairDomain,
    state: Set[Tuple[str, ...]],
    action: Dict[str, Any]
) -> Dict[str, Any]:
    action_name = action["name"]

    if action_name not in domain.actions:
        return {
            "ok": False,
            "reason": "unknown-action",
            "missing_positive": [],
            "message": f"Unknown action: {action_name}"
        }

    schema = domain.actions[action_name]

    if len(action["args"]) != len(schema.parameters):
        return {
            "ok": False,
            "reason": "wrong-number-of-arguments",
            "missing_positive": [],
            "message": (
                f"{action_name} expects {len(schema.parameters)} arguments "
                f"but got {len(action['args'])}"
            )
        }

    binding = dict(zip(schema.parameters, action["args"]))

    grounded_preconditions = [
        repair_ground_lit(lit, binding)
        for lit in repair_positive_literals(schema.precondition)
    ]

    missing_positive = [
        lit for lit in grounded_preconditions
        if lit not in state
    ]

    return {
        "ok": len(missing_positive) == 0,
        "reason": None if len(missing_positive) == 0 else "missing-positive-preconditions",
        "missing_positive": missing_positive,
        "message": "ok" if len(missing_positive) == 0 else "Some positive preconditions are missing."
    }


def repair_apply_action(
    domain: RepairDomain,
    state: Set[Tuple[str, ...]],
    action: Dict[str, Any]
) -> Set[Tuple[str, ...]]:
    schema = domain.actions[action["name"]]
    binding = dict(zip(schema.parameters, action["args"]))

    add_effects, delete_effects = repair_effect_literals(schema.effect)

    new_state = set(state)

    for lit in delete_effects:
        new_state.discard(repair_ground_lit(lit, binding))

    for lit in add_effects:
        new_state.add(repair_ground_lit(lit, binding))

    return new_state


def repair_goal_holds(goal_expr: Any, state: Set[Tuple[str, ...]]) -> bool:
    for lit in repair_positive_literals(goal_expr):
        if lit not in state:
            return False

    return True


def repair_missing_goal_literals(goal_expr: Any, state: Set[Tuple[str, ...]]) -> List[Tuple[str, ...]]:
    return [
        lit for lit in repair_positive_literals(goal_expr)
        if lit not in state
    ]


def repair_simulate_plan(
    domain: RepairDomain,
    problem: RepairProblem,
    plan: List[Dict[str, Any]]
) -> Dict[str, Any]:
    state = set(problem.init)

    for index, action in enumerate(plan):
        check = repair_check_preconditions(domain, state, action)

        if not check["ok"]:
            return {
                "valid": False,
                "all_actions_executable": False,
                "goal_reached": False,
                "fail_index": index,
                "failed_action": action,
                "state_before_failure": state,
                "missing_positive": check["missing_positive"],
                "reason": check["reason"],
                "message": check["message"]
            }

        state = repair_apply_action(domain, state, action)

    goal_reached = repair_goal_holds(problem.goal_expr, state)

    return {
        "valid": goal_reached,
        "all_actions_executable": True,
        "goal_reached": goal_reached,
        "fail_index": None,
        "failed_action": None,
        "state_before_failure": state,
        "missing_positive": repair_missing_goal_literals(problem.goal_expr, state),
        "reason": None if goal_reached else "goal-not-reached",
        "message": "Plan is valid." if goal_reached else "All actions execute, but the final goal is not reached."
    }


# -----------------------------
# Repair planning
# -----------------------------

def repair_missing_positive_goal(missing_positive: List[Tuple[str, ...]]) -> Any:
    return ["and"] + [list(lit) for lit in missing_positive]


def repair_objects_for_type(problem: RepairProblem, type_name: str) -> List[str]:
    if type_name == "object":
        return problem.objects

    typed_objects = [
        obj for obj in problem.objects
        if problem.object_types.get(obj, "object") == type_name
    ]

    # Fallback for simple models that do not type every object explicitly.
    return typed_objects if typed_objects else problem.objects


def repair_generate_ground_actions(domain: RepairDomain, problem: RepairProblem) -> List[Dict[str, Any]]:
    grounded = []

    for action_name, schema in domain.actions.items():
        object_lists = []

        for param in schema.parameters:
            param_type = schema.parameter_types.get(param, "object")
            object_lists.append(repair_objects_for_type(problem, param_type))

        for args in itertools.product(*object_lists):
            grounded.append({
                "name": action_name,
                "args": list(args)
            })

    return grounded


def repair_bfs_plan(
    domain: RepairDomain,
    problem: RepairProblem,
    start_state: Set[Tuple[str, ...]],
    repair_goal: Any,
    max_depth: int = 5
) -> Dict[str, Any]:
    """
    Small internal planner for repair. It searches for a short sequence of
    actions that achieves the missing positive preconditions.
    """
    grounded_actions = repair_generate_ground_actions(domain, problem)

    start_state_frozen = frozenset(start_state)
    queue = deque([(start_state_frozen, [])])
    visited = {start_state_frozen}

    while queue:
        current_state_frozen, current_plan = queue.popleft()
        current_state = set(current_state_frozen)

        if repair_goal_holds(repair_goal, current_state):
            return {
                "found": True,
                "plan": current_plan,
                "planner": "internal_bfs"
            }

        if len(current_plan) >= max_depth:
            continue

        for action in grounded_actions:
            check = repair_check_preconditions(domain, current_state, action)

            if not check["ok"]:
                continue

            next_state = repair_apply_action(domain, current_state, action)
            next_state_frozen = frozenset(next_state)

            if next_state_frozen in visited:
                continue

            visited.add(next_state_frozen)
            queue.append((next_state_frozen, current_plan + [action]))

    return {
        "found": False,
        "plan": [],
        "planner": "internal_bfs"
    }


def repair_get_add_effect_producers(domain: RepairDomain) -> Dict[str, List[str]]:
    producers = {}

    for action_name, schema in domain.actions.items():
        add_effects, _ = repair_effect_literals(schema.effect)

        for lit in add_effects:
            pred = lit[0]
            producers.setdefault(pred, []).append(action_name)

    return producers


def repair_diagnose_missing_facts(
    domain: RepairDomain,
    problem: RepairProblem,
    missing_literals: List[Tuple[str, ...]]
) -> Dict[str, Any]:
    producers = repair_get_add_effect_producers(domain)

    diagnosis = {
        "already_in_init": [],
        "achievable_by_action": [],
        "static_missing": []
    }

    for lit in missing_literals:
        pred = lit[0]

        if lit in problem.init:
            diagnosis["already_in_init"].append(lit)

        elif pred in producers:
            diagnosis["achievable_by_action"].append({
                "fact": lit,
                "producer_actions": producers[pred]
            })

        else:
            diagnosis["static_missing"].append(lit)

    return diagnosis


def repair_save_readable_report(report: Dict[str, Any], out_path: str) -> None:
    lines = []
    lines.append("=== PLAN REPAIR REPORT ===")
    lines.append("")
    lines.append(f"Status: {report.get('status')}")
    lines.append(f"Message: {report.get('message')}")
    lines.append(f"Original plan: {report.get('original_plan_path')}")
    lines.append(f"Final plan: {report.get('final_plan_path')}")
    lines.append("")

    for round_report in report.get("repair_rounds", []):
        lines.append("=" * 60)
        lines.append(f"Round: {round_report.get('round')}")
        lines.append(f"Repair type: {round_report.get('repair_type')}")
        lines.append(f"Failed action: {round_report.get('failed_action_string')}")
        lines.append("")
        lines.append("Missing positive preconditions:")
        for item in round_report.get("missing_positive_preconditions_string", []):
            lines.append(f"  - {item}")
        lines.append("")
        lines.append("Inserted actions:")
        for item in round_report.get("inserted_actions_string", []):
            lines.append(f"  - {item}")
        lines.append("")

    write_text(out_path, "\n".join(lines))


def repair_json_safe(obj: Any) -> Any:
    if isinstance(obj, tuple):
        return list(obj)

    if isinstance(obj, set):
        return [repair_json_safe(x) for x in obj]

    if isinstance(obj, list):
        return [repair_json_safe(x) for x in obj]

    if isinstance(obj, dict):
        return {str(k): repair_json_safe(v) for k, v in obj.items()}

    if hasattr(obj, "__dict__"):
        return repair_json_safe(obj.__dict__)

    return obj


def repair_plan_by_inserting_actions(
    domain_path: str,
    problem_path: str,
    user_plan_path: str,
    repaired_plan_path: str,
    json_report_path: str,
    txt_report_path: str,
    max_rounds: int = 5,
    repair_depth: int = 5
) -> Dict[str, Any]:
    """
    Repairs the user plan in the FULL node before validity analysis.

    Repair policy:
    - preserve the user's plan
    - insert extra actions before the first failed action
    - if all actions execute but the final goal is missing, append actions
    """
    domain = repair_parse_domain(domain_path)
    problem = repair_parse_problem(problem_path)
    plan = repair_read_plan(user_plan_path)

    Path(repaired_plan_path).parent.mkdir(parents=True, exist_ok=True)

    original_normalized = str(Path(repaired_plan_path).parent / "original_plan_before_repair.plan")
    repair_write_plan(plan, original_normalized)

    history = []

    report = {
        "pipeline": "plan_repair_before_validity_mea",
        "domain_path": domain_path,
        "problem_path": problem_path,
        "original_plan_path": user_plan_path,
        "original_normalized_plan_path": original_normalized,
        "final_plan_path": None,
        "status": None,
        "message": None,
        "repair_rounds": history,
        "final_plan": None
    }

    for round_id in range(1, max_rounds + 1):
        print("=" * 70)
        print(f"Plan repair round {round_id}")
        print("=" * 70)

        sim = repair_simulate_plan(domain, problem, plan)

        if sim["valid"]:
            repair_write_plan(plan, repaired_plan_path)

            report["status"] = "valid"
            report["message"] = (
                "The original user plan is already valid in the FULL model."
                if len(history) == 0
                else "The plan is valid after inserting repair actions."
            )
            report["final_plan_path"] = repaired_plan_path
            report["final_plan"] = plan

            write_json(json_report_path, repair_json_safe(report))
            repair_save_readable_report(report, txt_report_path)

            print(report["message"])
            print("Repaired plan:", repaired_plan_path)

            return report

        if not sim["all_actions_executable"]:
            fail_index = sim["fail_index"]
            failed_action = sim["failed_action"]
            missing_positive = sim["missing_positive"]
            state_before_failure = sim["state_before_failure"]

            print("Failed action index:", fail_index)
            print("Failed action:", repair_action_to_string(failed_action))
            print("Missing positive preconditions:")
            for lit in missing_positive:
                print("  ", repair_format_lit(lit))

            diagnosis = repair_diagnose_missing_facts(domain, problem, missing_positive)

            if diagnosis["static_missing"]:
                report["status"] = "repair-failed-static-missing-facts"
                report["message"] = (
                    "The plan cannot be repaired by inserting actions because "
                    "some missing positive facts are static and no action can produce them."
                )
                report["static_missing_facts"] = diagnosis["static_missing"]
                report["static_missing_facts_string"] = repair_lits_to_strings(diagnosis["static_missing"])
                report["final_plan"] = plan

                write_json(json_report_path, repair_json_safe(report))
                repair_save_readable_report(report, txt_report_path)
                return report

            repair_goal = repair_missing_positive_goal(missing_positive)

            planner_result = repair_bfs_plan(
                domain=domain,
                problem=problem,
                start_state=state_before_failure,
                repair_goal=repair_goal,
                max_depth=repair_depth
            )

            round_report = {
                "round": round_id,
                "repair_type": "insert-actions-before-failed-action",
                "failed_action_index": fail_index,
                "failed_action": failed_action,
                "failed_action_string": repair_action_to_string(failed_action),
                "missing_positive_preconditions": missing_positive,
                "missing_positive_preconditions_string": repair_lits_to_strings(missing_positive),
                "diagnosis": diagnosis,
                "repair_goal": repair_goal,
                "repair_goal_string": repair_lits_to_strings(repair_positive_literals(repair_goal)),
                "planner": planner_result.get("planner"),
                "planner_found_repair": planner_result["found"],
                "inserted_actions": planner_result["plan"],
                "inserted_actions_string": [
                    repair_action_to_string(a)
                    for a in planner_result["plan"]
                ]
            }

            history.append(round_report)

            if not planner_result["found"]:
                report["status"] = "repair-failed"
                report["message"] = "Could not find actions to make the failed action applicable."
                report["final_plan"] = plan

                write_json(json_report_path, repair_json_safe(report))
                repair_save_readable_report(report, txt_report_path)
                return report

            print("Repair actions found:")
            for action in planner_result["plan"]:
                print("  ", repair_action_to_string(action))

            prefix = plan[:fail_index]
            suffix = plan[fail_index:]
            plan = prefix + planner_result["plan"] + suffix

        else:
            final_state = sim["state_before_failure"]
            missing_goal = sim["missing_positive"]

            print("All actions executed, but the final goal was not reached.")
            print("Missing goal facts:")
            for lit in missing_goal:
                print("  ", repair_format_lit(lit))

            repair_goal = repair_missing_positive_goal(missing_goal)

            planner_result = repair_bfs_plan(
                domain=domain,
                problem=problem,
                start_state=final_state,
                repair_goal=repair_goal,
                max_depth=repair_depth
            )

            round_report = {
                "round": round_id,
                "repair_type": "append-actions-to-reach-final-goal",
                "missing_goal_facts": missing_goal,
                "missing_goal_facts_string": repair_lits_to_strings(missing_goal),
                "repair_goal": repair_goal,
                "repair_goal_string": repair_lits_to_strings(repair_positive_literals(repair_goal)),
                "planner": planner_result.get("planner"),
                "planner_found_repair": planner_result["found"],
                "inserted_actions": planner_result["plan"],
                "inserted_actions_string": [
                    repair_action_to_string(a)
                    for a in planner_result["plan"]
                ]
            }

            history.append(round_report)

            if not planner_result["found"]:
                report["status"] = "repair-failed"
                report["message"] = "Could not find actions to achieve the final goal."
                report["final_plan"] = plan

                write_json(json_report_path, repair_json_safe(report))
                repair_save_readable_report(report, txt_report_path)
                return report

            print("Actions appended to reach the final goal:")
            for action in planner_result["plan"]:
                print("  ", repair_action_to_string(action))

            plan = plan + planner_result["plan"]

    best_effort_path = str(Path(repaired_plan_path).parent / "best_effort_repaired_plan.plan")
    repair_write_plan(plan, best_effort_path)

    report["status"] = "max-rounds-reached"
    report["message"] = "Repair did not finish within the maximum number of rounds."
    report["final_plan_path"] = best_effort_path
    report["final_plan"] = plan

    write_json(json_report_path, repair_json_safe(report))
    repair_save_readable_report(report, txt_report_path)

    return report


In [ ]:
# ============================================================
# Cell 14. NON_EXISTENCE branch: VAL invalidity search
# ============================================================

def run_val_validation(
    domain_file: str,
    problem_file: str,
    plan_file: str,
    val_bin: str,
    trace_out: str
) -> Dict[str, Any]:
    """
    Runs VAL and saves the raw VAL output.

    Status:
    - VALID: plan is valid
    - REPAIRABLE: VAL gives repair advice
    - FAILED: plan failed to execute
    - ERROR: parser/runtime/VAL error
    """

    command = [
        val_bin,
        domain_file,
        problem_file,
        plan_file
    ]

    try:
        completed = subprocess.run(
            command,
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            text=True,
            timeout=120
        )

        combined_output = completed.stdout + "\n" + completed.stderr

        Path(trace_out).parent.mkdir(parents=True, exist_ok=True)
        Path(trace_out).write_text(combined_output, encoding="utf-8")

        output_lower = combined_output.lower()

        if "plan valid" in output_lower or "successful plans" in output_lower:
            status = "VALID"
            status_code = 0

        elif "repair" in output_lower or "plan repair advice" in output_lower:
            status = "REPAIRABLE"
            status_code = 4

        elif (
            "error" in output_lower
            or "failed to parse" in output_lower
            or "syntax" in output_lower
            or "segmentation fault" in output_lower
        ):
            status = "ERROR"
            status_code = 2

        elif (
            "plan failed to execute" in output_lower
            or "failed plans" in output_lower
            or "plan failed" in output_lower
            or "unsatisfied precondition" in output_lower
        ):
            status = "FAILED"
            status_code = 1

        else:
            # VAL sometimes gives short output.
            # If it is not clearly valid, treat it as failed.
            status = "FAILED"
            status_code = 1

        return {
            "status": status,
            "status_code": status_code,
            "return_code": completed.returncode,
            "stdout": completed.stdout,
            "stderr": completed.stderr,
            "combined_output": combined_output,
            "trace_file": trace_out,
            "command": " ".join(command)
        }

    except FileNotFoundError:
        message = f"VAL binary not found: {val_bin}"
        Path(trace_out).parent.mkdir(parents=True, exist_ok=True)
        Path(trace_out).write_text(message, encoding="utf-8")

        return {
            "status": "ERROR",
            "status_code": 2,
            "return_code": None,
            "stdout": "",
            "stderr": message,
            "combined_output": message,
            "trace_file": trace_out,
            "command": " ".join(command)
        }

    except subprocess.TimeoutExpired:
        message = "VAL validation timed out."
        Path(trace_out).parent.mkdir(parents=True, exist_ok=True)
        Path(trace_out).write_text(message, encoding="utf-8")

        return {
            "status": "ERROR",
            "status_code": 2,
            "return_code": None,
            "stdout": "",
            "stderr": message,
            "combined_output": message,
            "trace_file": trace_out,
            "command": " ".join(command)
        }

In [ ]:
def validate_plan_at_node_with_val(
    lattice_data,
    lattice_json_path,
    node_id,
    plan_file,
    trace_dir,
    val_bin
):
    paths = get_domain_problem_paths_for_node(
        lattice_data=lattice_data,
        lattice_json_path=lattice_json_path,
        node_id=node_id
    )

    trace_out = os.path.join(trace_dir, f"{node_id}_val_trace.txt")

    result = run_val_validation(
        domain_file=paths["domain"],
        problem_file=paths["problem"],
        plan_file=plan_file,
        val_bin=val_bin,
        trace_out=trace_out
    )

    result["node"] = node_id
    result["domain"] = paths["domain"]
    result["problem"] = paths["problem"]
    result["plan"] = plan_file

    return result

In [ ]:
def run_lattice_invalidity_search_with_val(
    lattice_data: Dict[str, Any],
    lattice_json_path: str,
    start_node_ids: List[str],
    plan_file: str,
    direction: str,
    trace_dir: str,
    val_bin: str,
    stop_on_first_invalid: bool = True,
    exclude_nodes: Optional[Set[str]] = None
) -> Dict[str, Any]:
    """
    Traverses lattice and uses VAL to find the first invalid node.

    FULL is excluded by default so the invalid-path explanation search does not
    validate or select the concrete FULL node. The search stays within the
    stakeholder-declared abstraction/bridge chain.
    """

    ensure_dir(trace_dir)

    nodes = lattice_data.get("nodes", {})
    exclude_nodes = set(exclude_nodes or {"FULL"})

    queue = deque()
    visited = set()

    for node_id in start_node_ids:
        if node_id in exclude_nodes:
            print(f"Start node is excluded from invalidity search: {node_id}. Skipping.")
            continue

        if node_id not in nodes:
            print(f"Start node not found: {node_id}. Skipping.")
            continue

        if node_id not in visited:
            visited.add(node_id)
            queue.append(node_id)

    valid_nodes = []
    invalid_nodes = []
    error_nodes = []
    detailed_results = []

    first_invalid_node = None
    first_invalid_result = None

    while queue:
        current = queue.popleft()

        if current in exclude_nodes:
            continue

        print(f"\n>>> VAL checking node: {current} L{get_node_level(current, lattice_data)}")

        node_result = validate_plan_at_node_with_val(
            lattice_data=lattice_data,
            lattice_json_path=lattice_json_path,
            node_id=current,
            plan_file=plan_file,
            trace_dir=trace_dir,
            val_bin=val_bin
        )

        detailed_results.append(node_result)

        status = node_result.get("status")

        print("VAL status:", status)
        print("Trace:", node_result.get("trace_file"))

        if status == "VALID":
            valid_nodes.append(current)

        elif status == "ERROR":
            error_nodes.append(current)

            first_invalid_node = current
            first_invalid_result = node_result

            if stop_on_first_invalid:
                print("Stopping at first invalid/error node:", current)
                break

        else:
            invalid_nodes.append(current)

            first_invalid_node = current
            first_invalid_result = node_result

            if stop_on_first_invalid:
                print("Stopping at first invalid node:", current)
                break

        neighbors = get_neighbors_from_lattice_json(
            lattice_data=lattice_data,
            node_id=current,
            direction=direction
        )

        for nb in neighbors:
            if nb in exclude_nodes:
                continue

            if nb not in visited:
                visited.add(nb)
                queue.append(nb)

    return {
        "pipeline": "invalidity_with_val_no_full_search",
        "start_nodes": start_node_ids,
        "direction": direction,
        "excluded_nodes": sorted(exclude_nodes),
        "full_model_checked_during_search": False,
        "visited_nodes": [item["node"] for item in detailed_results],
        "valid_nodes": valid_nodes,
        "invalid_nodes": invalid_nodes,
        "error_nodes": error_nodes,
        "first_invalid_node": first_invalid_node,
        "first_invalid_result": first_invalid_result,
        "detailed_results": detailed_results
    }

In [ ]:
# ============================================================
# Cell 15. NON_EXISTENCE branch: custom invalid-plan diagnosis
# ============================================================

def diagnose_invalid_plan(
    domain_path: str,
    problem_path: str,
    plan_path: str,
    output_json_path: str,
    output_txt_path: str
) -> Dict[str, Any]:
    """
    Step-by-step invalidity diagnosis after VAL selects the first invalid node.

    This matches the working MeansEnd behavior:
    - find the first failing action
    - report missing preconditions
    """

    domain_text = read_text(domain_path)
    problem_text = read_text(problem_path)

    domain_actions = extract_pddl_actions(domain_text)
    initial_state = extract_initial_state_atoms(problem_text)
    plan_actions = parse_plan_file(plan_path)

    current_state = set(initial_state)
    execution_trace = []
    first_failure = None

    for step_index, plan_action in enumerate(plan_actions, start=1):
        action_name = plan_action["action_name"]

        if action_name not in domain_actions:
            first_failure = {
                "step_number": step_index,
                "plan_action": plan_action["raw"],
                "failure_type": "missing_action_schema",
                "message": f"Action {action_name} does not exist in the domain.",
                "missing_preconditions": []
            }

            execution_trace.append({
                "step_number": step_index,
                "plan_action": plan_action["raw"],
                "action_name": action_name,
                "status": "FAILED",
                "failure_type": "missing_action_schema",
                "grounded_preconditions": [],
                "grounded_effects": [],
                "missing_preconditions": []
            })

            break

        action_block = domain_actions[action_name]

        params = extract_action_parameters(action_block)
        args = plan_action["args"]

        binding = {}

        for i, param in enumerate(params):
            if i < len(args):
                binding[param] = args[i]

        precondition_text = extract_pddl_section(action_block, "precondition")
        effect_text = extract_pddl_section(action_block, "effect")

        precondition_atoms = extract_predicate_atoms(precondition_text)
        effect_atoms = extract_predicate_atoms(effect_text)

        grounded_preconditions = {
            ground_atom(atom, binding)
            for atom in precondition_atoms
        }

        grounded_effects = {
            ground_atom(atom, binding)
            for atom in effect_atoms
        }

        missing_preconditions = sorted(
            p for p in grounded_preconditions
            if p not in current_state
        )

        step_record = {
            "step_number": step_index,
            "plan_action": plan_action["raw"],
            "action_name": action_name,
            "binding": binding,
            "grounded_preconditions": sorted(grounded_preconditions),
            "grounded_effects": sorted(grounded_effects),
            "missing_preconditions": missing_preconditions,
            "state_before_size": len(current_state)
        }

        if missing_preconditions:
            step_record["status"] = "FAILED"
            step_record["failure_type"] = "unsatisfied_preconditions"
            execution_trace.append(step_record)

            first_failure = {
                "step_number": step_index,
                "plan_action": plan_action["raw"],
                "failure_type": "unsatisfied_preconditions",
                "missing_preconditions": missing_preconditions,
                "message": (
                    f"Step {step_index} failed because required preconditions "
                    "were not true before executing the action."
                )
            }

            break

        current_state.update(grounded_effects)

        step_record["status"] = "EXECUTED"
        step_record["state_after_size"] = len(current_state)

        execution_trace.append(step_record)

    plan_executed = first_failure is None

    diagnosis = {
        "domain": domain_path,
        "problem": problem_path,
        "plan": plan_path,
        "plan_executed": plan_executed,
        "first_failure": first_failure,
        "execution_trace": execution_trace,
        "final_state_size": len(current_state)
    }

    write_json(output_json_path, diagnosis)

    report_lines = []
    report_lines.append("=== INVALID PLAN DIAGNOSIS ===")
    report_lines.append("")
    report_lines.append(f"Domain: {domain_path}")
    report_lines.append(f"Problem: {problem_path}")
    report_lines.append(f"Plan: {plan_path}")
    report_lines.append("")

    if plan_executed:
        report_lines.append("Result: The plan executed successfully under this structural simulator.")
        report_lines.append("")
        report_lines.append(
            "Note: VAL selected this node as invalid, but the lightweight step simulator "
            "did not find a missing action precondition. Check whether the final goal is missing "
            "or whether VAL is using semantics not captured by this simulator."
        )
    else:
        report_lines.append("Result: The plan failed.")
        report_lines.append("")
        report_lines.append(f"First failing step: {first_failure['step_number']}")
        report_lines.append(f"Failing action: {first_failure['plan_action']}")
        report_lines.append(f"Failure type: {first_failure['failure_type']}")
        report_lines.append("")
        report_lines.append("Missing preconditions:")
        for p in first_failure["missing_preconditions"]:
            report_lines.append(f"  - {p}")

    report_lines.append("")
    report_lines.append("Step trace:")

    for step in execution_trace:
        report_lines.append("")
        report_lines.append(f"Step {step['step_number']}: {step['plan_action']}")
        report_lines.append(f"Status: {step['status']}")

        report_lines.append("Required preconditions:")
        for p in step["grounded_preconditions"]:
            report_lines.append(f"  - {p}")

        if step["missing_preconditions"]:
            report_lines.append("Missing preconditions:")
            for p in step["missing_preconditions"]:
                report_lines.append(f"  - {p}")

        report_lines.append("Effects:")
        for e in step["grounded_effects"]:
            report_lines.append(f"  - {e}")

    write_text(output_txt_path, "\n".join(report_lines))

    return diagnosis

In [ ]:

# ============================================================
# Concrete/FULL consistency guard for NON_EXISTENCE queries
# ============================================================

def check_plan_in_full_model_first(
    lattice_data: Dict[str, Any],
    lattice_json_path: str,
    plan_file: str,
    trace_dir: str,
    val_bin: str
) -> Dict[str, Any]:
    """
    Checks the mapped user plan once in the concrete/FULL model before
    invalid-path diagnosis.

    Purpose:
    - This is NOT part of the stakeholder/bridge explanation-node search.
    - It is a consistency guard for NON_EXISTENCE queries.
    - If the path is valid in FULL, then the user's expectation that the
      path is invalid is not supported by the concrete CPS model, so the
      invalidity explanation should stop and report that mismatch.
    """

    ensure_dir(trace_dir)

    paths = get_domain_problem_paths_for_node(
        lattice_data=lattice_data,
        lattice_json_path=lattice_json_path,
        node_id="FULL"
    )

    trace_out = os.path.join(trace_dir, "FULL_concrete_consistency_guard_val_trace.txt")

    full_result = run_val_validation(
        domain_file=paths["domain"],
        problem_file=paths["problem"],
        plan_file=plan_file,
        val_bin=val_bin,
        trace_out=trace_out
    )

    full_result["node"] = "FULL"
    full_result["domain"] = paths["domain"]
    full_result["problem"] = paths["problem"]
    full_result["plan"] = plan_file
    full_result["purpose"] = "concrete_model_consistency_guard_for_invalidity_query"
    full_result["used_for_explanation_node_search"] = False

    return full_result


In [ ]:
# ============================================================
# Cell 16. Pipeline wrappers
# ============================================================

def run_validity_pipeline() -> Dict[str, Any]:
    print("\n==============================")
    print("RUNNING VALIDITY PIPELINE")
    print("==============================")

    lattice_data = read_json(LATTICE_JSON)

    # Read the stakeholder view declared in the query file once and use it
    # throughout the validity pipeline. No stakeholder inference is performed.
    stakeholder = find_declared_stakeholder_from_json(STAKEHOLDER_JSON)
    start_node = start_node_from_stakeholder(stakeholder)
    branch_prefix = infer_branch_prefix(start_node)

    # --------------------------------------------------------
    # Step 1. Repair the mapped user plan in the FULL model.
    # This is repair/grounding support only, not explanation-node search.
    # The valid-path explanation search below excludes FULL.
    # --------------------------------------------------------
    print("\n==============================")
    print("STEP 1: PLAN REPAIR IN FULL MODEL")
    print("==============================")

    repair_report = repair_plan_by_inserting_actions(
        domain_path=FULL_DOMAIN_PDDL,
        problem_path=FULL_PROBLEM_PDDL,
        user_plan_path=PLAN_FILE,
        repaired_plan_path=REPAIRED_PLAN_FILE,
        json_report_path=PLAN_REPAIR_JSON,
        txt_report_path=PLAN_REPAIR_TXT,
        max_rounds=5,
        repair_depth=5
    )

    if repair_report.get("status") != "valid":
        print("\nPlan repair failed. MEA will not run on an unrepaired invalid plan.")
        print("Repair status:", repair_report.get("status"))
        print("Repair report:", PLAN_REPAIR_JSON)

        final_mea_output = {
            "pipeline": "validity_no_full_search",
            "intent": "EXISTENCE",
            "declared_stakeholder": stakeholder,
            "stakeholder_source": "declared_in_query_file",
            "stakeholder_inference_used": False,
            "predicted_stakeholder": stakeholder,  # backward-compatible key; not inferred
            "start_node": start_node,
            "branch_prefix": branch_prefix,
            "status": "plan_repair_failed",
            "message": (
                "The user asked for a valid path explanation, but the plan could not "
                "be repaired into a valid executable plan. Therefore backward MEA was not run."
            ),
            "original_plan_file": PLAN_FILE,
            "repaired_plan_file": repair_report.get("final_plan_path"),
            "plan_repair_json": PLAN_REPAIR_JSON,
            "plan_repair_txt": PLAN_REPAIR_TXT,
            "plan_repair_result": repair_report,
            "mea_was_run": False,
            "abstract_valid_node": None,
            "abstract_valid_result": None,
            "first_valid_node": None,
            "last_valid_node": None,
            "valid_nodes": [],
            "invalid_nodes": [],
            "visited_nodes": [],
            "search_chain": [],
            "search_excludes_full_model": True,
            "full_model_checked_as_consistency_guard": True,
            "full_model_used_for_plan_repair": True,
            "full_model_checked_during_search": False,
            "full_model_used_as_explanation_node": False,
            "full_concrete_validity_guard_status": repair_report.get("status"),
            "full_concrete_validity_guard_message": repair_report.get("message"),
            "full_result": None,
            "full_valid_by_mea": None,
            "full_unachieved_subgoals": [],
            "relaxed_constraints": []
        }

        write_json(MEA_OUTPUT_JSON, final_mea_output)

        report_text = generate_mea_report(final_mea_output)
        write_text(MEA_OUTPUT_REPORT, report_text)

        return final_mea_output

    plan_file_for_mea = repair_report.get("final_plan_path", REPAIRED_PLAN_FILE)

    print("\nPlan file used for validity MEA:")
    print(plan_file_for_mea)

    # --------------------------------------------------------
    # Step 2. Build the bridge-aware lattice search chain.
    # Important: FULL is excluded from the explanation-node search.
    # --------------------------------------------------------
    search_chain = derive_downward_chain_from_lattice_json(
        lattice_data=lattice_data,
        start_node=start_node,
        branch_prefix=branch_prefix,
        include_bridge_nodes=True,
        include_full_model=False
    )
    search_chain = [node for node in search_chain if node != "FULL"]

    print("\nBridge-aware search chain excluding FULL:")
    print(" -> ".join(search_chain))

    print("\n=== DOMAIN/PROBLEM PATHS USED BY MEA ===")
    for node_id in search_chain:
        paths = get_domain_problem_paths_for_node(
            lattice_data=lattice_data,
            lattice_json_path=LATTICE_JSON,
            node_id=node_id
        )

        print("\nNode:", node_id)
        print("Domain:", paths["domain"])
        print("Domain exists:", os.path.exists(paths["domain"]))
        print("Problem:", paths["problem"])
        print("Problem exists:", os.path.exists(paths["problem"]))

    # --------------------------------------------------------
    # Step 3. Run backward MEA on the repaired plan over the
    # abstraction/bridge chain only. FULL is not checked here.
    # --------------------------------------------------------
    plan_actions = parse_plan_file(plan_file_for_mea)
    goal_predicate = read_goal_predicate(GOAL_FILE)

    mea_search_result = search_lattice_with_mea(
        lattice_data=lattice_data,
        lattice_json_path=LATTICE_JSON,
        search_chain=search_chain,
        plan_actions=plan_actions,
        goal_predicate=goal_predicate,
        stop_on_first_valid=False,
        # Do not stop at the first valid bridge. We need to inspect all bridge
        # candidates and select the first one whose backward-MEA trace contains
        # at least one precondition and one MEA-relevant postcondition.
        stop_on_first_valid_bridge=False
    )
    mea_search_result["search_excludes_full_model"] = True
    mea_search_result["full_model_checked_during_search"] = False

    first_valid_result = mea_search_result.get("first_valid_result")
    first_valid_node = mea_search_result.get("first_valid_node")

    first_bridge_valid_node = mea_search_result.get("first_valid_bridge_node")
    first_bridge_valid_result = mea_search_result.get("first_valid_bridge_result")

    (
        structured_bridge_valid_node,
        structured_bridge_valid_result,
        bridge_selection_candidates
    ) = select_first_valid_bridge_with_min_pre_eff(
        mea_search_result=mea_search_result,
        lattice_data=lattice_data,
        min_preconditions=1,
        min_effects=1
    )

    (
        structured_valid_node,
        structured_valid_result,
        pre_post_selection_candidates
    ) = select_first_valid_node_with_min_pre_eff(
        mea_search_result=mea_search_result,
        min_preconditions=1,
        min_effects=1
    )

    if structured_bridge_valid_result is not None:
        selected_valid_node = structured_bridge_valid_node
        selected_valid_result = structured_bridge_valid_result
        selected_policy = "first_valid_bridge_node_with_at_least_one_precondition_and_one_mea_postcondition_after_plan_repair_no_full_search"
    elif structured_valid_result is not None:
        selected_valid_node = structured_valid_node
        selected_valid_result = structured_valid_result
        selected_policy = "first_valid_node_in_bridge_aware_chain_with_at_least_one_precondition_and_one_mea_postcondition_after_plan_repair_no_full_search"
    else:
        selected_valid_node = None
        selected_valid_result = None
        selected_policy = "no_valid_node_satisfied_required_pre_post_constraint_for_explanation_no_full_search"

    if selected_valid_result:
        abstract_preconditions = collect_plan_preconditions(selected_valid_result)
    else:
        abstract_preconditions = set()

    # FULL is intentionally not computed here. Keep the legacy fields as None/[]
    # so older report/explanation code does not break, but do not run FULL MEA.
    full_result = None
    U_full = set()
    relaxed_constraints = []

    final_mea_output = {
        "pipeline": "validity_no_full_search",
        "intent": "EXISTENCE",
        "declared_stakeholder": stakeholder,
        "stakeholder_source": "declared_in_query_file",
        "stakeholder_inference_used": False,
        "predicted_stakeholder": stakeholder,  # backward-compatible key; not inferred
        "start_node": start_node,
        "branch_prefix": branch_prefix,
        "goal": goal_predicate,

        # Original mapped plan and repaired plan are both recorded.
        "original_plan_file": PLAN_FILE,
        "plan_file": plan_file_for_mea,
        "repaired_plan_file": plan_file_for_mea,
        "plan_repair_json": PLAN_REPAIR_JSON,
        "plan_repair_txt": PLAN_REPAIR_TXT,
        "plan_repair_result": repair_report,

        "search_result": mea_search_result,
        "first_valid_node": first_valid_node,
        "last_valid_node": mea_search_result.get("last_valid_node"),
        "valid_nodes": mea_search_result.get("valid_nodes", []),
        "invalid_nodes": mea_search_result.get("invalid_nodes", []),
        "visited_nodes": mea_search_result.get("visited_nodes", []),
        "search_chain": mea_search_result.get("search_chain", []),
        "search_excludes_full_model": True,
        "full_model_checked_as_consistency_guard": True,
        "full_model_used_for_plan_repair": True,
        "full_model_checked_during_search": False,
        "full_model_used_as_explanation_node": False,
        "full_concrete_validity_guard_status": repair_report.get("status"),
        "full_concrete_validity_guard_message": repair_report.get("message"),

        # Bridge selection evidence.
        "first_bridge_valid_node": first_bridge_valid_node,
        "first_bridge_valid_result": first_bridge_valid_result,
        "first_structured_bridge_valid_node": structured_bridge_valid_node,
        "first_structured_bridge_valid_result": structured_bridge_valid_result,
        "bridge_selection_constraint": (
            "valid bridge node in the bridge chain whose backward-MEA trace has "
            "at least one precondition and at least one MEA-relevant postcondition"
        ),
        "bridge_selection_candidates": bridge_selection_candidates,
        "first_structured_valid_node": structured_valid_node,
        "first_structured_valid_result": structured_valid_result,
        "pre_post_selection_candidates": pre_post_selection_candidates,

        "abstract_valid_node": selected_valid_node,
        "abstract_valid_result": selected_valid_result,
        "selected_valid_node_policy": selected_policy,

        "full_result": full_result,
        "full_valid_by_mea": None,
        "full_unachieved_subgoals": [],
        "abstract_preconditions": sorted(abstract_preconditions),
        "relaxed_constraints": relaxed_constraints
    }

    write_json(MEA_OUTPUT_JSON, final_mea_output)

    report_text = generate_mea_report(final_mea_output)
    write_text(MEA_OUTPUT_REPORT, report_text)

    print("\nValidity pipeline complete.")
    print("Plan repair status:", repair_report.get("status"))
    print("Repaired plan:", plan_file_for_mea)
    print("FULL checked as validity/repair guard:", final_mea_output.get("full_model_checked_as_consistency_guard"))
    print("FULL checked during explanation-node search:", final_mea_output.get("full_model_checked_during_search"))
    print("FULL used as explanation node:", final_mea_output.get("full_model_used_as_explanation_node"))
    print("Search excludes FULL:", final_mea_output["search_excludes_full_model"])
    print("Selected valid node:", final_mea_output["abstract_valid_node"])
    print("Selected policy:", final_mea_output["selected_valid_node_policy"])
    print("First valid node:", final_mea_output["first_valid_node"])
    print("First bridge valid node:", final_mea_output.get("first_bridge_valid_node"))
    print("First structured bridge valid node:", final_mea_output.get("first_structured_bridge_valid_node"))
    print("Last valid node visited:", final_mea_output["last_valid_node"])
    print("Saved:", MEA_OUTPUT_JSON)

    return final_mea_output


def run_invalidity_pipeline() -> Dict[str, Any]:
    """
    NON_EXISTENCE branch.

    Current flow:
    1. Use the stakeholder view declared in the query file.
    2. Run one FULL/concrete-model consistency guard.
       - If FULL says VALID, stop and report that the path is not actually
         invalid in the concrete model.
       - This FULL check is not used as an explanation node.
    3. If FULL is not valid, search from the declared stakeholder start node.
    4. Exclude FULL from the invalidity explanation-node search.
    5. Run custom diagnosis on the first invalid abstraction/bridge node.
    """

    print("\n==============================")
    print("RUNNING INVALIDITY PIPELINE WITH FULL CONSISTENCY GUARD")
    print("==============================")

    check_file_exists(VAL_BIN, "VAL binary")

    lattice_data = read_json(LATTICE_JSON)

    stakeholder = find_declared_stakeholder_from_json(STAKEHOLDER_JSON)
    start_node = start_node_from_stakeholder(stakeholder)

    # --------------------------------------------------------
    # Step 1. FULL/concrete consistency guard.
    # This answers: is the path that the user thinks is invalid actually
    # valid in the concrete CPS model?
    # --------------------------------------------------------
    print("\n==============================")
    print("STEP 1: FULL CONSISTENCY GUARD FOR INVALIDITY QUERY")
    print("==============================")

    full_concrete_check = check_plan_in_full_model_first(
        lattice_data=lattice_data,
        lattice_json_path=LATTICE_JSON,
        plan_file=PLAN_FILE,
        trace_dir=INVALIDITY_TRACE_DIR,
        val_bin=VAL_BIN
    )

    print("FULL consistency-guard status:", full_concrete_check.get("status"))
    print("FULL consistency-guard trace:", full_concrete_check.get("trace_file"))

    if full_concrete_check.get("status") == "VALID":
        invalidity_result = {
            "pipeline": "invalidity_with_full_consistency_guard_no_full_search",
            "intent": "NON_EXISTENCE",
            "declared_stakeholder": stakeholder,
            "stakeholder_source": "declared_in_query_file",
            "stakeholder_inference_used": False,
            "predicted_stakeholder": stakeholder,  # backward-compatible key; not inferred
            "start_node": start_node,
            "status": "not_invalid_in_concrete_model",
            "message": (
                "The user asked for an invalid-path explanation, but the mapped path "
                "is valid in the concrete/FULL model. Therefore the path is not "
                "invalid with respect to the concrete CPS model, and no first "
                "invalid abstraction/bridge node is diagnosed."
            ),
            "full_concrete_check": full_concrete_check,
            "full_model_checked_as_consistency_guard": True,
            "full_model_checked_during_search": False,
            "full_model_used_as_explanation_node": False,
            "search_excludes_full_model": True,
            "excluded_nodes": ["FULL"],
            "visited_nodes": [],
            "valid_nodes": [],
            "invalid_nodes": [],
            "error_nodes": [],
            "first_invalid_node": None,
            "first_invalid_result": None,
            "detailed_results": []
        }

        write_json(INVALIDITY_RESULT_JSON, invalidity_result)

        print("\nInvalidity pipeline stopped.")
        print("Reason: path is valid in the concrete/FULL model.")
        print("Saved invalidity result:", INVALIDITY_RESULT_JSON)

        return {
            "invalidity_result": invalidity_result,
            "diagnosis": {}
        }

    # --------------------------------------------------------
    # Step 2. Stakeholder/bridge invalidity search.
    # FULL is excluded here. The search identifies where the path first
    # becomes invalid in the stakeholder-visible abstraction chain.
    # --------------------------------------------------------
    print("\n==============================")
    print("STEP 2: STAKEHOLDER/BRIDGE INVALIDITY SEARCH EXCLUDING FULL")
    print("==============================")

    invalidity_result = run_lattice_invalidity_search_with_val(
        lattice_data=lattice_data,
        lattice_json_path=LATTICE_JSON,
        start_node_ids=[start_node],
        plan_file=PLAN_FILE,
        direction="down",
        trace_dir=INVALIDITY_TRACE_DIR,
        val_bin=VAL_BIN,
        stop_on_first_invalid=True,
        exclude_nodes={"FULL"}
    )

    invalidity_result["pipeline"] = "invalidity_with_full_consistency_guard_no_full_search"
    invalidity_result["intent"] = "NON_EXISTENCE"
    invalidity_result["declared_stakeholder"] = stakeholder
    invalidity_result["stakeholder_source"] = "declared_in_query_file"
    invalidity_result["stakeholder_inference_used"] = False
    invalidity_result["predicted_stakeholder"] = stakeholder  # backward-compatible key; not inferred
    invalidity_result["start_node"] = start_node
    invalidity_result["search_excludes_full_model"] = True
    invalidity_result["full_model_checked_as_consistency_guard"] = True
    invalidity_result["full_model_checked_during_search"] = False
    invalidity_result["full_model_used_as_explanation_node"] = False
    invalidity_result["full_concrete_check"] = full_concrete_check
    invalidity_result["status"] = "invalidity_lattice_search_no_full_after_full_guard"

    first_invalid_result = invalidity_result.get("first_invalid_result")

    if first_invalid_result is None:
        invalidity_result["status"] = "no_invalid_node_found_before_full"
        invalidity_result["message"] = (
            "The concrete/FULL consistency guard did not validate the path, but no "
            "invalid node was found in the stakeholder/bridge search chain before FULL. "
            "FULL was excluded from explanation-node search."
        )
        write_json(INVALIDITY_RESULT_JSON, invalidity_result)
        raise ValueError(
            "VAL did not find any invalid node before FULL. FULL was used only as a "
            "consistency guard and was excluded from explanation-node search."
        )

    write_json(INVALIDITY_RESULT_JSON, invalidity_result)

    diagnosis = diagnose_invalid_plan(
        domain_path=first_invalid_result.get("domain"),
        problem_path=first_invalid_result.get("problem"),
        plan_path=first_invalid_result.get("plan"),
        output_json_path=INVALID_DIAGNOSIS_JSON,
        output_txt_path=INVALID_DIAGNOSIS_TXT
    )

    # Add search context to the diagnosis so the explanation generator can
    # connect the failing predicates to the selected invalid node without
    # using FULL as an explanation node.
    diagnosis["invalidity_search_context"] = {
        "selected_invalid_node": invalidity_result.get("first_invalid_node"),
        "selected_invalid_result_status": first_invalid_result.get("status"),
        "selected_invalid_trace_file": first_invalid_result.get("trace_file"),
        "full_model_checked_as_consistency_guard": invalidity_result.get("full_model_checked_as_consistency_guard", True),
        "full_model_checked_during_search": invalidity_result.get("full_model_checked_during_search", False),
        "full_model_used_as_explanation_node": invalidity_result.get("full_model_used_as_explanation_node", False),
        "search_excludes_full_model": invalidity_result.get("search_excludes_full_model", True)
    }
    write_json(INVALID_DIAGNOSIS_JSON, diagnosis)

    print("\nInvalidity pipeline complete.")
    print("FULL checked as consistency guard:", invalidity_result.get("full_model_checked_as_consistency_guard"))
    print("FULL checked during explanation-node search:", invalidity_result.get("full_model_checked_during_search"))
    print("Search excludes FULL:", invalidity_result.get("search_excludes_full_model"))
    print("First invalid node:", invalidity_result.get("first_invalid_node"))
    print("VAL status:", first_invalid_result.get("status"))
    print("VAL trace:", first_invalid_result.get("trace_file"))
    print("Diagnosis first failure:")
    print(json.dumps(diagnosis.get("first_failure"), indent=2))
    print("Saved invalidity result:", INVALIDITY_RESULT_JSON)
    print("Saved diagnosis:", INVALID_DIAGNOSIS_JSON)
    print("Saved readable diagnosis:", INVALID_DIAGNOSIS_TXT)

    return {
        "invalidity_result": invalidity_result,
        "diagnosis": diagnosis
    }


In [ ]:
# ============================================================
# Cell 17. Final controller
# ============================================================

def check_required_files_for_controller() -> None:
    required = {
        "path intent JSON": PATH_INTENT_JSON,
        "declared stakeholder/action mapping JSON": STAKEHOLDER_JSON,
        "mapped plan file": PLAN_FILE,
        "goal predicate file": GOAL_FILE,
        "lattice JSON": LATTICE_JSON,
    }

    missing = []

    for label, path in required.items():
        if not os.path.exists(path):
            missing.append((label, path))

    if missing:
        print("Missing files:")
        for label, path in missing:
            print(f"- {label}: {path}")

        raise FileNotFoundError(
            "Run preprocessing stages before running the controller."
        )

    print("All required files for controller are available.")


def run_earg_validity_or_invalidity_pipeline() -> Dict[str, Any]:
    check_required_files_for_controller()

    intent_result = find_path_intent_from_json(PATH_INTENT_JSON)
    intent = intent_result.get("intent", "UNKNOWN")

    print("Detected path intent:", intent)

    if intent == "EXISTENCE":
        result = run_validity_pipeline()

        return {
            "intent": intent,
            "ran": "validity_pipeline",
            "result": result
        }

    elif intent == "NON_EXISTENCE":
        result = run_invalidity_pipeline()

        return {
            "intent": intent,
            "ran": "invalidity_pipeline",
            "result": result
        }

    else:
        raise ValueError(
            f"Unknown path intent: {intent}. Expected EXISTENCE or NON_EXISTENCE."
        )

In [ ]:
# ============================================================
# Cell 19. Explanation helper functions
# ============================================================

def format_user_steps_for_prompt(user_data: Dict[str, Any]) -> str:
    steps = user_data.get("steps", [])

    if not steps:
        return "No user steps were provided."

    lines = []
    for i, step in enumerate(steps, start=1):
        lines.append(f"{i}. {step}")

    return "\n".join(lines)


def get_selected_first_valid_result(mea_output: Dict[str, Any]) -> Optional[Dict[str, Any]]:
    """
    Selects the first valid MEA result.

    Priority:
    1. abstract_valid_result
    2. search_result.first_valid_result
    3. search_result.last_valid_result as fallback
    """
    selected = mea_output.get("abstract_valid_result")

    if selected is not None:
        return selected

    selected = mea_output.get("search_result", {}).get("first_valid_result")

    if selected is not None:
        return selected

    return mea_output.get("search_result", {}).get("last_valid_result")


def extract_predicate_dependency_chain(selected_result: Dict[str, Any]) -> List[Dict[str, Any]]:
    """
    Converts backward MEA trace into predicate-only dependency links.

    Important:
    This intentionally does NOT include action names.

    Each link says:
      required predicates -> achieved predicates

    Example:
      (plc-offline plc) -> (has-fault-valve-blocked-close-due-to-compromised plc)
    """

    predicate_links = []

    for item in selected_result.get("backward_trace", []):
        achieved = item.get("achieved_subgoals_removed", [])
        required = item.get("preconditions_added", [])
        subgoals_before = item.get("subgoals_before", [])
        subgoals_after = item.get("subgoals_after", [])

        # Ignore actions that do not participate in the selected abstraction's
        # predicate dependency chain.
        if not achieved and not required:
            continue

        predicate_links.append({
            "original_step": item.get("original_step"),
            "required_predicates": required,
            "achieved_predicates": achieved,
            "subgoals_before": subgoals_before,
            "subgoals_after": subgoals_after
        })

    return predicate_links


def dedupe_preserve_order(items: List[Any]) -> List[Any]:
    """Deduplicates a list while preserving order."""
    seen = set()
    out = []

    for item in items:
        key = str(item)
        if key not in seen:
            seen.add(key)
            out.append(item)

    return out


def extract_repair_predicate_summary(mea_output: Dict[str, Any]) -> Dict[str, Any]:
    """
    Extracts repair information for the valid-path explanation.

    The explanation now reports:
    1. whether the user path needed repair,
    2. how many missing actions were inserted, and
    3. which missing predicates those inserted actions established.
    """

    repair = mea_output.get("plan_repair_result", {}) or {}
    rounds = repair.get("repair_rounds", []) or []

    plan_was_repaired = len(rounds) > 0

    missing_predicates = []
    repair_goal_predicates = []
    inserted_action_names = []
    inserted_actions_by_round = []

    for r in rounds:
        round_inserted = r.get("inserted_actions_string", []) or []

        missing_predicates.extend(
            r.get("missing_positive_preconditions_string", []) or []
        )

        repair_goal_predicates.extend(
            r.get("repair_goal_string", []) or []
        )

        inserted_action_names.extend(round_inserted)

        inserted_actions_by_round.append({
            "round": r.get("round"),
            "repair_type": r.get("repair_type"),
            "failed_original_action": r.get("failed_action_string"),
            "missing_predicates_before_repair": r.get("missing_positive_preconditions_string", []) or [],
            "predicates_established_by_repair": r.get("repair_goal_string", []) or [],
            "inserted_action_count": len(round_inserted),
            "inserted_actions_for_trace_only": round_inserted
        })

    # Count every inserted action, not only unique inserted actions.
    # This is the number you want in the explanation when the repaired plan
    # contains multiple added steps.
    inserted_repair_action_count = len(inserted_action_names)

    return {
        "plan_was_repaired": plan_was_repaired,
        "repair_status": repair.get("status"),
        "repair_message": repair.get("message"),
        "repair_round_count": len(rounds),
        "inserted_repair_action_count": inserted_repair_action_count,
        "missing_action_added_count": inserted_repair_action_count,
        "missing_predicates_before_repair": dedupe_preserve_order(missing_predicates),
        "predicates_established_by_repair": dedupe_preserve_order(repair_goal_predicates),
        "inserted_actions_for_trace_only": dedupe_preserve_order(inserted_action_names),
        "inserted_actions_by_round": inserted_actions_by_round
    }



def load_repair_summary_for_explanation(
    repair_report_json: Optional[str],
    mea_output: Dict[str, Any]
) -> Dict[str, Any]:
    """
    Loads the plan-repair report explicitly for the explanation generator.

    Design intent:
    - Validity analysis records the repair report as trace evidence.
    - Defensive strategies are NOT precomputed by validity analysis.
    - The explanation generator receives this repair report and uses its
      predicate evidence to produce missing-knowledge and defense sections.
    """

    if repair_report_json and os.path.exists(repair_report_json):
        repair_report = read_json(repair_report_json)
        return extract_repair_predicate_summary({"plan_repair_result": repair_report})

    # Fallback for older runs where the repair report was embedded in the MEA JSON.
    return extract_repair_predicate_summary(mea_output)


def format_repair_report_evidence_for_prompt(repair_summary: Dict[str, Any]) -> str:
    """
    Formats repair-report predicate evidence for the explanation generator.
    This evidence is used for missing user knowledge and defensive strategies.
    """

    lines = []
    lines.append("REPAIR REPORT EVIDENCE")
    lines.append("Use this evidence only for missing-user-knowledge and defensive-strategy generation.")
    lines.append("Do not use inserted action names as the basis for defenses.")
    lines.append("")
    lines.append(f"Plan was repaired: {repair_summary.get('plan_was_repaired')}")
    lines.append(f"Repair status: {repair_summary.get('repair_status')}")
    lines.append(f"Repair message: {repair_summary.get('repair_message')}")
    lines.append(f"Number of repair rounds: {repair_summary.get('repair_round_count', 0)}")
    lines.append(f"Number of missing actions added: {repair_summary.get('missing_action_added_count', 0)}")
    lines.append("")

    missing = repair_summary.get("missing_predicates_before_repair", []) or []
    established = repair_summary.get("predicates_established_by_repair", []) or []

    lines.append("Missing predicates before repair:")
    if missing:
        for p in missing:
            lines.append(f"  - {p}")
    else:
        lines.append("  - None")

    lines.append("")
    lines.append("Predicates established by repair:")
    if established:
        for p in established:
            lines.append(f"  - {p}")
    else:
        lines.append("  - None")

    lines.append("")
    lines.append("Repair rounds, predicate view:")
    rounds = repair_summary.get("inserted_actions_by_round", []) or []
    if rounds:
        for r in rounds:
            lines.append(f"  Round {r.get('round')}:")
            lines.append(f"    Repair type: {r.get('repair_type')}")
            lines.append(f"    Inserted action count: {r.get('inserted_action_count', 0)}")
            round_missing = r.get("missing_predicates_before_repair", []) or []
            round_established = r.get("predicates_established_by_repair", []) or []
            lines.append("    Missing predicates:")
            if round_missing:
                for p in round_missing:
                    lines.append(f"      - {p}")
            else:
                lines.append("      - None")
            lines.append("    Repair-established predicates:")
            if round_established:
                for p in round_established:
                    lines.append(f"      - {p}")
            else:
                lines.append("      - None")
    else:
        lines.append("  - No repair rounds were recorded.")

    return "\n".join(lines)


def sanitize_repair_only_defensive_strategies(
    strategies: List[Dict[str, Any]],
    repair_summary: Dict[str, Any]
) -> List[Dict[str, Any]]:
    """
    Lightweight guardrail for LLM-generated defensive strategies.

    The explanation generator is responsible for creating defenses. This helper
    does not precompute defenses; it only removes strategies that are not tied to
    any repair predicate so the output remains bounded by missing-user-knowledge
    evidence.
    """

    repair_predicates = []
    repair_predicates.extend(repair_summary.get("missing_predicates_before_repair", []) or [])
    repair_predicates.extend(repair_summary.get("predicates_established_by_repair", []) or [])
    repair_predicates = [str(p).strip() for p in dedupe_preserve_order(repair_predicates) if str(p).strip()]

    if not repair_predicates:
        return []

    cleaned = []
    for item in strategies or []:
        if not isinstance(item, dict):
            continue

        evidence_text = str(item.get("supported_by_evidence", ""))
        combined_text = " ".join(str(v) for v in item.values())

        # Keep the strategy if it explicitly cites or mentions at least one repair predicate.
        if any(pred in evidence_text or pred in combined_text for pred in repair_predicates):
            cleaned.append(item)

    return cleaned



def derive_repair_report_defensive_strategy_fallback(
    repair_summary: Dict[str, Any]
) -> List[Dict[str, str]]:
    """
    Explanation-generator fallback for repair-predicate defenses.

    This function is intentionally called from the explanation generator, not from
    the validity analysis stage. It uses only the repair report predicates that
    were missing from the user's mental model.

    Why this is needed:
    - The LLM may omit defensive_strategies.
    - The LLM may provide strategies but fail to cite the exact repair predicate.
    - sanitize_repair_only_defensive_strategies() correctly removes unsupported
      strategies, but the final explanation should still include deterministic,
      repair-predicate-grounded defenses when repair predicates exist.
    """

    repair_predicates = []
    repair_predicates.extend(repair_summary.get("missing_predicates_before_repair", []) or [])
    repair_predicates.extend(repair_summary.get("predicates_established_by_repair", []) or [])
    repair_predicates = [
        str(p).strip()
        for p in dedupe_preserve_order(repair_predicates)
        if str(p).strip()
    ]

    if not repair_predicates:
        return []

    def predicate_head(predicate: str) -> str:
        """
        Return the predicate symbol only, not object names.

        This prevents over-grouping. For example, the object name
        allen-bradley-controllogix-plc should not make every predicate look like
        a PLC-port predicate or a process-control predicate.
        """
        text = str(predicate).strip()
        match = re.match(r"^\(\s*([^\s\)]+)", text)
        if match:
            return match.group(1).lower()
        return text.split()[0].lower() if text else ""

    predicate_heads = {p: predicate_head(p) for p in repair_predicates}

    strategies: List[Dict[str, str]] = []
    added_keys = set()

    def matched_predicates(*tokens: str) -> List[str]:
        toks = [str(t).lower() for t in tokens]
        return [
            p for p, head in predicate_heads.items()
            if any(t in head for t in toks)
        ]

    def add(key: str, strategy: str, why: str, predicate_matches: List[str]):
        if key in added_keys:
            return
        evidence = predicate_matches if predicate_matches else repair_predicates
        evidence = dedupe_preserve_order(evidence)
        if not evidence:
            return
        added_keys.add(key)
        strategies.append({
            "strategy": strategy,
            "why_it_stops_the_path": why,
            "supported_by_evidence": "Repair predicate(s): " + "; ".join(evidence),
            "source": "explanation_generator_repair_report_fallback"
        })

    network_matches = matched_predicates(
        "access-to-network", "internal-network", "external-network", "network-access"
    )
    if network_matches:
        add(
            "network_access",
            "Segment the network and require strong authentication before the missing network-access condition can become true.",
            "The repaired path needed a network-reachability predicate that was absent from the user's original explanation. Keeping that predicate false prevents the path from reaching PLC/process conditions.",
            network_matches
        )

    plc_port_matches = matched_predicates(
        "access-to-plc-port", "plc-port", "port-1132", "1132-tcp"
    )
    if plc_port_matches:
        add(
            "plc_port_access",
            "Allowlist PLC communication and block unauthorized access to the missing PLC-port condition.",
            "The repaired path depended on PLC-port reachability. Blocking or tightly authorizing that predicate prevents the attacker from satisfying the missing access condition before the path propagates to the hazard.",
            plc_port_matches
        )

    firewall_matches = matched_predicates(
        "firewall", "vpn", "improper", "configuration", "misconfiguration"
    )
    if firewall_matches:
        add(
            "firewall_configuration",
            "Harden and continuously audit the firewall/VPN configuration represented by the missing repair predicate.",
            "The repaired path depends on a configuration predicate that should not be allowed to hold. Correcting and monitoring that configuration prevents the omitted access condition from enabling later compromise.",
            firewall_matches
        )

    vulnerability_matches = matched_predicates("cve", "vulnerab", "exploit")
    if vulnerability_matches:
        add(
            "vulnerability_condition",
            "Patch, mitigate, or virtually patch the vulnerability condition represented by the repair predicate.",
            "The omitted repair predicate corresponds to an exploitable condition. Removing or mitigating that condition prevents the repaired path from satisfying the prerequisite needed for propagation.",
            vulnerability_matches
        )

    compromised_matches = matched_predicates("compromised", "compromise")
    if compromised_matches:
        add(
            "compromised_state",
            "Detect, isolate, and recover the asset state represented by the compromised repair predicate.",
            "The repaired path required a compromised-state predicate that was missing from the user's mental model. Containing that state prevents it from enabling downstream process or fault predicates.",
            compromised_matches
        )

    process_control_matches = matched_predicates(
        "impair-process-control", "process-control"
    )
    if process_control_matches:
        add(
            "process_control_impairment",
            "Monitor for the missing process-control impairment condition and trigger isolation or safe-mode response.",
            "The repaired path requires a process-control condition that enables the hazard chain. Detecting or preventing that condition interrupts propagation before the physical fault.",
            process_control_matches
        )

    physical_fault_matches = matched_predicates(
        "fault", "pilot-extinction", "flare-flameout", "low-supply-pressure",
        "valve-blocked", "blocked-close", "flameout"
    )
    if physical_fault_matches:
        add(
            "fault_condition",
            "Add monitoring, alarms, or interlocks around the missing fault condition established by repair.",
            "The repair evidence includes a process/fault predicate that was absent from the user's explanation. Monitoring or interlocking that condition gives defenders a control point before the hazard fully develops.",
            physical_fault_matches
        )

    credential_matches = matched_predicates(
        "credential", "authenticated", "logged-in", "login", "privilege", "admin"
    )
    if credential_matches:
        add(
            "credential_or_privilege_condition",
            "Strengthen authentication, least privilege, and monitoring for the missing credential or privilege condition.",
            "The repair evidence contains an omitted credential or privilege predicate. Preventing that predicate from becoming true blocks the path at the missing prerequisite.",
            credential_matches
        )

    # Always provide at least one repair-predicate-specific strategy if repair
    # predicates exist but no specialized rule matched.
    if not strategies:
        add(
            "generic_repair_predicate",
            "Place a blocking, monitoring, or hardening control on the missing repair predicate.",
            "The original path became valid only after repair established this omitted predicate. Keeping this predicate false or monitored stops the path before it reaches the hazard.",
            repair_predicates
        )

    return strategies

def derive_defensive_strategy_candidates(mea_output: Dict[str, Any]) -> List[Dict[str, str]]:
    """
    Creates evidence-bounded starter defensive strategies for a valid path.

    Important design choice:
    - The valid-path explanation is still based on the first-valid backward-MEA
      predicate chain.
    - Defensive strategies are derived ONLY from repair predicates, not from
      inserted repair action names and not from the whole validated path.

    Rationale:
    The repair predicates represent the missing conditions in the user's mental
    model. Therefore, defensive suggestions should target those missing
    conditions directly, rather than the names of the actions that happened to
    establish them.
    """

    repair_summary = extract_repair_predicate_summary(mea_output)

    # If the path did not need repair, there are no missing user-model
    # predicates to target. In that case, leave this section empty rather than
    # suggesting defenses from the whole path.
    if not repair_summary.get("plan_was_repaired"):
        return []

    # Predicate-only repair evidence.
    # Do NOT include inserted action names here. They are retained elsewhere
    # only for traceability and action-count reporting.
    repair_predicates = []
    repair_predicates.extend(repair_summary.get("missing_predicates_before_repair", []))
    repair_predicates.extend(repair_summary.get("predicates_established_by_repair", []))

    repair_predicates = [
        str(p).strip()
        for p in dedupe_preserve_order(repair_predicates)
        if str(p).strip()
    ]

    joined = " ".join(repair_predicates).lower()
    strategies = []
    added_keys = set()

    def matched_predicates(*tokens: str) -> List[str]:
        toks = [t.lower() for t in tokens]
        return [p for p in repair_predicates if any(t in p.lower() for t in toks)]

    def add(key: str, strategy: str, why: str, predicate_matches: List[str]):
        if key in added_keys:
            return
        added_keys.add(key)
        evidence = predicate_matches if predicate_matches else repair_predicates
        strategies.append({
            "strategy": strategy,
            "why_it_stops_the_path": why,
            "supported_by_evidence": "Repair predicate(s): " + "; ".join(evidence),
            "source": "repair_predicates_only"
        })

    network_matches = matched_predicates("access-to-network", "internal-network", "network-access", "network")
    if network_matches:
        add(
            "network_access",
            "Block, segment, or strongly authenticate the missing network-access condition.",
            "The user's path became executable only after the repair predicates established network reachability. Preventing that predicate from becoming true stops the later PLC/process conditions from being reached.",
            network_matches
        )

    plc_port_matches = matched_predicates("access-to-plc-port", "plc-port", "port-", "tcp", "plc")
    if plc_port_matches:
        add(
            "plc_port_access",
            "Allowlist PLC communication and block the missing PLC-port access predicate unless it is explicitly authorized.",
            "The repaired path depends on a PLC-reachability predicate that the user omitted. Filtering or allowlisting that predicate stops the path at the missing access condition before it can propagate to the hazard.",
            plc_port_matches
        )

    vulnerability_matches = matched_predicates("cve", "vulnerab", "exploit")
    if vulnerability_matches:
        add(
            "vulnerability_condition",
            "Patch, mitigate, or virtually patch the vulnerability predicate that repair had to establish.",
            "The missing predicate evidence includes a vulnerability/exploit condition. Removing that condition prevents the repaired path from satisfying the omitted prerequisite.",
            vulnerability_matches
        )

    compromised_matches = matched_predicates("compromised", "compromise")
    if compromised_matches:
        add(
            "compromised_asset_condition",
            "Detect, isolate, and recover the asset state represented by the compromised predicate.",
            "The path depends on a compromised-state predicate that was missing from the user's original explanation. Containing that state prevents it from enabling the downstream chain.",
            compromised_matches
        )

    credential_matches = matched_predicates("credential", "authenticated", "logged-in", "login", "privilege", "admin")
    if credential_matches:
        add(
            "credential_or_privilege_condition",
            "Harden authentication and privilege controls for the missing credential/privilege predicate.",
            "The repair evidence contains an omitted credential or privilege condition. Strong authentication, least privilege, and monitoring prevent that predicate from becoming true.",
            credential_matches
        )

    connection_matches = matched_predicates("connected", "reachable", "route", "remote")
    if connection_matches:
        add(
            "connectivity_condition",
            "Remove unnecessary connectivity and monitor the missing reachability predicate.",
            "The repaired path needs a reachability/connectivity predicate that the user did not include. Network segmentation and monitoring stop that missing condition from enabling later predicates.",
            connection_matches
        )

    if not strategies:
        add(
            "generic_missing_repair_predicate",
            "Place a blocking, hardening, or monitoring control on the missing repair predicate.",
            "The user's original path became valid only after repair established this omitted predicate. Preventing that predicate from becoming true stops the path before the hazard.",
            repair_predicates
        )

    return strategies

def summarize_validity_evidence(mea_output: Dict[str, Any]) -> str:
    """
    Predicate-only validity evidence for the LLM.

    This function intentionally avoids action names in the MEA explanation
    evidence so the LLM explains the path using backward-MEA predicates.

    It still mentions whether the plan was repaired, and which predicates
    were missing/established during repair.
    """

    selected_result = get_selected_first_valid_result(mea_output)

    if selected_result is None:
        return "No valid MEA result was found."

    repair_summary = extract_repair_predicate_summary(mea_output)
    predicate_chain = extract_predicate_dependency_chain(selected_result)

    lines = []

    lines.append("VALIDITY EVIDENCE SOURCE")
    lines.append("Use the first valid backward-MEA result only.")
    lines.append("Do not use action names to explain the causal chain.")
    lines.append("Explain the path using predicates and their dependencies.")
    lines.append("")

    lines.append("Goal predicate:")
    lines.append(f"  {mea_output.get('goal')}")
    lines.append("")

    lines.append("Plan repair status:")
    lines.append(f"  Number of missing actions added: {repair_summary.get('missing_action_added_count', 0)}")
    lines.append(f"  Number of repair rounds: {repair_summary.get('repair_round_count', 0)}")

    if repair_summary["plan_was_repaired"]:
        lines.append("  The original user plan required repair before validity analysis.")
        lines.append(f"  Repair status: {repair_summary.get('repair_status')}")
        lines.append(f"  Repair message: {repair_summary.get('repair_message')}")

        lines.append("")
        lines.append("  Missing predicates before repair:")
        for p in repair_summary["missing_predicates_before_repair"]:
            lines.append(f"    - {p}")

        lines.append("")
        lines.append("  Predicates established by repair:")
        for p in repair_summary["predicates_established_by_repair"]:
            lines.append(f"    - {p}")

        lines.append("")
        lines.append("  Inserted repair action names are intentionally omitted from this explanation evidence.")
        lines.append("  Use the repair predicates above, not action names, when suggesting defensive strategies.")

        lines.append("")
        lines.append(
            "  Important: Mention in the explanation that the original plan "
            "needed repair and state the number of missing actions added. "
            "After that, explain the validity chain below using the "
            "first-valid-node predicates."
        )
    else:
        lines.append("  The original user plan did not require repair.")

    lines.append("")
    lines.append("First-valid backward-MEA predicate chain:")
    if predicate_chain:
        for i, link in enumerate(predicate_chain, start=1):
            required = link.get("required_predicates", [])
            achieved = link.get("achieved_predicates", [])

            lines.append(f"  Link {i}:")

            if required:
                lines.append("    Required predicate(s):")
                for p in required:
                    lines.append(f"      - {p}")
            else:
                lines.append("    Required predicate(s):")
                lines.append("      - None at this abstraction")

            if achieved:
                lines.append("    Achieved predicate(s):")
                for p in achieved:
                    lines.append(f"      - {p}")
            else:
                lines.append("    Achieved predicate(s):")
                lines.append("      - None")

            lines.append("")
    else:
        lines.append("  No non-empty predicate dependency links were found.")

    lines.append("Final required predicates at the first valid node:")
    final_subgoals = selected_result.get("final_required_subgoals", [])

    if final_subgoals:
        for sg in final_subgoals:
            lines.append(f"  - {sg}")
    else:
        lines.append("  - None; all required predicates are supported at this abstraction.")

    lines.append("")
    lines.append("Unachieved predicates at the first valid node:")
    unachieved = selected_result.get("unachieved_subgoals", [])

    if unachieved:
        for sg in unachieved:
            lines.append(f"  - {sg}")
    else:
        lines.append("  - None; the predicate chain is valid.")

    lines.append("")
    lines.append("Defense strategy generation:")
    lines.append(
        "  Defensive strategies are not precomputed in validity evidence. "
        "The explanation generator receives the repair report separately and "
        "derives defenses only from repair predicates."
    )

    return "\n".join(lines)


def summarize_invalidity_evidence(diagnosis: Dict[str, Any]) -> str:
    """
    Predicate-centered invalid-plan diagnosis evidence for the LLM.

    The action name is kept only as context for locating the failure.
    The explanation should be based on missing predicates, required predicates,
    and achieved predicates before the failure.

    The invalidity-causing missing predicates are also explicitly marked as
    hardening pointers. From a security perspective, these are conditions that
    prevented the path from continuing at the selected invalid node. Keeping
    those conditions false, restricted, monitored, or hardened preserves the
    block that stops the path.
    """

    first_failure = diagnosis.get("first_failure")
    execution_trace = diagnosis.get("execution_trace", [])
    invalidity_context = diagnosis.get("invalidity_search_context", {}) or {}

    lines = []

    lines.append("INVALIDITY EVIDENCE SOURCE")
    lines.append("Explain invalidity using grounded predicates, not action names.")
    lines.append("Use the action name only to locate the first failing step.")
    lines.append("Use the invalidity-causing missing predicates as pointers for hardening recommendations.")
    lines.append("")

    if invalidity_context:
        lines.append("Invalidity search context:")
        if invalidity_context.get("selected_invalid_node"):
            lines.append(f"  Selected invalid node: {invalidity_context.get('selected_invalid_node')}")
        lines.append(f"  FULL used as explanation node: {invalidity_context.get('full_model_used_as_explanation_node', False)}")
        lines.append(f"  FULL checked during explanation search: {invalidity_context.get('full_model_checked_during_search', False)}")
        lines.append("")

    lines.append(f"Plan executed completely: {diagnosis.get('plan_executed')}")
    lines.append("")

    if first_failure:
        lines.append("First invalid point:")
        lines.append(f"  Step number: {first_failure.get('step_number')}")
        lines.append(f"  Failing step context: {first_failure.get('plan_action')}")
        lines.append(f"  Failure type: {first_failure.get('failure_type')}")
        lines.append(f"  Message: {first_failure.get('message')}")
        lines.append("")

        lines.append("Missing predicates that caused invalidity:")
        missing = first_failure.get("missing_preconditions", [])
        if missing:
            for p in missing:
                lines.append(f"  - {p}")
        else:
            lines.append("  - None listed")

        lines.append("")
        lines.append("Hardening pointer:")
        if missing:
            lines.append(
                "  The missing predicates above are the exact conditions that make the "
                "path fail at this node. Defensive hardening should focus on keeping "
                "these conditions false, restricted, monitored, or difficult for an "
                "attacker to establish."
            )
        else:
            lines.append("  No predicate-level hardening pointer is available because no missing predicate was listed.")
    else:
        lines.append("No first failure was found.")
        lines.append("")

    lines.append("")
    lines.append("Predicate execution trace before failure:")

    for step in execution_trace:
        lines.append("")
        lines.append(f"Step {step.get('step_number')}")
        lines.append(f"  Status: {step.get('status')}")

        # Keep action only as context, not the main explanation basis.
        lines.append(f"  Step context: {step.get('plan_action')}")

        lines.append("  Required predicates:")
        required = step.get("grounded_preconditions", [])
        if required:
            for p in required:
                lines.append(f"    - {p}")
        else:
            lines.append("    - None")

        lines.append("  Missing predicates:")
        missing = step.get("missing_preconditions", [])
        if missing:
            for p in missing:
                lines.append(f"    - {p}")
        else:
            lines.append("    - None")

        lines.append("  Predicates established if step executes:")
        effects = step.get("grounded_effects", [])
        if effects:
            for e in effects:
                lines.append(f"    - {e}")
        else:
            lines.append("    - None")

    return "\n".join(lines)


def extract_invalidity_condition_summary(diagnosis: Dict[str, Any]) -> Dict[str, Any]:
    """
    Extracts the predicate evidence that caused the selected invalid node to fail.

    This helper is used by the explanation generator, not by the invalidity
    analysis itself. It keeps hardening recommendations tied to the same
    missing predicates used to explain invalidity.
    """

    first_failure = diagnosis.get("first_failure") or {}
    execution_trace = diagnosis.get("execution_trace", []) or []
    invalidity_context = diagnosis.get("invalidity_search_context", {}) or {}

    missing = first_failure.get("missing_preconditions", []) or []
    missing = [str(p).strip() for p in dedupe_preserve_order(missing) if str(p).strip()]

    failed_step_number = first_failure.get("step_number")
    failed_trace = None
    for step in execution_trace:
        if step.get("step_number") == failed_step_number:
            failed_trace = step
            break

    return {
        "selected_invalid_node": invalidity_context.get("selected_invalid_node"),
        "failed_step_number": failed_step_number,
        "failing_step_context": first_failure.get("plan_action"),
        "failure_type": first_failure.get("failure_type"),
        "invalidity_causing_predicates": missing,
        "required_predicates_at_failed_step": (failed_trace or {}).get("grounded_preconditions", []),
        "effects_if_failed_step_executed": (failed_trace or {}).get("grounded_effects", []),
        "full_model_used_as_explanation_node": invalidity_context.get("full_model_used_as_explanation_node", False),
        "full_model_checked_during_search": invalidity_context.get("full_model_checked_during_search", False)
    }


def format_invalidity_hardening_evidence_for_prompt(summary: Dict[str, Any]) -> str:
    """
    Formats invalidity-causing predicates for the invalid explanation generator.
    """

    lines = []
    lines.append("INVALIDITY-BASED HARDENING EVIDENCE")
    lines.append("Generate hardening recommendations only from the invalidity-causing predicates listed below.")
    lines.append("These predicates are the conditions that were missing at the selected invalid node.")
    lines.append("Security interpretation: preserving or strengthening the absence of these conditions keeps the path blocked.")
    lines.append("")

    if summary.get("selected_invalid_node"):
        lines.append(f"Selected invalid node: {summary.get('selected_invalid_node')}")
    lines.append(f"Failed step number: {summary.get('failed_step_number')}")
    lines.append(f"Failing step context: {summary.get('failing_step_context')}")
    lines.append(f"Failure type: {summary.get('failure_type')}")
    lines.append("")

    lines.append("Invalidity-causing predicates:")
    predicates = summary.get("invalidity_causing_predicates", []) or []
    if predicates:
        for p in predicates:
            lines.append(f"  - {p}")
    else:
        lines.append("  - None")

    lines.append("")
    lines.append("Required predicates at failed step:")
    required = summary.get("required_predicates_at_failed_step", []) or []
    if required:
        for p in required:
            lines.append(f"  - {p}")
    else:
        lines.append("  - None")

    return "\n".join(lines)


def sanitize_invalidity_hardening_recommendations(
    recommendations: List[Dict[str, Any]],
    invalidity_summary: Dict[str, Any]
) -> List[Dict[str, Any]]:
    """
    Guardrail for LLM-generated hardening recommendations.

    Keeps only recommendations that cite at least one invalidity-causing
    predicate. This prevents the invalid explanation from suggesting broad
    hardening for the entire path when the requested security pointer is the
    condition that made the selected node invalid.
    """

    predicates = [
        str(p).strip()
        for p in invalidity_summary.get("invalidity_causing_predicates", []) or []
        if str(p).strip()
    ]

    if not predicates:
        return []

    cleaned = []
    for item in recommendations or []:
        if not isinstance(item, dict):
            continue
        combined_text = " ".join(str(v) for v in item.values())
        supported = item.get("supported_by_invalid_condition", [])
        if isinstance(supported, str):
            combined_text += " " + supported
        elif isinstance(supported, list):
            combined_text += " " + " ".join(str(v) for v in supported)

        if any(pred in combined_text for pred in predicates):
            cleaned.append(item)

    return cleaned


def fallback_invalidity_hardening_recommendations(
    invalidity_summary: Dict[str, Any]
) -> List[Dict[str, Any]]:
    """
    Minimal deterministic fallback used only inside the explanation generator
    when the LLM omits evidence-grounded hardening recommendations.
    """

    recs = []
    for pred in invalidity_summary.get("invalidity_causing_predicates", []) or []:
        p = str(pred)
        lower = p.lower()

        if "port" in lower or "plc" in lower or "access" in lower or "network" in lower:
            strategy = "Maintain segmentation, allowlisting, and access control for the missing reachability condition."
            why = "The path fails because this reachability or access condition is not established; hardening should preserve that block."
        elif "vulnerab" in lower or "cve" in lower or "exploit" in lower:
            strategy = "Patch, configuration-harden, or compensate for the missing vulnerability/exploitability condition."
            why = "The path fails because the required exploitable condition is not established; hardening should prevent that condition from becoming true."
        elif "compromised" in lower or "compromise" in lower:
            strategy = "Detect, isolate, and recover the asset state represented by the missing compromised condition."
            why = "The path fails because the compromised-state condition is absent; hardening should keep that state from being established."
        elif "fault" in lower or "pressure" in lower or "pilot" in lower or "valve" in lower or "flame" in lower:
            strategy = "Add process monitoring, alarms, or interlocks around the missing process/fault condition."
            why = "The path fails because the required process-side condition is absent; hardening should detect and prevent that condition from emerging."
        else:
            strategy = "Monitor and control the missing condition so it remains false during operation."
            why = "The path fails at this condition; hardening should preserve this failure point as a defensive barrier."

        recs.append({
            "invalidity_condition": p,
            "hardening_strategy": strategy,
            "why_this_hardens_the_system": why,
            "supported_by_invalid_condition": [p]
        })

    return recs



In [ ]:

# ============================================================
# Generate explanation when a NON_EXISTENCE query is valid in FULL
# ============================================================

def generate_not_invalid_in_concrete_model_explanation(
    user_query_path: str,
    invalidity_result_json: str,
    output_json: str,
    output_txt: str,
    mapped_plan_path: Optional[str] = None,
    goal_file: Optional[str] = None,
    generator=None
) -> Dict[str, Any]:
    """
    Generates a deterministic explanation for the case where the user asks why
    a path is invalid, but the path is actually valid in the concrete/FULL model.
    No LLM is required because the reason is fully determined by the FULL check.
    """

    invalidity_result = {}
    if invalidity_result_json and os.path.exists(invalidity_result_json):
        invalidity_result = read_json(invalidity_result_json)

    full_check = invalidity_result.get("full_concrete_check", {}) or {}
    full_status = full_check.get("status", "UNKNOWN")
    full_trace = full_check.get("trace_file")

    explanation_text = (
        "The provided path is not invalid in the concrete model. "
        "Although the query asks for an invalid-path explanation, the mapped path "
        "was checked against the concrete/FULL CPS model and VAL reported it as valid. "
        "Therefore, the user's expectation that the path should fail does not match "
        "the concrete model result."
    )

    parsed = {
        "status": "not_invalid_in_concrete_model",
        "answer_type": "path_expected_invalid_but_valid_in_concrete_model",
        "short_explanation": "The path is valid in the concrete model, so it is not an invalid path.",
        "detailed_explanation": explanation_text,
        "reason": (
            "The FULL/concrete consistency guard returned VALID before the "
            "stakeholder/bridge invalidity explanation search."
        ),
        "full_concrete_status": full_status,
        "full_concrete_trace": full_trace,
        "full_model_checked_as_consistency_guard": True,
        "full_model_checked_during_search": False,
        "full_model_used_as_explanation_node": False,
        "search_excludes_full_model": True,
        "mapped_plan_path": mapped_plan_path,
        "goal_file": goal_file,
        "invalidity_result": invalidity_result
    }

    write_json(output_json, parsed)

    lines = []
    lines.append("=== NOT INVALID IN CONCRETE MODEL EXPLANATION ===")
    lines.append("")
    lines.append("Short answer:")
    lines.append(parsed["short_explanation"])
    lines.append("")
    lines.append("Explanation:")
    lines.append(explanation_text)
    lines.append("")
    lines.append("Concrete/FULL consistency guard:")
    lines.append(f"  Status: {full_status}")
    lines.append(f"  Trace: {full_trace}")
    lines.append("")
    lines.append("Search boundary:")
    lines.append("  FULL model checked as consistency guard: True")
    lines.append("  FULL model checked during explanation-node search: False")
    lines.append("  FULL model used as explanation node: False")
    lines.append("  Stakeholder/bridge search excludes FULL: True")

    write_text(output_txt, "\n".join(lines))

    return parsed


In [ ]:
# ============================================================
# Generate explanation when plan repair fails
# ============================================================

def generate_plan_repair_failed_explanation_with_llm(
    user_query_path: str,
    repair_report_json: str,
    mapped_plan_path: str,
    goal_file: str,
    output_json: str,
    output_txt: str,
    generator=None
) -> Dict[str, Any]:

    explanation_text = (
        "The provided path is not valid in the concrete model. "
        "Because the plan could not be repaired into a valid executable path, "
        "the system cannot validate this path for the declared goal."
    )

    parsed = {
        "status": "plan_repair_failed",
        "short_explanation": explanation_text,
        "detailed_explanation": explanation_text,
        "likely_reason": (
            "The plan expected to be valid failed the concrete/FULL validity guard and was not repairable."
        ),
        "missing_requirements": [],
        "what_to_check_or_fix": []
    }

    parsed["repair_report_json"] = repair_report_json
    parsed["mapped_plan_path"] = mapped_plan_path
    parsed["goal_file"] = goal_file

    write_json(output_json, parsed)

    lines = []
    lines.append("=== PLAN REPAIR FAILED EXPLANATION ===")
    lines.append("")
    lines.append(explanation_text)

    write_text(output_txt, "\n".join(lines))

    return parsed

In [ ]:
# ============================================================
# Cell 20. LLM explanation for valid path
# ============================================================

def generate_valid_path_explanation_with_llm(
    user_query_path: str,
    mea_result_json: str,
    output_json: str,
    output_txt: str,
    generator,
    repair_report_json: Optional[str] = None
) -> Dict[str, Any]:
    """
    Generates a user-facing explanation for why the attack-fault path is valid.

    Current behavior:
    - Validity analysis provides the selected backward-MEA trace for why the
      repaired path reaches the hazard.
    - The plan-repair report is passed separately into this explanation
      generator.
    - The explanation generator uses the repair report to produce:
        1. repair action count,
        2. missing user knowledge, and
        3. defensive starter strategies.
    - Defensive strategies are generated from repair predicates only, because
      those predicates represent conditions missing from the user's mental model.
    """

    user_data = parse_user_txt(read_text(user_query_path))
    mea_output = read_json(mea_result_json)

    # The validity evidence explains why the path reaches the hazard.
    validity_evidence = summarize_validity_evidence(mea_output)

    # The repair report is passed to the explanation generator as a separate
    # evidence source. It is used for missing knowledge and defense generation.
    repair_summary = load_repair_summary_for_explanation(
        repair_report_json=repair_report_json,
        mea_output=mea_output
    )
    repair_report_evidence = format_repair_report_evidence_for_prompt(repair_summary)

    prompt = f"""
You are the Explanation Generator for an Explainability Augmented Resiliency Graph, EARG.

The user is asking why an attack-fault path is valid or why the path reaches the expected outcome.

Generate a natural, user-facing explanation using TWO evidence sources:

1. Validity evidence:
   - Use this to explain why the repaired path reaches the hazard.
   - This evidence comes from the selected first-valid backward-MEA predicate chain.

2. Repair report evidence:
   - Use this to explain what was missing from the user's knowledge.
   - Use this to generate defensive starter strategies.
   - Defensive strategies must be based only on repair predicates, not action names and not the whole validity chain.

Critical reasoning rules:
- Base the validity explanation on the FIRST VALID backward-MEA predicate chain.
- Explain the causal chain using predicates/conditions, not PDDL action names.
- Do not structure the validity explanation around action names.
- Do not say "the action causes..." unless you translate it into the condition it establishes.
- If the repair report says the original plan required repair, explicitly mention that the original plan needed repair before validity analysis.
- If repair occurred, state the exact number of missing actions added: {repair_summary.get("missing_action_added_count", 0)}.
- Use the repair report to summarize missing user knowledge: what necessary predicates/conditions the user did not include in the original path.
- Generate defensive starter strategies ONLY from the repair predicates: missing predicates before repair and predicates established by repair.
- Do not generate defensive strategies from predicates that appear only in the backward-MEA validity chain.
- Do not invent predicates, vulnerabilities, objects, or causal links.
- Use only the evidence provided.

Style rules:
- Start the short answer with: "The path is valid because..."
- Use natural language, not raw PDDL when possible.
- Do not mention VAL.
- Do not mention lattice nodes.
- Do not mention FULL.
- Do not mention abstraction levels.
- Do not mention means-end analysis.
- Do not compare this explanation to another model.
- Return only valid JSON.

User query:
{user_data["query"]}

User steps:
{format_user_steps_for_prompt(user_data)}

Predicate-level validity evidence:
{validity_evidence}

Repair report evidence for missing knowledge and defenses:
{repair_report_evidence}

Return JSON with this exact schema:

{{
  "answer_type": "attack_fault_path_validity_explanation",
  "short_answer": "The path is valid because ...",
  "repair_action_count": {repair_summary.get("missing_action_added_count", 0)},
  "plan_repair_note": "If repair occurred, state exactly how many missing actions were added and explain what missing conditions they established. If no repair occurred, say that no repair was needed and the count is 0.",
  "missing_user_knowledge": [
    {{
      "missing_condition": "predicate or condition missing from the user's original mental model",
      "why_it_matters": "why this condition was necessary for the path to execute or propagate",
      "supported_by_repair_evidence": "repair predicate from the repair report"
    }}
  ],
  "predicate_chain_explanation": "Explain the first-valid backward-MEA predicate chain in natural language, using conditions rather than action names.",
  "explanation": "Natural user-facing explanation of why the repaired or original path is valid.",
  "defensive_strategies": [
    {{
      "strategy": "starter defensive strategy grounded in a repair predicate",
      "why_it_stops_the_path": "why this blocks the attack-fault chain before the hazard",
      "supported_by_evidence": "specific repair predicate that supports this strategy"
    }}
  ],
  "step_by_step_explanation": [
    {{
      "step_number": 1,
      "predicate_dependency": "condition A supports condition B",
      "explanation": "plain-English explanation of this predicate dependency"
    }}
  ],
  "key_reason": "one sentence summary of why the path reaches the expected outcome"
}}
""".strip()

    output = generator(prompt)[0]["generated_text"]
    parsed = extract_json_from_text(output)

    # Deterministic metadata guardrails.
    # - Keep the repair count synchronized with the repair report.
    # - Remove any LLM defense that is not tied to repair-predicate evidence.
    # - If the LLM omits repair-supported defenses, generate a fallback here in
    #   the explanation generator from the repair report only.
    parsed["repair_action_count"] = repair_summary.get("missing_action_added_count", 0)
    parsed.setdefault("missing_user_knowledge", [])

    repair_only_strategies = sanitize_repair_only_defensive_strategies(
        parsed.get("defensive_strategies", []),
        repair_summary
    )

    if not repair_only_strategies:
        repair_only_strategies = derive_repair_report_defensive_strategy_fallback(repair_summary)

    parsed["defensive_strategies"] = repair_only_strategies

    write_json(output_json, {
        "input_user_query": user_query_path,
        "input_mea_result": mea_result_json,
        "input_repair_report": repair_report_json,
        "evidence_style": "validity_chain_plus_repair_report_explanation_generated_defense",
        "repair_summary": repair_summary,
        "predicate_level_evidence": validity_evidence,
        "repair_report_evidence_for_explanation_generator": repair_report_evidence,
        "llm_explanation": parsed,
        "raw_llm_output": output
    })

    lines = []
    lines.append("=== VALID PATH EXPLANATION ===")
    lines.append("")
    lines.append("Short answer:")
    lines.append(parsed.get("short_answer", ""))
    lines.append("")
    lines.append("Repair action count:")
    lines.append(str(parsed.get("repair_action_count", repair_summary.get("missing_action_added_count", 0))))
    lines.append("")
    lines.append("Plan repair note:")
    lines.append(parsed.get("plan_repair_note", ""))
    lines.append("")
    lines.append("What was missing from the user's knowledge:")

    missing_knowledge = parsed.get("missing_user_knowledge", []) or []
    if missing_knowledge:
        for i, item in enumerate(missing_knowledge, start=1):
            lines.append("")
            lines.append(f"Missing knowledge {i}:")
            lines.append(f"  Missing condition: {item.get('missing_condition')}")
            lines.append(f"  Why it matters: {item.get('why_it_matters')}")
            lines.append(f"  Supported by repair evidence: {item.get('supported_by_repair_evidence')}")
    else:
        lines.append("")
        lines.append("No missing user knowledge was identified because the original path did not require repair.")

    lines.append("")
    lines.append("Predicate-chain explanation:")
    lines.append(parsed.get("predicate_chain_explanation", ""))
    lines.append("")
    lines.append("Explanation:")
    lines.append(parsed.get("explanation", ""))
    lines.append("")
    lines.append("Defensive starter strategies from the repair report only:")
    lines.append(
        "These strategies are generated by the explanation generator from the "
        "repair predicates that were missing from the user's mental model."
    )

    repair_only_strategies = parsed.get("defensive_strategies", []) or []
    if repair_only_strategies:
        for i, item in enumerate(repair_only_strategies, start=1):
            lines.append("")
            lines.append(f"Strategy {i}:")
            lines.append(f"  Strategy: {item.get('strategy')}")
            lines.append(f"  Why it stops the path: {item.get('why_it_stops_the_path')}")
            lines.append(f"  Supported by repair evidence: {item.get('supported_by_evidence')}")
    else:
        lines.append("")
        lines.append("No repair-predicate-specific defensive strategy was generated because no repair predicates were available in the repair report.")

    lines.append("")
    lines.append("Step-by-step predicate explanation:")

    for item in parsed.get("step_by_step_explanation", []):
        lines.append("")
        lines.append(f"Step {item.get('step_number')}:")
        lines.append(f"  Predicate dependency: {item.get('predicate_dependency')}")
        lines.append(f"  Explanation: {item.get('explanation')}")

    lines.append("")
    lines.append("Key reason:")
    lines.append(parsed.get("key_reason", ""))

    write_text(output_txt, "\n".join(lines))

    print("Saved valid-path explanation JSON:", output_json)
    print("Saved valid-path explanation TXT:", output_txt)

    return parsed


In [ ]:
# ============================================================
# Cell 21. LLM explanation for invalid path
# ============================================================

def generate_invalid_path_explanation_with_llm(
    user_query_path: str,
    diagnosis_json: str,
    output_json: str,
    output_txt: str,
    generator
) -> Dict[str, Any]:
    """
    Generates a user-facing explanation for why the attack-fault path is invalid.

    Design choice:
    - Invalidity is explained from the first failing predicates in the selected
      invalid node.
    - Security hardening recommendations are generated by the explanation
      generator from those same invalidity-causing predicates.
    - The invalidity analysis does not precompute hardening recommendations.
    """

    user_data = parse_user_txt(read_text(user_query_path))
    diagnosis = read_json(diagnosis_json)

    invalidity_evidence = summarize_invalidity_evidence(diagnosis)
    invalidity_condition_summary = extract_invalidity_condition_summary(diagnosis)
    hardening_evidence = format_invalidity_hardening_evidence_for_prompt(invalidity_condition_summary)

    prompt = f"""
You are the Explanation Generator for an Explainability Augmented Resiliency Graph, EARG.

The user is asking why an attack-fault path is invalid or why the path does not reach the expected outcome.

Generate a clear, natural, user-facing explanation.

Important style rules:
- Start the short answer with: "The path is invalid because..."
- Use natural language, not raw PDDL.
- Do not show raw predicates in the prose unless necessary for traceability.
- Do not show raw PDDL action names unless needed; translate them into plain English.
- Base the invalidity explanation on the missing grounded predicates.
- Use the failing action only to identify where the predicate failure occurs.
- Do not structure the explanation around the action name.
- Explain which required predicates were not true before the failing step.
- Explain why those missing predicates prevent the remaining path from reaching the goal.
- Explain why that missing condition blocks the remaining attack-fault sequence.
- Avoid vague phrases like "some condition", "such as", or "for example" when exact evidence is provided.
- Do not mention VAL.
- Do not mention lattice nodes in the user-facing prose.
- Do not mention FULL.
- Do not mention abstraction levels.
- Do not mention means-end analysis.
- Do not compare this explanation to another model.
- Do not invent vulnerabilities, objects, predicates, faults, or causal links.
- Use only the evidence provided.
- Return only valid JSON.

Hardening recommendation rules:
- Generate hardening recommendations from the invalidity-causing predicates only.
- Treat the condition that made the selected node invalid as the security hardening pointer.
- The security interpretation is: if defenders keep this missing condition false, restricted, monitored, or difficult to establish, the path remains blocked.
- Do not recommend defenses for the whole path.
- Do not recommend defenses from successful earlier steps unless those predicates are also listed as invalidity-causing predicates.
- Each hardening recommendation must cite at least one raw invalidity-causing predicate in supported_by_invalid_condition.
- If the missing predicate is a network/access condition, recommend segmentation, allowlisting, authentication, or monitoring for that condition.
- If the missing predicate is a PLC-port condition, recommend PLC communication allowlisting and port restriction.
- If the missing predicate is a vulnerability or exploitability condition, recommend patching, configuration hardening, or compensating controls.
- If the missing predicate is a compromised-state condition, recommend detection, isolation, and recovery controls.
- If the missing predicate is a process/fault condition, recommend monitoring, alarms, interlocks, or operating constraints that prevent that condition from emerging.

How to translate evidence:
- If the failing action is an exploit action, describe it as "the attack tries to exploit [vulnerability] on [target]".
- If the missing condition is a vulnerability condition, explain that the target has not been established as vulnerable in the required way.
- If the missing condition is an exploitability/setup condition, explain that the required exploit setup or exploitable state has not been established.
- If the missing condition is a network/access condition, explain that the attacker does not yet have the required access or connectivity.
- If the missing condition is a process/fault condition, explain that the needed physical or process-side condition has not occurred yet.

User query:
{user_data["query"]}

User steps:
{format_user_steps_for_prompt(user_data)}

Invalidity evidence:
{invalidity_evidence}

Hardening evidence:
{hardening_evidence}

Return JSON with this exact schema:

{{
  "answer_type": "attack_fault_path_invalidity_explanation",
  "short_answer": "The path is invalid because ...",
  "explanation": "Natural user-facing explanation of why the attack-fault path is invalid.",
  "what_worked_before_failure": "Plain-English explanation of which earlier steps succeeded before the failure.",
  "first_failing_step": {{
    "step_number": 1,
    "user_step": "original user step if available",
    "plain_english_action": "plain-English version of the failing action",
    "why_it_fails": "plain-English explanation of why this step fails",
    "missing_conditions": [
      "plain-English missing condition, not raw PDDL"
    ]
  }},
  "invalidity_condition_as_hardening_pointer": "Explain how the condition that made the node invalid identifies a security hardening point.",
  "hardening_recommendations": [
    {{
      "invalidity_condition": "plain-English condition that caused invalidity",
      "hardening_strategy": "security hardening recommendation tied to that condition",
      "why_this_hardens_the_system": "explain why preserving or enforcing this missing condition blocks the path",
      "supported_by_invalid_condition": [
        "raw invalidity-causing predicate from the evidence"
      ]
    }}
  ],
  "why_goal_is_not_reached": "Explain why the final attack-fault outcome is not reached.",
  "repair_hint": "Explain what kind of missing setup, condition, or intermediate step would be needed for the path to continue, without inventing unsupported details."
}}
""".strip()

    output = generator(prompt)[0]["generated_text"]
    parsed = extract_json_from_text(output)

    # Guardrail: keep only hardening recommendations tied to the exact
    # invalidity-causing predicates. If the LLM omits them or makes them too
    # broad, create a minimal predicate-grounded fallback inside the
    # explanation generator.
    hardening_recs = parsed.get("hardening_recommendations", [])
    hardening_recs = sanitize_invalidity_hardening_recommendations(
        hardening_recs,
        invalidity_condition_summary
    )
    if not hardening_recs:
        hardening_recs = fallback_invalidity_hardening_recommendations(
            invalidity_condition_summary
        )
    parsed["hardening_recommendations"] = hardening_recs

    write_json(output_json, {
        "input_user_query": user_query_path,
        "input_invalid_diagnosis": diagnosis_json,
        "invalidity_condition_summary": invalidity_condition_summary,
        "hardening_generation_source": "explanation_generator_from_invalidity_causing_predicates",
        "hardening_scope": "selected_invalid_node_missing_predicates_only",
        "llm_explanation": parsed,
        "raw_llm_output": output
    })

    lines = []
    lines.append("=== INVALID PATH EXPLANATION ===")
    lines.append("")
    lines.append("Short answer:")
    lines.append(parsed.get("short_answer", ""))
    lines.append("")
    lines.append("Explanation:")
    lines.append(parsed.get("explanation", ""))
    lines.append("")
    lines.append("What worked before failure:")
    lines.append(parsed.get("what_worked_before_failure", ""))

    first = parsed.get("first_failing_step", {})
    lines.append("")
    lines.append("First failing step:")
    lines.append(f"  Step number: {first.get('step_number')}")
    lines.append(f"  User step: {first.get('user_step')}")
    lines.append(f"  Action: {first.get('plain_english_action')}")
    lines.append(f"  Why it fails: {first.get('why_it_fails')}")

    lines.append("  Missing conditions:")
    for condition in first.get("missing_conditions", []):
        lines.append(f"    - {condition}")

    lines.append("")
    lines.append("Invalidity condition as hardening pointer:")
    lines.append(parsed.get("invalidity_condition_as_hardening_pointer", ""))

    lines.append("")
    lines.append("Hardening recommendations from invalidity-causing conditions only:")
    lines.append(
        "These recommendations target the missing predicates that made the selected "
        "node invalid. They do not target the entire path."
    )

    for idx, rec in enumerate(parsed.get("hardening_recommendations", []) or [], start=1):
        lines.append("")
        lines.append(f"Recommendation {idx}:")
        lines.append(f"  Invalidity condition: {rec.get('invalidity_condition', '')}")
        lines.append(f"  Hardening strategy: {rec.get('hardening_strategy', '')}")
        lines.append(f"  Why this hardens the system: {rec.get('why_this_hardens_the_system', '')}")
        supported = rec.get("supported_by_invalid_condition", []) or []
        if isinstance(supported, str):
            supported = [supported]
        lines.append("  Supported by invalidity condition:")
        if supported:
            for p in supported:
                lines.append(f"    - {p}")
        else:
            lines.append("    - None")

    lines.append("")
    lines.append("Why the goal is not reached:")
    lines.append(parsed.get("why_goal_is_not_reached", ""))

    lines.append("")
    lines.append("Repair hint:")
    lines.append(parsed.get("repair_hint", ""))

    write_text(output_txt, "\n".join(lines))

    print("Saved invalid-path explanation JSON:", output_json)
    print("Saved invalid-path explanation TXT:", output_txt)

    return parsed


In [ ]:
# ============================================================
# Define preprocessing stage runner
# ============================================================

def run_preprocessing_stages(generator) -> Dict[str, Any]:
    """
    Creates the intermediate files needed by the final controller:

    1. declared stakeholder view + stakeholder_and_action_mapping_output.json
    2. mapped_plan_fragments.txt
    3. path_existence_intent.json
    4. inferred_goal_predicate.txt
    5. goal_predicate_inference_output.json
    """

    print("\n==============================")
    print("RUNNING PREPROCESSING STAGES")
    print("==============================")

    check_file_exists(INPUT_TXT, "input query txt")
    check_file_exists(FULL_DOMAIN_PDDL, "full domain PDDL")

    # Stage 1: declared stakeholder view + action mapping
    mapping = generate_stakeholder_and_plan_mapping(
        input_txt=INPUT_TXT,
        domain_pddl=FULL_DOMAIN_PDDL,
        output_json=STAKEHOLDER_JSON,
        output_plan_txt=PLAN_FILE,
        generator=generator
    )

    print("Created:", STAKEHOLDER_JSON)
    print("Declared stakeholder view:", mapping.get("declared_stakeholder"))
    print("Created:", PLAN_FILE)

    verify_mapped_plan_file_is_grounded(PLAN_FILE)

    check_plan_action_arity_against_domain(
        domain_path=FULL_DOMAIN_PDDL,
        plan_path=PLAN_FILE
    )

    # Stage 2: path intent classification
    intent = classify_path_existence_intent(
        input_txt=INPUT_TXT,
        output_json=PATH_INTENT_JSON,
        generator=generator
    )

    print("Created:", PATH_INTENT_JSON)

    # Stage 3: goal predicate inference

    check_file_exists(FULL_PROBLEM_PDDL, "full problem PDDL")

    goal = infer_goal_predicate_with_llm(
        input_txt=INPUT_TXT,
        domain_pddl=FULL_DOMAIN_PDDL,
        problem_pddl=FULL_PROBLEM_PDDL,
        output_goal_txt=GOAL_FILE,
        output_json=GOAL_OUTPUT_JSON,
        generator=generator
    )

    print("Created:", GOAL_FILE)
    print("Created:", GOAL_OUTPUT_JSON)

    return {
        "mapping": mapping,
        "intent": intent,
        "goal": goal
    }

In [ ]:
# ============================================================
# Cell 22. Run full EARG pipeline
# ============================================================

goal_update_result = None
goal_restore_result = None
pipeline_result = None
preprocess_result = None
explanation = None

start_time = time.time()

try:
    # --------------------------------------------------------
    # Step A: Run preprocessing
    # This creates the inferred goal file.
    # --------------------------------------------------------
    preprocess_result = run_preprocessing_stages(generator)

    # --------------------------------------------------------
    # Step B: Update lattice problem goals only when different.
    # --------------------------------------------------------
    goal_update_result = update_all_lattice_problem_goals(
        lattice_json_path=LATTICE_JSON,
        goal_file=GOAL_FILE,
        backup_dir=GOAL_BACKUP_DIR,
        report_txt_path=GOAL_UPDATE_REPORT_TXT
    )

    if goal_update_result.get("failed_files"):
        raise RuntimeError(
            "Goal update failed for one or more problem files. "
            f"See report: {GOAL_UPDATE_REPORT_TXT}"
        )

    print("\nGoal update summary:")
    print("Updated:", len(goal_update_result.get("updated_files", [])))
    print("Skipped:", len(goal_update_result.get("skipped_files", [])))
    print("Failed:", len(goal_update_result.get("failed_files", [])))

    # --------------------------------------------------------
    # Step C: Run validity or invalidity pipeline.
    # The pipeline now uses the updated goals.
    # If all files were skipped, the existing goals already matched.
    # --------------------------------------------------------
    pipeline_result = run_earg_validity_or_invalidity_pipeline()

    print("\n==============================")
    print("PIPELINE FINISHED")
    print("==============================")
    print("Intent:", pipeline_result["intent"])
    print("Ran:", pipeline_result["ran"])

    # --------------------------------------------------------
    # Step D: Generate explanation
    # --------------------------------------------------------
    if pipeline_result["intent"] == "EXISTENCE":
      validity_result = pipeline_result["result"]

      if validity_result.get("status") == "plan_repair_failed":
          print("\nPlan repair failed.")
          print("Backward MEA was not run.")
          print("Generating plan-repair-failure explanation instead of valid-path explanation.")

          repair_json_path = validity_result.get("plan_repair_json", "")

          explanation = generate_plan_repair_failed_explanation_with_llm(
              user_query_path=INPUT_TXT,
              repair_report_json=repair_json_path,
              mapped_plan_path=PLAN_FILE,
              goal_file=GOAL_FILE,
              output_json=PLAN_REPAIR_FAILED_EXPLANATION_JSON,
              output_txt=PLAN_REPAIR_FAILED_EXPLANATION_TXT,
              generator=generator
          )

          print("\nPlan-repair-failure explanation saved to:")
          print(PLAN_REPAIR_FAILED_EXPLANATION_TXT)
          print(PLAN_REPAIR_FAILED_EXPLANATION_JSON)

      else:
          explanation = generate_valid_path_explanation_with_llm(
              user_query_path=INPUT_TXT,
              mea_result_json=MEA_OUTPUT_JSON,
              output_json=VALID_EXPLANATION_JSON,
              output_txt=VALID_EXPLANATION_TXT,
              generator=generator,
              repair_report_json=PLAN_REPAIR_JSON
          )

          print("\nFinal valid-path explanation saved to:")
          print(VALID_EXPLANATION_TXT)
          print(VALID_EXPLANATION_JSON)

    elif pipeline_result["intent"] == "NON_EXISTENCE":
        # The invalidity pipeline runs a FULL/concrete consistency guard first.
        # FULL is still excluded from the explanation-node search.
        invalidity_result = pipeline_result["result"]["invalidity_result"]

        explanation = generate_invalid_path_explanation_with_llm(
                user_query_path=INPUT_TXT,
                diagnosis_json=INVALID_DIAGNOSIS_JSON,
                output_json=INVALID_EXPLANATION_JSON,
                output_txt=INVALID_EXPLANATION_TXT,
                generator=generator
            )

            print("\nFinal invalid-path explanation saved to:")
            print(INVALID_EXPLANATION_TXT)
            print(INVALID_EXPLANATION_JSON)


    else:
        raise ValueError(f"Unknown intent: {pipeline_result['intent']}")

    # --------------------------------------------------------
    # Step E: Runtime
    # --------------------------------------------------------
    end_time = time.time()
    total_time = end_time - start_time

    print(f"\nTotal time taken: {total_time:.2f} seconds")

    # --------------------------------------------------------
    # Step F: Save metrics
    # --------------------------------------------------------
    metrics_row = build_test_case_metrics_row_no_json(
        query_name=query,
        preprocess_result=preprocess_result,
        pipeline_result=pipeline_result,
        total_runtime_seconds=total_time,
        notes=(
            f"goal_updated={len(goal_update_result.get('updated_files', []))}; "
            f"goal_skipped={len(goal_update_result.get('skipped_files', []))}"
        )
    )

    append_metrics_row(metrics_row, METRICS_CSV)
    export_metrics_to_excel(METRICS_CSV, METRICS_XLSX)

    print("\nMetrics saved:")
    print("CSV:", METRICS_CSV)
    print("Excel:", METRICS_XLSX)

finally:
    # --------------------------------------------------------
    # Step G: Restore original problem files.
    # This always runs, even if the pipeline crashes.
    # Only files actually modified in Step B are restored.
    # --------------------------------------------------------
    if goal_update_result is not None:
        goal_restore_result = restore_lattice_problem_goals_from_backup(
            goal_update_result=goal_update_result,
            report_txt_path=GOAL_RESTORE_REPORT_TXT
        )
    else:
        print("\nNo goal update was performed, so no restore was needed.")


In [ ]:
# --------------------------------------------------------
# Step F: Debug
# --------------------------------------------------------
print("MEA_OUTPUT_REPORT:", MEA_OUTPUT_REPORT)
print("Exists:", os.path.exists(MEA_OUTPUT_REPORT))

print("VALID_EXPLANATION_TXT:", VALID_EXPLANATION_TXT)
print("Exists:", os.path.exists(VALID_EXPLANATION_TXT))

print("\nMEA report preview:")
print(read_file_content_for_metrics(MEA_OUTPUT_REPORT)[:500])

print("\nValid explanation preview:")
print(read_file_content_for_metrics(VALID_EXPLANATION_TXT)[:500])